In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working

!git clone https://github.com/ahmad903501/small-model-alignment-starter.git

%cd /kaggle/working/small-model-alignment-starter

!pwd
!find . -maxdepth 2 -type f | sort

In [ ]:
!find ./data ./eval_sets -maxdepth 3 -type f | sort

In [ ]:
!python -m pip uninstall -y -q torchao || true

!python -m pip install -q \
  "transformers>=4.43.0,<5" \
  "datasets>=2.18,<5" \
  huggingface_hub \
  accelerate \
  tqdm \
  pyyaml \
  sentencepiece \
  safetensors \
  pytest

!python -m pip install -q -e . --no-deps

In [ ]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

In [ ]:
!echo "===== submission_adapter.py ====="
!sed -n '1,240p' submission_adapter.py

!echo
!echo "===== public smoke tests ====="
!sed -n '1,320p' tests/test_public_adapter_smoke.py

!echo
!echo "===== toy LM ====="
!sed -n '1,320p' tests/toy_lm.py

In [ ]:
!cat pyproject.toml

print("\n===== README =====")
!sed -n '1,260p' README.md

In [ ]:
%%writefile alignment_core.py

from __future__ import annotations

from typing import Sequence

import torch


IGNORE_INDEX = -100


def _encode_without_special_tokens(tokenizer, text: str) -> list[int]:
    """
    Encode one string without allowing the tokenizer to insert BOS/EOS
    or any other special tokens automatically.
    """
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    # Hugging Face tokenizers normally return a Python list, but converting
    # explicitly makes this helper tolerant of other list-like results too.
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.tolist()

    return list(token_ids)


def _get_padding_token_id(tokenizer) -> int:
    """
    Return a usable padding token ID.

    Prefer the tokenizer's explicit pad token. If it has no pad token but
    does have EOS, EOS can safely be used as the physical padding value
    because padding positions are excluded by attention_mask and labels=-100.
    """
    pad_token_id = getattr(tokenizer, "pad_token_id", None)

    if pad_token_id is not None:
        return int(pad_token_id)

    eos_token_id = getattr(tokenizer, "eos_token_id", None)

    if eos_token_id is not None:
        return int(eos_token_id)

    raise ValueError(
        "Tokenizer must provide either pad_token_id or eos_token_id "
        "so batches can be padded."
    )


def build_sft_batch_impl(
    tokenizer,
    prompts: Sequence[str],
    responses: Sequence[str],
    max_length: int,
) -> dict[str, torch.Tensor]:
    """
    Build a padded causal-LM batch for supervised fine-tuning.

    Prompt tokens and padding tokens receive label -100 so that only
    assistant-response tokens contribute to the training loss.

    EOS, when provided by the tokenizer, is appended to the response
    BEFORE sequence truncation.
    """

    if not isinstance(max_length, int):
        raise TypeError("max_length must be an integer.")

    if max_length <= 0:
        raise ValueError("max_length must be positive.")

    if len(prompts) != len(responses):
        raise ValueError(
            "prompts and responses must contain the same number of rows."
        )

    pad_token_id = _get_padding_token_id(tokenizer)
    eos_token_id = getattr(tokenizer, "eos_token_id", None)

    input_rows: list[list[int]] = []
    attention_rows: list[list[int]] = []
    label_rows: list[list[int]] = []

    for prompt, response in zip(prompts, responses):
        if not isinstance(prompt, str) or not isinstance(response, str):
            raise TypeError("Every prompt and response must be a string.")

        prompt_ids = _encode_without_special_tokens(tokenizer, prompt)
        response_ids = _encode_without_special_tokens(tokenizer, response)

        # The manual requires EOS to be appended before truncation.
        if eos_token_id is not None:
            response_ids = response_ids + [int(eos_token_id)]

        # Keep prompt and response separate until labels are built so that
        # the response-only supervision boundary is exact.
        combined_ids = prompt_ids + response_ids

        # Truncate only after EOS has already been appended.
        combined_ids = combined_ids[:max_length]

        # Determine how many prompt tokens survived truncation.
        surviving_prompt_length = min(len(prompt_ids), len(combined_ids))

        # Labels equal the actual token IDs for response tokens only.
        labels = (
            [IGNORE_INDEX] * surviving_prompt_length
            + combined_ids[surviving_prompt_length:]
        )

        real_length = len(combined_ids)
        padding_length = max_length - real_length

        input_ids = combined_ids + [pad_token_id] * padding_length
        attention_mask = [1] * real_length + [0] * padding_length
        labels = labels + [IGNORE_INDEX] * padding_length

        input_rows.append(input_ids)
        attention_rows.append(attention_mask)
        label_rows.append(labels)

    # Handle a zero-row input cleanly as well.
    if len(input_rows) == 0:
        shape = (0, max_length)

        return {
            "input_ids": torch.empty(shape, dtype=torch.long),
            "attention_mask": torch.empty(shape, dtype=torch.long),
            "labels": torch.empty(shape, dtype=torch.long),
        }

    return {
        "input_ids": torch.tensor(input_rows, dtype=torch.long),
        "attention_mask": torch.tensor(attention_rows, dtype=torch.long),
        "labels": torch.tensor(label_rows, dtype=torch.long),
    }

In [ ]:
%%writefile submission_adapter.py

"""Stable grading interface for the SFT and DPO implementation.

Keep the function signatures below unchanged. Each function is a thin
wrapper around the actual implementation in alignment_core.py.
"""

from __future__ import annotations

from alignment_core import build_sft_batch_impl


def build_sft_batch(tokenizer, prompts, responses, max_length):
    """Return input_ids, attention_mask, and response-only labels."""

    return build_sft_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )


def compute_response_logprobs(model, tokenizer, prompts, responses, max_length):
    """Return one summed response-token log-probability per input row."""

    raise NotImplementedError("Response log-probabilities not implemented yet")


def compute_sft_loss(model, tokenizer, prompts, responses, max_length):
    """Return a scalar response-only SFT loss using either documented reduction."""

    raise NotImplementedError("SFT loss not implemented yet")


def build_dpo_batch(
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    max_length,
):
    """Return separate response-masked LM batches for chosen and rejected rows."""

    raise NotImplementedError("DPO batching not implemented yet")


def compute_dpo_loss_from_logps(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps,
    beta,
):
    """Return the scalar mean DPO objective from per-example log probabilities."""

    raise NotImplementedError("Tensor-level DPO loss not implemented yet")


def compute_dpo_loss(
    policy_model,
    reference_model,
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    beta,
    max_length,
):
    """Return a scalar mean DPO loss with a frozen reference computation."""

    raise NotImplementedError("DPO loss not implemented yet")

In [ ]:
!PYTHONPATH=. python -m pytest -q \
    tests/test_public_adapter_smoke.py \
    -k "sft_batch_masks"

In [ ]:
from tests.toy_lm import ToyTokenizer
from alignment_core import build_sft_batch_impl

tokenizer = ToyTokenizer()

# --------------------------------------------------
# 1. EOS survives when there is enough room
# --------------------------------------------------
batch = build_sft_batch_impl(
    tokenizer,
    prompts=["P:"],
    responses=["abc"],
    max_length=6,
)

prompt_len = len(tokenizer.encode("P:", add_special_tokens=False))

assert batch["labels"][0, :prompt_len].eq(-100).all()
assert batch["labels"][0, -1].item() == tokenizer.eos_token_id

print("✓ EOS included when space is available")


# --------------------------------------------------
# 2. EOS is appended BEFORE truncation, so it may
#    legitimately disappear if max_length cuts it off
# --------------------------------------------------
batch = build_sft_batch_impl(
    tokenizer,
    prompts=["P:"],
    responses=["abc"],
    max_length=5,
)

assert tokenizer.eos_token_id not in batch["labels"][0][batch["labels"][0] != -100]

print("✓ EOS can be truncated at max_length")


# --------------------------------------------------
# 3. Prompt itself can consume the entire context.
#    In that case no response token survives.
# --------------------------------------------------
batch = build_sft_batch_impl(
    tokenizer,
    prompts=["abcdefgh"],
    responses=["xyz"],
    max_length=4,
)

assert batch["labels"].eq(-100).all()
assert batch["attention_mask"].eq(1).all()

print("✓ Prompt-only truncation handled correctly")


# --------------------------------------------------
# 4. Mixed sequence lengths are padded correctly
# --------------------------------------------------
batch = build_sft_batch_impl(
    tokenizer,
    prompts=["A:", "Longer:"],
    responses=["x", "response"],
    max_length=16,
)

assert batch["input_ids"].shape == (2, 16)
assert batch["attention_mask"].shape == (2, 16)
assert batch["labels"].shape == (2, 16)

# Every padding position must also be ignored in labels.
padding_positions = batch["attention_mask"] == 0
assert batch["labels"][padding_positions].eq(-100).all()

print("✓ Mixed lengths and padding handled correctly")


# --------------------------------------------------
# 5. Input row order is preserved
# --------------------------------------------------
expected_first = tokenizer.encode("A:", add_special_tokens=False)
expected_second = tokenizer.encode("B:", add_special_tokens=False)

batch = build_sft_batch_impl(
    tokenizer,
    prompts=["A:", "B:"],
    responses=["first", "second"],
    max_length=16,
)

assert batch["input_ids"][0, :len(expected_first)].tolist() == expected_first
assert batch["input_ids"][1, :len(expected_second)].tolist() == expected_second

print("✓ Row order preserved")

print("\nAll SFT batching edge-case checks passed.")

In [ ]:
%%writefile -a alignment_core.py


def _get_model_device(model) -> torch.device:
    """
    Infer the device on which the model expects its inputs.

    This works for the toy model used by the public grader as well as
    ordinary Hugging Face causal language models placed on one device.
    """
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")


def compute_response_logprobs_impl(
    model,
    tokenizer,
    prompts: Sequence[str],
    responses: Sequence[str],
    max_length: int,
) -> torch.Tensor:
    """
    Return one summed response-token log probability per input example.

    Prompt tokens and padding tokens are excluded.

    For a causal LM, logits at sequence position t predict token t+1.
    Therefore we score labels[:, 1:] using logits[:, :-1, :].
    """

    batch = build_sft_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )

    device = _get_model_device(model)

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

    logits = outputs.logits

    # Causal-language-model shift:
    #
    # logits[:, 0] predicts input_ids[:, 1]
    # logits[:, 1] predicts input_ids[:, 2]
    # ...
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]

    # Only response tokens have labels other than -100.
    response_mask = shift_labels.ne(IGNORE_INDEX)

    # gather() requires every index to be a valid vocabulary ID, so replace
    # ignored (-100) positions temporarily with 0. They are masked out later.
    safe_labels = shift_labels.masked_fill(~response_mask, 0)

    log_probs = torch.log_softmax(shift_logits, dim=-1)

    token_logps = log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    # Prompt/padding locations contribute exactly zero.
    token_logps = token_logps * response_mask.to(token_logps.dtype)

    # DPO requires the SUM of response-token log probabilities per example.
    response_logps = token_logps.sum(dim=-1)

    return response_logps

In [ ]:
from pathlib import Path

path = Path("submission_adapter.py")
text = path.read_text()

text = text.replace(
    "from alignment_core import build_sft_batch_impl",
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
)"""
)

text = text.replace(
    '''def compute_response_logprobs(model, tokenizer, prompts, responses, max_length):
    """Return one summed response-token log-probability per input row."""

    raise NotImplementedError("Response log-probabilities not implemented yet")''',
    '''def compute_response_logprobs(model, tokenizer, prompts, responses, max_length):
    """Return one summed response-token log-probability per input row."""

    return compute_response_logprobs_impl(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )'''
)

path.write_text(text)

print(path.read_text())

In [ ]:
!PYTHONPATH=. python -m pytest -q \
    tests/test_public_adapter_smoke.py \
    -k "response_logprobs"

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

print("Reloaded:", alignment_core.__file__)
print(
    "compute_response_logprobs_impl exists:",
    hasattr(alignment_core, "compute_response_logprobs_impl"),
)

In [ ]:
import torch

from tests.toy_lm import ToyTokenizer, ToyCausalLM
from alignment_core import compute_response_logprobs_impl


tokenizer = ToyTokenizer()
model = ToyCausalLM(tokenizer.vocab_size, bias_scale=0.7)

prompt = "Task: "
response = "abc"

actual = compute_response_logprobs_impl(
    model=model,
    tokenizer=tokenizer,
    prompts=[prompt],
    responses=[response],
    max_length=20,
)[0]

response_ids = tokenizer.encode(
    response,
    add_special_tokens=False,
) + [tokenizer.eos_token_id]

single_position_logps = torch.log_softmax(
    model.logit_bias,
    dim=-1,
)

expected = single_position_logps[
    torch.tensor(response_ids)
].sum()

print("Response token IDs:", response_ids)
print("Actual log-prob:   ", actual.item())
print("Expected log-prob: ", expected.item())

torch.testing.assert_close(actual, expected)

print("\n✓ Exact summed response log-probability is correct")


model.zero_grad()

logps = compute_response_logprobs_impl(
    model=model,
    tokenizer=tokenizer,
    prompts=["Prompt: ", "Other: "],
    responses=["abc", "xyz"],
    max_length=20,
)

(-logps.mean()).backward()

assert model.logit_bias.grad is not None
assert model.logit_bias.grad.abs().sum() > 0

print("✓ Gradients flow through response log-probabilities")

In [ ]:
%%writefile -a alignment_core.py


def compute_sft_loss_impl(
    model,
    tokenizer,
    prompts: Sequence[str],
    responses: Sequence[str],
    max_length: int,
) -> torch.Tensor:
    """
    Compute response-only SFT loss.

    Reduction used:
        mean negative log probability over all unmasked response tokens
        in the batch.

    Prompt tokens and padding tokens do not contribute to the loss.
    """

    batch = build_sft_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )

    response_logps = compute_response_logprobs_impl(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )

    # Because causal LM scoring uses labels[:, 1:], count only those
    # response targets that can actually be predicted.
    response_token_count = (
        batch["labels"][:, 1:]
        .ne(IGNORE_INDEX)
        .sum()
        .to(response_logps.device)
        .clamp_min(1)
    )

    loss = -response_logps.sum() / response_token_count

    return loss

In [ ]:
from pathlib import Path

path = Path("submission_adapter.py")
text = path.read_text()

text = text.replace(
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
)""",
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
)"""
)

text = text.replace(
    '''def compute_sft_loss(model, tokenizer, prompts, responses, max_length):
    """Return a scalar response-only SFT loss using either documented reduction."""

    raise NotImplementedError("SFT loss not implemented yet")''',
    '''def compute_sft_loss(model, tokenizer, prompts, responses, max_length):
    """Return a scalar response-only SFT loss using either documented reduction."""

    return compute_sft_loss_impl(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )'''
)

path.write_text(text)

print(path.read_text())

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

print(
    "compute_sft_loss_impl exists:",
    hasattr(alignment_core, "compute_sft_loss_impl"),
)

In [ ]:
!PYTHONPATH=. python -m pytest -q \
    tests/test_public_adapter_smoke.py \
    -k "sft_loss"

In [ ]:
import torch
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

from tests.toy_lm import ToyTokenizer, ToyCausalLM
from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
)


tokenizer = ToyTokenizer()
model = ToyCausalLM(tokenizer.vocab_size, bias_scale=0.4)

prompts = [
    "Do: ",
    "Say: ",
]

responses = [
    "a",
    "longer",
]

max_length = 16


# --------------------------------------------------
# Our SFT loss
# --------------------------------------------------
actual_loss = compute_sft_loss_impl(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    responses=responses,
    max_length=max_length,
)


# --------------------------------------------------
# Independently construct the expected token-mean loss
# --------------------------------------------------
batch = build_sft_batch_impl(
    tokenizer=tokenizer,
    prompts=prompts,
    responses=responses,
    max_length=max_length,
)

response_logps = compute_response_logprobs_impl(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    responses=responses,
    max_length=max_length,
)

response_token_count = (
    batch["labels"][:, 1:]
    .ne(-100)
    .sum()
)

expected_loss = -response_logps.sum() / response_token_count


print("Response log-probs:", response_logps.tolist())
print("Response token count:", response_token_count.item())
print("Actual SFT loss:     ", actual_loss.item())
print("Expected SFT loss:   ", expected_loss.item())

torch.testing.assert_close(actual_loss, expected_loss)

print("\n✓ Exact response-token-mean SFT loss is correct")


# --------------------------------------------------
# Confirm prompt tokens truly do not contribute
# --------------------------------------------------
model2 = ToyCausalLM(tokenizer.vocab_size, bias_scale=0.4)

loss_a = compute_sft_loss_impl(
    model=model2,
    tokenizer=tokenizer,
    prompts=["aaaa: "],
    responses=["same"],
    max_length=20,
)

loss_b = compute_sft_loss_impl(
    model=model2,
    tokenizer=tokenizer,
    prompts=["zzzz: "],
    responses=["same"],
    max_length=20,
)

torch.testing.assert_close(loss_a, loss_b)

print("✓ Changing only the prompt does not change response-only loss for the toy LM")

In [ ]:
%%writefile -a alignment_core.py


def build_dpo_batch_impl(
    tokenizer,
    prompts: Sequence[str],
    chosen_responses: Sequence[str],
    rejected_responses: Sequence[str],
    max_length: int,
) -> dict[str, dict[str, torch.Tensor]]:
    """
    Build separate response-masked causal-LM batches for the chosen
    and rejected responses used by DPO.

    Both sides use the same prompt-processing, EOS, truncation,
    padding, and response-only masking logic as SFT.
    """

    if len(prompts) != len(chosen_responses):
        raise ValueError(
            "prompts and chosen_responses must contain the same number of rows."
        )

    if len(prompts) != len(rejected_responses):
        raise ValueError(
            "prompts and rejected_responses must contain the same number of rows."
        )

    chosen_batch = build_sft_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        responses=chosen_responses,
        max_length=max_length,
    )

    rejected_batch = build_sft_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        responses=rejected_responses,
        max_length=max_length,
    )

    return {
        "chosen": chosen_batch,
        "rejected": rejected_batch,
    }

In [ ]:
from pathlib import Path

path = Path("submission_adapter.py")
text = path.read_text()

text = text.replace(
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
)""",
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
    build_dpo_batch_impl,
)"""
)

text = text.replace(
    '''def build_dpo_batch(
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    max_length,
):
    """Return separate response-masked LM batches for chosen and rejected rows."""

    raise NotImplementedError("DPO batching not implemented yet")''',
    '''def build_dpo_batch(
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    max_length,
):
    """Return separate response-masked LM batches for chosen and rejected rows."""

    return build_dpo_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        chosen_responses=chosen_responses,
        rejected_responses=rejected_responses,
        max_length=max_length,
    )'''
)

path.write_text(text)

print(path.read_text())

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

print(
    "build_dpo_batch_impl exists:",
    hasattr(alignment_core, "build_dpo_batch_impl"),
)

In [ ]:
!PYTHONPATH=. python -m pytest -q \
    tests/test_public_adapter_smoke.py \
    -k "dpo_batch"

In [ ]:
%%writefile -a alignment_core.py


def compute_dpo_loss_from_logps_impl(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta,
) -> torch.Tensor:
    """
    Compute the standard batch-mean DPO objective from per-example
    summed response log probabilities.

    For each example:

        policy_gap =
            log pi_policy(chosen | prompt)
            - log pi_policy(rejected | prompt)

        reference_gap =
            log pi_ref(chosen | prompt)
            - log pi_ref(rejected | prompt)

        loss =
            -log sigmoid(
                beta * (policy_gap - reference_gap)
            )
    """

    policy_gap = (
        policy_chosen_logps
        - policy_rejected_logps
    )

    reference_gap = (
        reference_chosen_logps
        - reference_rejected_logps
    )

    preference_logits = beta * (
        policy_gap - reference_gap
    )

    losses = -torch.nn.functional.logsigmoid(
        preference_logits
    )

    return losses.mean()

In [ ]:
from pathlib import Path

path = Path("submission_adapter.py")
text = path.read_text()

text = text.replace(
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
    build_dpo_batch_impl,
)""",
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
    build_dpo_batch_impl,
    compute_dpo_loss_from_logps_impl,
)"""
)

text = text.replace(
    '''def compute_dpo_loss_from_logps(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps,
    beta,
):
    """Return the scalar mean DPO objective from per-example log probabilities."""

    raise NotImplementedError("Tensor-level DPO loss not implemented yet")''',
    '''def compute_dpo_loss_from_logps(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps,
    beta,
):
    """Return the scalar mean DPO objective from per-example log probabilities."""

    return compute_dpo_loss_from_logps_impl(
        policy_chosen_logps=policy_chosen_logps,
        policy_rejected_logps=policy_rejected_logps,
        reference_chosen_logps=reference_chosen_logps,
        reference_rejected_logps=reference_rejected_logps,
        beta=beta,
    )'''
)

path.write_text(text)

print(path.read_text())

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

print(
    "compute_dpo_loss_from_logps_impl exists:",
    hasattr(
        alignment_core,
        "compute_dpo_loss_from_logps_impl",
    ),
)

In [ ]:
!PYTHONPATH=. python -m pytest -q \
    tests/test_public_adapter_smoke.py \
    -k "equal_logps"

In [ ]:
import torch
import math
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

from alignment_core import compute_dpo_loss_from_logps_impl


beta = 0.1

# --------------------------------------------------
# Case 1:
# Policy prefers chosen MORE strongly than reference.
# --------------------------------------------------
policy_chosen = torch.tensor([-2.0])
policy_rejected = torch.tensor([-5.0])

reference_chosen = torch.tensor([-3.0])
reference_rejected = torch.tensor([-4.0])

loss_better = compute_dpo_loss_from_logps_impl(
    policy_chosen,
    policy_rejected,
    reference_chosen,
    reference_rejected,
    beta=beta,
)

# policy gap = (-2) - (-5) = 3
# reference gap = (-3) - (-4) = 1
# difference = 2
# beta * difference = 0.2
expected_better = -torch.nn.functional.logsigmoid(
    torch.tensor(0.2)
)

print("Better-than-reference loss:", loss_better.item())
print("Expected:                  ", expected_better.item())

torch.testing.assert_close(loss_better, expected_better)

assert loss_better.item() < math.log(2)

print("✓ Stronger chosen preference gives loss < log(2)")


# --------------------------------------------------
# Case 2:
# Policy prefers chosen LESS strongly than reference.
# --------------------------------------------------
policy_chosen = torch.tensor([-3.0])
policy_rejected = torch.tensor([-4.0])

reference_chosen = torch.tensor([-2.0])
reference_rejected = torch.tensor([-5.0])

loss_worse = compute_dpo_loss_from_logps_impl(
    policy_chosen,
    policy_rejected,
    reference_chosen,
    reference_rejected,
    beta=beta,
)

print("\nWorse-than-reference loss:", loss_worse.item())

assert loss_worse.item() > math.log(2)

print("✓ Weaker chosen preference gives loss > log(2)")

In [ ]:
%%writefile -a alignment_core.py


def compute_dpo_loss_impl(
    policy_model,
    reference_model,
    tokenizer,
    prompts: Sequence[str],
    chosen_responses: Sequence[str],
    rejected_responses: Sequence[str],
    beta,
    max_length: int,
) -> torch.Tensor:
    """
    Compute the full DPO loss using a trainable policy model and
    a frozen reference model.

    Policy computations retain gradients.
    Reference computations are performed under torch.no_grad().
    """

    # -----------------------------
    # Policy log probabilities
    # -----------------------------
    policy_chosen_logps = compute_response_logprobs_impl(
        model=policy_model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=chosen_responses,
        max_length=max_length,
    )

    policy_rejected_logps = compute_response_logprobs_impl(
        model=policy_model,
        tokenizer=tokenizer,
        prompts=prompts,
        responses=rejected_responses,
        max_length=max_length,
    )

    # -----------------------------
    # Reference log probabilities
    #
    # IMPORTANT:
    # No gradients should ever flow
    # through the reference model.
    # -----------------------------
    with torch.no_grad():
        reference_chosen_logps = compute_response_logprobs_impl(
            model=reference_model,
            tokenizer=tokenizer,
            prompts=prompts,
            responses=chosen_responses,
            max_length=max_length,
        )

        reference_rejected_logps = compute_response_logprobs_impl(
            model=reference_model,
            tokenizer=tokenizer,
            prompts=prompts,
            responses=rejected_responses,
            max_length=max_length,
        )

    # The real Kaggle DPO workflow may place:
    #
    #   policy    -> cuda:0
    #   reference -> cuda:1
    #
    # Move the small reference log-probability tensors back to the
    # policy device before combining them.
    reference_chosen_logps = reference_chosen_logps.to(
        policy_chosen_logps.device
    )

    reference_rejected_logps = reference_rejected_logps.to(
        policy_rejected_logps.device
    )

    loss = compute_dpo_loss_from_logps_impl(
        policy_chosen_logps=policy_chosen_logps,
        policy_rejected_logps=policy_rejected_logps,
        reference_chosen_logps=reference_chosen_logps,
        reference_rejected_logps=reference_rejected_logps,
        beta=beta,
    )

    return loss

In [ ]:
from pathlib import Path

path = Path("submission_adapter.py")
text = path.read_text()

text = text.replace(
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
    build_dpo_batch_impl,
    compute_dpo_loss_from_logps_impl,
)""",
    """from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
    compute_sft_loss_impl,
    build_dpo_batch_impl,
    compute_dpo_loss_from_logps_impl,
    compute_dpo_loss_impl,
)"""
)

text = text.replace(
    '''def compute_dpo_loss(
    policy_model,
    reference_model,
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    beta,
    max_length,
):
    """Return a scalar mean DPO loss with a frozen reference computation."""

    raise NotImplementedError("DPO loss not implemented yet")''',
    '''def compute_dpo_loss(
    policy_model,
    reference_model,
    tokenizer,
    prompts,
    chosen_responses,
    rejected_responses,
    beta,
    max_length,
):
    """Return a scalar mean DPO loss with a frozen reference computation."""

    return compute_dpo_loss_impl(
        policy_model=policy_model,
        reference_model=reference_model,
        tokenizer=tokenizer,
        prompts=prompts,
        chosen_responses=chosen_responses,
        rejected_responses=rejected_responses,
        beta=beta,
        max_length=max_length,
    )'''
)

path.write_text(text)

print(path.read_text())

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

print(
    "compute_dpo_loss_impl exists:",
    hasattr(alignment_core, "compute_dpo_loss_impl"),
)

In [ ]:
!PYTHONPATH=. python -m pytest -q \
    tests/test_public_adapter_smoke.py \
    -k "dpo_loss_is_scalar"

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer))
print("Model max length:", tokenizer.model_max_length)

print("\n===== SPECIAL TOKENS =====")
print("PAD token:", repr(tokenizer.pad_token))
print("PAD token ID:", tokenizer.pad_token_id)

print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)

print("BOS token:", repr(tokenizer.bos_token))
print("BOS token ID:", tokenizer.bos_token_id)

print("UNK token:", repr(tokenizer.unk_token))
print("UNK token ID:", tokenizer.unk_token_id)

print("\n===== TOKENIZER SETTINGS =====")
print("Padding side:", tokenizer.padding_side)
print("Truncation side:", tokenizer.truncation_side)

print("\n===== CHAT TEMPLATE =====")
print("Has chat template:", tokenizer.chat_template is not None)

if tokenizer.chat_template is not None:
    print(tokenizer.chat_template)

In [ ]:
example_instruction = "Explain why the sky is blue."

messages = [
    {
        "role": "user",
        "content": example_instruction,
    }
]

chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("===== CHAT PROMPT repr =====")
print(repr(chat_prompt))

print("\n===== CHAT PROMPT rendered =====")
print(chat_prompt)

print("\n===== IMPORTANT SPECIAL TOKEN IDS =====")

for token in [
    "<|im_start|>",
    "<|im_end|>",
    "<|endoftext|>",
]:
    print(
        token,
        "->",
        tokenizer.convert_tokens_to_ids(token),
    )

print("\nEOS token ID:", tokenizer.eos_token_id)

print("\n===== LAST TOKENS OF CHAT PROMPT =====")

ids = tokenizer.encode(
    chat_prompt,
    add_special_tokens=False,
)

for token_id in ids[-20:]:
    print(
        token_id,
        repr(tokenizer.convert_ids_to_tokens(token_id)),
    )

In [ ]:
from datasets import get_dataset_split_names, load_dataset

SFT_DATASET_NAME = "HuggingFaceH4/ultrachat_200k"

print("Dataset:", SFT_DATASET_NAME)

split_names = get_dataset_split_names(SFT_DATASET_NAME)
print("Splits:", split_names)

print("\n===== FIRST 3 EXAMPLES FROM train_sft =====")

ds_preview = load_dataset(
    SFT_DATASET_NAME,
    split="train_sft",
    streaming=True,
)

for i, example in enumerate(ds_preview.take(3)):
    print(f"\n========== EXAMPLE {i} ==========")
    print("Keys:", example.keys())

    for key, value in example.items():
        print(f"\n{key}:")
        print(value)

In [ ]:
from datasets import load_dataset

SFT_DATASET_NAME = "HuggingFaceH4/ultrachat_200k"

ds_check = load_dataset(
    SFT_DATASET_NAME,
    split="train_sft",
    streaming=True,
)

checked = 0
valid = 0
bad_too_short = 0
bad_roles = 0
prompt_mismatch = 0
empty_instruction = 0
empty_response = 0

examples_of_problems = []

for example in ds_check.take(5000):
    checked += 1

    messages = example.get("messages", [])
    prompt = example.get("prompt", "")

    if len(messages) < 2:
        bad_too_short += 1
        if len(examples_of_problems) < 5:
            examples_of_problems.append(
                ("too_short", example)
            )
        continue

    if (
        messages[0].get("role") != "user"
        or messages[1].get("role") != "assistant"
    ):
        bad_roles += 1
        if len(examples_of_problems) < 5:
            examples_of_problems.append(
                ("bad_roles", messages[:2])
            )
        continue

    instruction = messages[0].get("content", "")
    response = messages[1].get("content", "")

    if not instruction.strip():
        empty_instruction += 1
        continue

    if not response.strip():
        empty_response += 1
        continue

    if prompt.strip() != instruction.strip():
        prompt_mismatch += 1
        if len(examples_of_problems) < 5:
            examples_of_problems.append(
                (
                    "prompt_mismatch",
                    {
                        "prompt": prompt,
                        "messages_0": instruction,
                    },
                )
            )

    valid += 1


print("===== ULTRACHAT STRUCTURE CHECK =====")
print("Checked:", checked)
print("Valid first user/assistant pairs:", valid)
print("Too short:", bad_too_short)
print("Bad first two roles:", bad_roles)
print("Empty instruction:", empty_instruction)
print("Empty response:", empty_response)
print("Prompt != first user message:", prompt_mismatch)

print("\n===== PROBLEM EXAMPLES =====")
for problem in examples_of_problems:
    print(problem)

In [ ]:
%%writefile data_utils.py

from __future__ import annotations


def extract_single_turn(example):
    """
    Convert an UltraChat-style conversation into one single-turn
    instruction-response example.

    Only the first user message and first assistant response are used.
    Later conversation turns are intentionally discarded.
    """

    messages = example.get("messages", [])

    if len(messages) < 2:
        return None

    if messages[0].get("role") != "user":
        return None

    if messages[1].get("role") != "assistant":
        return None

    instruction = messages[0].get("content", "").strip()
    response = messages[1].get("content", "").strip()

    if not instruction or not response:
        return None

    return {
        "instruction": instruction,
        "response": response,
        "source_id": example.get("prompt_id"),
    }


def format_instruction_prompt(instruction: str) -> str:
    """
    Shared prompt format for SFT, scratch DPO, and evaluation.
    """

    instruction = instruction.strip()

    return (
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
    )

In [ ]:
from datasets import load_dataset

from data_utils import (
    extract_single_turn,
    format_instruction_prompt,
)

preview = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
    streaming=True,
)

example = next(iter(preview))

single_turn = extract_single_turn(example)

print("===== INSTRUCTION =====")
print(single_turn["instruction"])

print("\n===== RESPONSE =====")
print(single_turn["response"])

print("\n===== FORMATTED MODEL PROMPT =====")
formatted_prompt = format_instruction_prompt(
    single_turn["instruction"]
)
print(repr(formatted_prompt))

print("\n===== SOURCE ID =====")
print(single_turn["source_id"])

In [ ]:
from datasets import load_dataset
import numpy as np
from tqdm.auto import tqdm

from data_utils import (
    extract_single_turn,
    format_instruction_prompt,
)


SEED = 42
CONTEXT_LENGTH = 512
N_ANALYZE = 40_000


dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
)

dataset = dataset.shuffle(seed=SEED)


prompt_lengths = []
response_lengths = []
total_lengths = []

valid_seen = 0
zero_response_after_truncation = 0
partial_response_after_truncation = 0
full_example_fits = 0


for example in tqdm(dataset, desc="Measuring lengths"):
    single_turn = extract_single_turn(example)

    if single_turn is None:
        continue

    prompt = format_instruction_prompt(
        single_turn["instruction"]
    )

    response = single_turn["response"]

    prompt_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    response_ids = tokenizer.encode(
        response,
        add_special_tokens=False,
    )

    # Our batch builder appends EOS before truncation.
    if tokenizer.eos_token_id is not None:
        response_ids = response_ids + [
            tokenizer.eos_token_id
        ]

    prompt_len = len(prompt_ids)
    response_len = len(response_ids)
    total_len = prompt_len + response_len

    prompt_lengths.append(prompt_len)
    response_lengths.append(response_len)
    total_lengths.append(total_len)

    if total_len <= CONTEXT_LENGTH:
        full_example_fits += 1

    response_tokens_surviving = max(
        0,
        CONTEXT_LENGTH - prompt_len,
    )

    response_tokens_surviving = min(
        response_len,
        response_tokens_surviving,
    )

    if response_tokens_surviving == 0:
        zero_response_after_truncation += 1

    elif response_tokens_surviving < response_len:
        partial_response_after_truncation += 1

    valid_seen += 1

    if valid_seen >= N_ANALYZE:
        break


def print_stats(name, values):
    values = np.asarray(values)

    print(f"\n===== {name} =====")
    print("Count:", len(values))
    print("Mean:", round(values.mean(), 2))
    print("Median:", round(np.median(values), 2))

    for percentile in [75, 90, 95, 99]:
        print(
            f"P{percentile}:",
            round(np.percentile(values, percentile), 2),
        )

    print("Max:", values.max())


print_stats("PROMPT TOKEN LENGTH", prompt_lengths)
print_stats("RESPONSE TOKEN LENGTH (INCLUDING EOS)", response_lengths)
print_stats("TOTAL TOKEN LENGTH", total_lengths)

print("\n===== 512-TOKEN CONTEXT BEHAVIOR =====")
print("Examples analyzed:", valid_seen)

print(
    "Full prompt + response fits:",
    full_example_fits,
    f"({100 * full_example_fits / valid_seen:.2f}%)",
)

print(
    "Response partially truncated:",
    partial_response_after_truncation,
    f"({100 * partial_response_after_truncation / valid_seen:.2f}%)",
)

print(
    "No response token survives:",
    zero_response_after_truncation,
    f"({100 * zero_response_after_truncation / valid_seen:.2f}%)",
)

In [ ]:
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm
import json
from pathlib import Path

from data_utils import (
    extract_single_turn,
    format_instruction_prompt,
)


SEED = 42
CONTEXT_LENGTH = 512

N_TRAIN = 30_000
N_VAL = 1_000

DATASET_NAME = "HuggingFaceH4/ultrachat_200k"


def example_fits_context(single_turn, tokenizer, max_length):
    """
    Return True only if the complete formatted prompt,
    response, and EOS fit inside max_length.
    """

    prompt = format_instruction_prompt(
        single_turn["instruction"]
    )

    prompt_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    response_ids = tokenizer.encode(
        single_turn["response"],
        add_special_tokens=False,
    )

    eos_extra = (
        1
        if tokenizer.eos_token_id is not None
        else 0
    )

    total_length = (
        len(prompt_ids)
        + len(response_ids)
        + eos_extra
    )

    return total_length <= max_length


def build_fixed_subset(
    split_name,
    n_examples,
    seed,
):
    """
    Shuffle deterministically, keep valid single-turn rows whose
    complete prompt+response+EOS fits in the context window,
    and stop after n_examples.
    """

    dataset = load_dataset(
        DATASET_NAME,
        split=split_name,
    )

    dataset = dataset.shuffle(seed=seed)

    selected = []

    scanned = 0
    malformed = 0
    too_long = 0

    for example in tqdm(
        dataset,
        desc=f"Selecting {split_name}",
    ):
        scanned += 1

        single_turn = extract_single_turn(example)

        if single_turn is None:
            malformed += 1
            continue

        if not example_fits_context(
            single_turn,
            tokenizer,
            CONTEXT_LENGTH,
        ):
            too_long += 1
            continue

        selected.append(single_turn)

        if len(selected) >= n_examples:
            break

    if len(selected) != n_examples:
        raise RuntimeError(
            f"Could only find {len(selected)} valid examples "
            f"for {split_name}; needed {n_examples}."
        )

    stats = {
        "source_dataset": DATASET_NAME,
        "source_split": split_name,
        "seed": seed,
        "context_length": CONTEXT_LENGTH,
        "selection_rule": (
            "first user message + first assistant response; "
            "full formatted prompt + response + EOS must fit "
            "within context length"
        ),
        "requested_examples": n_examples,
        "selected_examples": len(selected),
        "source_rows_scanned": scanned,
        "malformed_rows_skipped": malformed,
        "too_long_rows_skipped": too_long,
    }

    return Dataset.from_list(selected), stats


train_subset, train_stats = build_fixed_subset(
    split_name="train_sft",
    n_examples=N_TRAIN,
    seed=SEED,
)

val_subset, val_stats = build_fixed_subset(
    split_name="test_sft",
    n_examples=N_VAL,
    seed=SEED,
)


print("\n===== TRAIN STATS =====")
print(json.dumps(train_stats, indent=2))

print("\n===== VALIDATION STATS =====")
print(json.dumps(val_stats, indent=2))

print("\nTrain rows:", len(train_subset))
print("Validation rows:", len(val_subset))

print("\n===== FIRST TRAIN EXAMPLE =====")
print(train_subset[0])

In [ ]:
from pathlib import Path
import json
import hashlib


SFT_DATA_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/data/sft_fixed"
)

SFT_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TRAIN_PATH = SFT_DATA_DIR / "ultrachat_train_30000_seed42.jsonl"
VAL_PATH = SFT_DATA_DIR / "ultrachat_val_1000_seed42.jsonl"
META_PATH = SFT_DATA_DIR / "selection_metadata.json"
PROMPT_PATH = SFT_DATA_DIR / "prompt_format.txt"


# --------------------------------------------------
# Save exact selected rows
# --------------------------------------------------
train_subset.to_json(
    str(TRAIN_PATH),
    orient="records",
    lines=True,
    force_ascii=False,
)

val_subset.to_json(
    str(VAL_PATH),
    orient="records",
    lines=True,
    force_ascii=False,
)


# --------------------------------------------------
# Save prompt format explicitly
# --------------------------------------------------
PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Response:
"""

PROMPT_PATH.write_text(
    PROMPT_TEMPLATE,
    encoding="utf-8",
)


# --------------------------------------------------
# Compute SHA256 checksums
# --------------------------------------------------
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


train_sha256 = sha256_file(TRAIN_PATH)
val_sha256 = sha256_file(VAL_PATH)


metadata = {
    "dataset": "HuggingFaceH4/ultrachat_200k",

    "train": train_stats,
    "validation": val_stats,

    "single_turn_conversion": {
        "instruction": 'messages[0]["content"]',
        "response": 'messages[1]["content"]',
        "required_roles": [
            "user",
            "assistant",
        ],
        "later_turns_discarded": True,
    },

    "prompt_template": PROMPT_TEMPLATE,

    "context_length": 512,
    "seed": 42,

    "loss_reduction": (
        "mean negative log-probability over all "
        "unmasked response tokens in the batch"
    ),

    "files": {
        "train": {
            "path": str(TRAIN_PATH),
            "sha256": train_sha256,
        },
        "validation": {
            "path": str(VAL_PATH),
            "sha256": val_sha256,
        },
    },
}


META_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("===== SAVED SFT DATA =====")
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Metadata:", META_PATH)
print("Prompt format:", PROMPT_PATH)

print("\nTrain SHA256:")
print(train_sha256)

print("\nValidation SHA256:")
print(val_sha256)

print("\n===== FILE SIZES =====")
for path in [
    TRAIN_PATH,
    VAL_PATH,
    META_PATH,
    PROMPT_PATH,
]:
    print(
        path.name,
        f"{path.stat().st_size / (1024 ** 2):.2f} MB",
    )

In [ ]:
from datasets import load_dataset

saved_sft = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "validation": str(VAL_PATH),
    },
)

print(saved_sft)

assert len(saved_sft["train"]) == 30_000
assert len(saved_sft["validation"]) == 1_000

print("\n✓ Exact SFT subsets reload successfully")

In [ ]:
import torch
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SFT_DEVICE = torch.device("cuda:0")

torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

model = model.to(SFT_DEVICE)

# During training we do not need the KV cache.
model.config.use_cache = False

print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Parameter dtype:", next(model.parameters()).dtype)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nGPU 0 allocated:")
print(
    round(
        torch.cuda.memory_allocated(0) / 1024**3,
        3,
    ),
    "GB",
)

print("GPU 0 reserved:")
print(
    round(
        torch.cuda.memory_reserved(0) / 1024**3,
        3,
    ),
    "GB",
)

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(alignment_core)

from alignment_core import compute_sft_loss_impl
from data_utils import format_instruction_prompt


model.train()
model.zero_grad(set_to_none=True)

sample_rows = saved_sft["train"].select([0, 1])

sample_prompts = [
    format_instruction_prompt(row["instruction"])
    for row in sample_rows
]

sample_responses = [
    row["response"]
    for row in sample_rows
]

print("Prompt 0 token count:",
      len(tokenizer.encode(
          sample_prompts[0],
          add_special_tokens=False,
      )))

print("Response 0 token count:",
      len(tokenizer.encode(
          sample_responses[0],
          add_special_tokens=False,
      )) + 1)


torch.cuda.reset_peak_memory_stats(0)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    loss = compute_sft_loss_impl(
        model=model,
        tokenizer=tokenizer,
        prompts=sample_prompts,
        responses=sample_responses,
        max_length=512,
    )

print("\nSFT loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

loss.backward()

# Verify that real Qwen parameters received gradients.
grad_found = False
grad_norm_example = None

for parameter in model.parameters():
    if parameter.grad is not None:
        grad_found = True
        grad_norm_example = parameter.grad.detach().float().norm().item()
        if grad_norm_example > 0:
            break

print("Gradient found:", grad_found)
print("Example gradient norm:", grad_norm_example)

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated(0) / 1024**3,
        3,
    ),
    "GB",
)

# Clear test gradients before actual training.
model.zero_grad(set_to_none=True)

In [ ]:
import math
import random
import json
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import DataLoader

from data_utils import format_instruction_prompt


# ============================================================
# SFT CONFIGURATION
# ============================================================

SEED = 42

SFT_CONFIG = {
    "model_name": "Qwen/Qwen2.5-0.5B",

    "train_examples": 30_000,
    "validation_examples": 1_000,

    "context_length": 512,

    "per_device_batch_size": 2,
    "gradient_accumulation_steps": 16,

    "learning_rate": 2e-5,
    "warmup_ratio": 0.03,
    "scheduler": "cosine",

    "weight_decay": 0.1,
    "gradient_clip_norm": 1.0,

    "epochs": 1,

    "precision": "fp16_autocast",
    "trainable_weight_dtype": "float32",

    "seed": SEED,

    # Our implementation choice for monitoring/checkpointing.
    "eval_every_updates": 200,

    # Our documented SFT loss reduction.
    "loss_reduction": "response_token_mean",

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),
}


# ============================================================
# REPRODUCIBILITY SEEDS
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# COLLATOR
#
# Keep strings here.
# Our already-tested alignment_core implementation performs
# tokenization, response masking, EOS handling, and padding.
# ============================================================

def sft_collate_fn(rows):
    prompts = [
        format_instruction_prompt(row["instruction"])
        for row in rows
    ]

    responses = [
        row["response"]
        for row in rows
    ]

    return {
        "prompts": prompts,
        "responses": responses,
    }


# ============================================================
# DATA LOADERS
# ============================================================

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    saved_sft["train"],
    batch_size=SFT_CONFIG["per_device_batch_size"],
    shuffle=True,
    generator=train_generator,
    collate_fn=sft_collate_fn,
    num_workers=0,
    drop_last=False,
)

val_loader = DataLoader(
    saved_sft["validation"],
    batch_size=SFT_CONFIG["per_device_batch_size"],
    shuffle=False,
    collate_fn=sft_collate_fn,
    num_workers=0,
    drop_last=False,
)


# ============================================================
# STEP COUNTS
# ============================================================

micro_batches_per_epoch = len(train_loader)

total_update_steps = math.ceil(
    micro_batches_per_epoch
    / SFT_CONFIG["gradient_accumulation_steps"]
)

warmup_steps = int(
    SFT_CONFIG["warmup_ratio"]
    * total_update_steps
)

SFT_CONFIG["micro_batches_per_epoch"] = micro_batches_per_epoch
SFT_CONFIG["total_update_steps"] = total_update_steps
SFT_CONFIG["warmup_steps"] = warmup_steps

SFT_CONFIG["nominal_effective_batch_size"] = (
    SFT_CONFIG["per_device_batch_size"]
    * SFT_CONFIG["gradient_accumulation_steps"]
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

SFT_RUN_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/sft_qwen2.5_0.5b_ultrachat_seed42"
)

SFT_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONFIG_PATH = SFT_RUN_DIR / "config.json"

CONFIG_PATH.write_text(
    json.dumps(
        SFT_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# VERIFY EVERYTHING
# ============================================================

print("===== SFT CONFIG =====")
print(json.dumps(SFT_CONFIG, indent=2))

print("\n===== DATA LOADER =====")
print("Training examples:", len(saved_sft["train"]))
print("Validation examples:", len(saved_sft["validation"]))
print("Train micro-batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n===== OPTIMIZATION STEPS =====")
print("Gradient accumulation:", SFT_CONFIG["gradient_accumulation_steps"])
print(
    "Nominal effective batch size:",
    SFT_CONFIG["nominal_effective_batch_size"],
)
print("Total optimizer updates:", total_update_steps)
print("Warmup updates:", warmup_steps)

print("\nConfig saved to:")
print(CONFIG_PATH)


# Inspect one batch without touching the model.
first_batch = next(iter(train_loader))

print("\n===== SAMPLE BATCH =====")
print("Number of prompts:", len(first_batch["prompts"]))
print("Number of responses:", len(first_batch["responses"]))

print("\nPrompt 0:")
print(first_batch["prompts"][0][:500])

print("\nResponse 0:")
print(first_batch["responses"][0][:500])

In [ ]:
SFT_CONFIG.update({
    "optimizer": "AdamW",
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_epsilon": 1e-8,
})

CONFIG_PATH.write_text(
    json.dumps(
        SFT_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)

print(json.dumps(SFT_CONFIG, indent=2))

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup


optimizer = AdamW(
    model.parameters(),
    lr=SFT_CONFIG["learning_rate"],
    betas=(
        SFT_CONFIG["adam_beta1"],
        SFT_CONFIG["adam_beta2"],
    ),
    eps=SFT_CONFIG["adam_epsilon"],
    weight_decay=SFT_CONFIG["weight_decay"],
)


scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=SFT_CONFIG["warmup_steps"],
    num_training_steps=SFT_CONFIG["total_update_steps"],
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)


print("Optimizer:", type(optimizer).__name__)
print("Initial LR:", optimizer.param_groups[0]["lr"])
print("Warmup updates:", SFT_CONFIG["warmup_steps"])
print("Total scheduler updates:", SFT_CONFIG["total_update_steps"])
print("GradScaler enabled:", scaler.is_enabled())

In [ ]:
%%writefile sft_training.py

from __future__ import annotations

import math

import torch

from alignment_core import (
    build_sft_batch_impl,
    compute_response_logprobs_impl,
)


@torch.inference_mode()
def evaluate_sft(
    model,
    tokenizer,
    dataloader,
    max_length: int,
    max_batches: int | None = None,
):
    """
    Evaluate SFT using the exact documented response-token-mean
    negative log likelihood.

    Prompt tokens and padding do not contribute.
    """

    was_training = model.training
    model.eval()

    total_negative_log_likelihood = 0.0
    total_response_tokens = 0
    total_examples = 0
    batches_evaluated = 0

    for batch_index, batch in enumerate(dataloader):
        if (
            max_batches is not None
            and batch_index >= max_batches
        ):
            break

        prompts = batch["prompts"]
        responses = batch["responses"]

        # Build once here solely to count the response targets that
        # actually participate after causal shifting.
        lm_batch = build_sft_batch_impl(
            tokenizer=tokenizer,
            prompts=prompts,
            responses=responses,
            max_length=max_length,
        )

        response_token_count = (
            lm_batch["labels"][:, 1:]
            .ne(-100)
            .sum()
            .item()
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            response_logps = compute_response_logprobs_impl(
                model=model,
                tokenizer=tokenizer,
                prompts=prompts,
                responses=responses,
                max_length=max_length,
            )

        batch_nll = (
            -response_logps.detach().float().sum().item()
        )

        total_negative_log_likelihood += batch_nll
        total_response_tokens += response_token_count
        total_examples += len(prompts)
        batches_evaluated += 1

    if total_response_tokens == 0:
        raise RuntimeError(
            "Validation contained no scorable response tokens."
        )

    validation_loss = (
        total_negative_log_likelihood
        / total_response_tokens
    )

    perplexity = math.exp(validation_loss)

    if was_training:
        model.train()

    return {
        "validation_loss": validation_loss,
        "perplexity": perplexity,
        "examples": total_examples,
        "response_tokens": total_response_tokens,
        "batches": batches_evaluated,
    }

In [ ]:
import importlib
import sft_training

sft_training = importlib.reload(sft_training)

from sft_training import evaluate_sft


torch.cuda.reset_peak_memory_stats(0)

smoke_val = evaluate_sft(
    model=model,
    tokenizer=tokenizer,
    dataloader=val_loader,
    max_length=SFT_CONFIG["context_length"],
    max_batches=10,
)

print("===== VALIDATION SMOKE TEST =====")
print(json.dumps(smoke_val, indent=2))

print(
    "\nPeak GPU memory:",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

In [ ]:
import json
from huggingface_hub import hf_hub_download


ALPACA_REPO = "tatsu-lab/alpaca_eval"
ALPACA_FILENAME = "alpaca_eval.json"


# Download the official raw AlpacaEval evaluation file.
alpaca_json_path = hf_hub_download(
    repo_id=ALPACA_REPO,
    repo_type="dataset",
    filename=ALPACA_FILENAME,
)


with open(
    alpaca_json_path,
    "r",
    encoding="utf-8",
) as f:
    alpaca_eval_data = json.load(f)


print("File:")
print(alpaca_json_path)

print("\nNumber of examples:")
print(len(alpaca_eval_data))

print("\n===== FIRST 3 EXAMPLES =====")

for i, example in enumerate(alpaca_eval_data[:3]):
    print(f"\n========== EXAMPLE {i} ==========")
    print("Keys:", example.keys())

    for key, value in example.items():
        print(f"\n{key}:")
        print(value)

In [ ]:
import random
import json
import hashlib
from pathlib import Path


ALPACA_SEED = 42
ALPACA_N = 300

ALPACA_EVAL_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "eval_sets"
)

ALPACA_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ALPACA_SUBSET_PATH = (
    ALPACA_EVAL_DIR
    / "alpaca_eval_300_seed42.json"
)

ALPACA_META_PATH = (
    ALPACA_EVAL_DIR
    / "alpaca_eval_300_seed42_metadata.json"
)


# --------------------------------------------------
# Deterministically sample 300 distinct source rows.
# --------------------------------------------------

rng = random.Random(ALPACA_SEED)

sampled_indices = rng.sample(
    range(len(alpaca_eval_data)),
    ALPACA_N,
)


# Preserve the random sampled order exactly.
alpaca_eval_subset = []

for subset_position, source_index in enumerate(sampled_indices):
    source = alpaca_eval_data[source_index]

    alpaca_eval_subset.append({
        "id": f"alpaca_eval_{source_index:04d}",
        "source_index": source_index,
        "subset_position": subset_position,

        "instruction": source["instruction"],

        # Preserve source metadata for reproducibility.
        "source_dataset": source.get("dataset"),
        "reference_generator": source.get("generator"),
    })


# --------------------------------------------------
# Save exact fixed subset.
# --------------------------------------------------

with open(
    ALPACA_SUBSET_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        alpaca_eval_subset,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------
# SHA256 checksum.
# --------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


subset_sha256 = sha256_file(
    ALPACA_SUBSET_PATH
)


metadata = {
    "source_repository": "tatsu-lab/alpaca_eval",
    "source_file": "alpaca_eval.json",
    "source_examples": len(alpaca_eval_data),

    "subset_examples": ALPACA_N,
    "seed": ALPACA_SEED,

    "sampling_method": (
        "random.Random(seed).sample("
        "range(source_examples), subset_examples)"
    ),

    "order": "preserved random sample order",

    "subset_file": str(ALPACA_SUBSET_PATH),
    "sha256": subset_sha256,
}


with open(
    ALPACA_META_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("===== FIXED ALPACAEVAL SUBSET =====")
print("Source examples:", len(alpaca_eval_data))
print("Selected examples:", len(alpaca_eval_subset))
print("Seed:", ALPACA_SEED)

print("\nSubset path:")
print(ALPACA_SUBSET_PATH)

print("\nSHA256:")
print(subset_sha256)

print("\nFirst 10 sampled source indices:")
print(sampled_indices[:10])

print("\n===== FIRST 3 FIXED PROMPTS =====")

for row in alpaca_eval_subset[:3]:
    print("\n", row)

In [ ]:
with open(
    ALPACA_SUBSET_PATH,
    "r",
    encoding="utf-8",
) as f:
    reloaded_alpaca_subset = json.load(f)

assert len(reloaded_alpaca_subset) == 300

assert (
    sha256_file(ALPACA_SUBSET_PATH)
    == subset_sha256
)

assert [
    row["source_index"]
    for row in reloaded_alpaca_subset
] == sampled_indices

print("✓ Exact 300-prompt AlpacaEval subset reloads correctly")

In [ ]:
import torch

from data_utils import format_instruction_prompt


BASE_GENERATOR_NAME = "qwen2.5-0.5b-base"

GENERATION_CONFIG = {
    "max_new_tokens": 256,
    "do_sample": False,
}


# Generation benefits from KV caching.
old_use_cache = model.config.use_cache
model.config.use_cache = True

model.eval()


def generate_one(
    model,
    tokenizer,
    instruction,
    max_new_tokens=256,
):
    prompt = format_instruction_prompt(
        instruction
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded["input_ids"].to(SFT_DEVICE)
    attention_mask = encoded["attention_mask"].to(SFT_DEVICE)

    prompt_length = input_ids.shape[1]

    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    # Decode only newly generated tokens, not the prompt.
    new_tokens = generated[0, prompt_length:]

    output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return {
        "prompt_token_count": prompt_length,
        "generated_token_count": len(new_tokens),
        "output": output,
    }


print("===== BASELINE GENERATION SANITY CHECK =====")

sanity_outputs = []

for row in reloaded_alpaca_subset[:5]:
    result = generate_one(
        model=model,
        tokenizer=tokenizer,
        instruction=row["instruction"],
        max_new_tokens=GENERATION_CONFIG["max_new_tokens"],
    )

    sanity_outputs.append(result)

    print("\n" + "=" * 80)
    print("ID:", row["id"])

    print("\nINSTRUCTION:")
    print(row["instruction"])

    print("\nOUTPUT:")
    print(result["output"])

    print("\nPrompt tokens:",
          result["prompt_token_count"])

    print("Generated tokens:",
          result["generated_token_count"])


# Restore training setting afterward.
model.config.use_cache = old_use_cache

In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm
import torch

from data_utils import format_instruction_prompt


BASE_GENERATOR_NAME = "qwen2.5-0.5b-base"

BASE_EVAL_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/eval/base_qwen2.5_0.5b"
)

BASE_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BASE_PROGRESS_PATH = (
    BASE_EVAL_DIR
    / "alpaca_eval_300_seed42_base_progress.jsonl"
)

BASE_FINAL_PATH = (
    BASE_EVAL_DIR
    / "alpaca_eval_300_seed42_base.json"
)

BASE_GENERATION_CONFIG_PATH = (
    BASE_EVAL_DIR
    / "generation_config.json"
)


BASE_GENERATION_CONFIG = {
    "checkpoint": "Qwen/Qwen2.5-0.5B",
    "generator": BASE_GENERATOR_NAME,

    "evaluation_set": (
        "eval_sets/alpaca_eval_300_seed42.json"
    ),

    "evaluation_set_sha256": (
        "0e5763f8986c2149b4f690fe73225dcb"
        "71f0b125f36de6355200284276b772e8"
    ),

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "do_sample": False,
    "max_new_tokens": 256,

    "temperature": None,
    "top_p": None,

    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 0,

    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}


BASE_GENERATION_CONFIG_PATH.write_text(
    json.dumps(
        BASE_GENERATION_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# Load any already-completed generations.
# This makes the cell resumable.
# ============================================================

completed = {}

if BASE_PROGRESS_PATH.exists():
    with open(
        BASE_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                completed[row["id"]] = row

print("Already completed:", len(completed))


# ============================================================
# Generation
# ============================================================

old_use_cache = model.config.use_cache

model.config.use_cache = True
model.eval()


for row in tqdm(
    reloaded_alpaca_subset,
    desc="Generating base AlpacaEval",
):
    if row["id"] in completed:
        continue

    prompt = format_instruction_prompt(
        row["instruction"]
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded["input_ids"].to(SFT_DEVICE)

    attention_mask = (
        encoded["attention_mask"].to(SFT_DEVICE)
    )

    prompt_token_count = input_ids.shape[1]

    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_new_tokens=(
                BASE_GENERATION_CONFIG[
                    "max_new_tokens"
                ]
            ),

            do_sample=False,

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,

            use_cache=True,
        )

    new_tokens = generated[
        0,
        prompt_token_count:,
    ]

    output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    generated_token_count = len(new_tokens)

    result = {
        "id": row["id"],
        "instruction": row["instruction"],
        "output": output,

        "generator": BASE_GENERATOR_NAME,
        "dataset": "alpaca_eval",

        "source_index": row["source_index"],
        "subset_position": row["subset_position"],

        "prompt_token_count": prompt_token_count,
        "generated_token_count": (
            generated_token_count
        ),

        "hit_max_new_tokens": (
            generated_token_count
            >= BASE_GENERATION_CONFIG[
                "max_new_tokens"
            ]
        ),
    }

    # Append immediately so progress survives interruption.
    with open(
        BASE_PROGRESS_PATH,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            json.dumps(
                result,
                ensure_ascii=False,
            )
            + "\n"
        )

    completed[row["id"]] = result


model.config.use_cache = old_use_cache


# ============================================================
# Reconstruct final JSON in the exact fixed-prompt order.
# ============================================================

final_base_outputs = [
    completed[row["id"]]
    for row in reloaded_alpaca_subset
]

assert len(final_base_outputs) == 300


with open(
    BASE_FINAL_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_base_outputs,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("\n===== BASE GENERATION COMPLETE =====")
print("Examples:", len(final_base_outputs))
print("Final file:", BASE_FINAL_PATH)

print(
    "Hit max_new_tokens:",
    sum(
        row["hit_max_new_tokens"]
        for row in final_base_outputs
    ),
)

avg_generated_tokens = sum(
    row["generated_token_count"]
    for row in final_base_outputs
) / len(final_base_outputs)

print(
    "Average generated tokens:",
    round(avg_generated_tokens, 2),
)

In [ ]:
import json
import hashlib
from collections import Counter


# ============================================================
# Reload final base artifact
# ============================================================

with open(
    BASE_FINAL_PATH,
    "r",
    encoding="utf-8",
) as f:
    base_outputs_check = json.load(f)


assert len(base_outputs_check) == 300

assert [
    row["id"]
    for row in base_outputs_check
] == [
    row["id"]
    for row in reloaded_alpaca_subset
]


# ============================================================
# SHA256
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


base_sha256 = sha256_file(BASE_FINAL_PATH)

print("===== BASE ARTIFACT VERIFIED =====")
print("Rows:", len(base_outputs_check))
print("SHA256:", base_sha256)


# ============================================================
# Capped-output diagnostics
# ============================================================

capped = [
    row
    for row in base_outputs_check
    if row["hit_max_new_tokens"]
]

print("\n===== LENGTH DIAGNOSTICS =====")
print("Hit 256-token cap:", len(capped))
print(
    "Percentage:",
    round(100 * len(capped) / len(base_outputs_check), 2),
    "%",
)


def repeated_ngram_fraction(text, n=4):
    """
    Fraction of generated n-gram occurrences that are repetitions.
    0 means every n-gram occurs only once.
    """
    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]

    counts = Counter(ngrams)

    repeated_occurrences = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return repeated_occurrences / len(ngrams)


for row in capped:
    row["_repeat_4gram_fraction"] = repeated_ngram_fraction(
        row["output"],
        n=4,
    )


repeat_scores = [
    row["_repeat_4gram_fraction"]
    for row in capped
]

print(
    "Mean repeated 4-gram fraction among capped outputs:",
    round(sum(repeat_scores) / len(repeat_scores), 4),
)

print(
    "Capped outputs with >20% repeated 4-grams:",
    sum(score > 0.20 for score in repeat_scores),
)

print(
    "Capped outputs with >40% repeated 4-grams:",
    sum(score > 0.40 for score in repeat_scores),
)


# ============================================================
# Show capped examples:
# first five + five most repetitive
# ============================================================

print("\n===== FIRST 5 CAPPED OUTPUTS =====")

for row in capped[:5]:
    print("\n" + "=" * 80)
    print("ID:", row["id"])
    print(
        "Repeated 4-gram fraction:",
        round(row["_repeat_4gram_fraction"], 4),
    )

    print("\nInstruction:")
    print(row["instruction"][:500])

    print("\nLAST 1200 CHARS OF OUTPUT:")
    print(row["output"][-1200:])


print("\n===== 5 MOST REPETITIVE CAPPED OUTPUTS =====")

most_repetitive = sorted(
    capped,
    key=lambda x: x["_repeat_4gram_fraction"],
    reverse=True,
)[:5]

for row in most_repetitive:
    print("\n" + "=" * 80)
    print("ID:", row["id"])
    print(
        "Repeated 4-gram fraction:",
        round(row["_repeat_4gram_fraction"], 4),
    )

    print("\nInstruction:")
    print(row["instruction"][:500])

    print("\nLAST 1200 CHARS OF OUTPUT:")
    print(row["output"][-1200:])

In [ ]:
ANTI_REPEAT_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 256,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,
}


def generate_one_with_antirepeat(
    model,
    tokenizer,
    instruction,
):
    prompt = format_instruction_prompt(instruction)

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded["input_ids"].to(SFT_DEVICE)
    attention_mask = encoded["attention_mask"].to(SFT_DEVICE)

    prompt_length = input_ids.shape[1]

    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=ANTI_REPEAT_CONFIG["max_new_tokens"],
            do_sample=False,
            repetition_penalty=ANTI_REPEAT_CONFIG["repetition_penalty"],
            no_repeat_ngram_size=ANTI_REPEAT_CONFIG["no_repeat_ngram_size"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = generated[0, prompt_length:]

    output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return {
        "output": output,
        "generated_token_count": len(new_tokens),
        "repeat_4gram_fraction": repeated_ngram_fraction(
            output,
            n=4,
        ),
    }


print("===== ANTI-REPETITION PILOT =====")

for old_row in most_repetitive:
    new_result = generate_one_with_antirepeat(
        model=model,
        tokenizer=tokenizer,
        instruction=old_row["instruction"],
    )

    print("\n" + "=" * 80)
    print("ID:", old_row["id"])

    print(
        "OLD repeated 4-gram fraction:",
        round(old_row["_repeat_4gram_fraction"], 4),
    )

    print(
        "NEW repeated 4-gram fraction:",
        round(new_result["repeat_4gram_fraction"], 4),
    )

    print(
        "NEW generated tokens:",
        new_result["generated_token_count"],
    )

    print("\nNEW OUTPUT:")
    print(new_result["output"])
    

In [ ]:
import json
import hashlib
from pathlib import Path
from tqdm.auto import tqdm
import torch

from data_utils import format_instruction_prompt


FINAL_BASE_GENERATOR_NAME = "qwen2.5-0.5b-base"

FINAL_BASE_EVAL_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/eval/base_qwen2.5_0.5b"
)

FINAL_BASE_PROGRESS_PATH = (
    FINAL_BASE_EVAL_DIR
    / "alpaca_eval_300_seed42_base_nr4_progress.jsonl"
)

FINAL_BASE_PATH = (
    FINAL_BASE_EVAL_DIR
    / "alpaca_eval_300_seed42_base_nr4.json"
)

FINAL_BASE_CONFIG_PATH = (
    FINAL_BASE_EVAL_DIR
    / "generation_config_final.json"
)


FINAL_GENERATION_CONFIG = {
    "checkpoint": "Qwen/Qwen2.5-0.5B",
    "generator": FINAL_BASE_GENERATOR_NAME,

    "evaluation_set": (
        "eval_sets/alpaca_eval_300_seed42.json"
    ),

    "evaluation_set_sha256": (
        "0e5763f8986c2149b4f690fe73225dcb"
        "71f0b125f36de6355200284276b772e8"
    ),

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "do_sample": False,
    "max_new_tokens": 256,

    "temperature": None,
    "top_p": None,

    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,

    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,

    "decoding_change_reason": (
        "Initial greedy generation showed severe repetitive "
        "loops on some base-model outputs. A pilot on the five "
        "most repetitive cases showed that no_repeat_ngram_size=4 "
        "removed the loops while preserving coherent outputs."
    ),
}


FINAL_BASE_CONFIG_PATH.write_text(
    json.dumps(
        FINAL_GENERATION_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# Resume support
# ============================================================

completed_final = {}

if FINAL_BASE_PROGRESS_PATH.exists():
    with open(
        FINAL_BASE_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                completed_final[row["id"]] = row

print("Already completed:", len(completed_final))


# ============================================================
# Generate
# ============================================================

old_use_cache = model.config.use_cache

model.config.use_cache = True
model.eval()


for row in tqdm(
    reloaded_alpaca_subset,
    desc="Generating final base AlpacaEval",
):
    if row["id"] in completed_final:
        continue

    prompt = format_instruction_prompt(
        row["instruction"]
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded["input_ids"].to(SFT_DEVICE)
    attention_mask = encoded["attention_mask"].to(SFT_DEVICE)

    prompt_token_count = input_ids.shape[1]

    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_new_tokens=FINAL_GENERATION_CONFIG[
                "max_new_tokens"
            ],

            do_sample=False,

            repetition_penalty=FINAL_GENERATION_CONFIG[
                "repetition_penalty"
            ],

            no_repeat_ngram_size=FINAL_GENERATION_CONFIG[
                "no_repeat_ngram_size"
            ],

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,

            use_cache=True,
        )

    new_tokens = generated[
        0,
        prompt_token_count:,
    ]

    output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    generated_token_count = len(new_tokens)

    result = {
        "id": row["id"],
        "instruction": row["instruction"],
        "output": output,

        "generator": FINAL_BASE_GENERATOR_NAME,
        "dataset": "alpaca_eval",

        "source_index": row["source_index"],
        "subset_position": row["subset_position"],

        "prompt_token_count": prompt_token_count,
        "generated_token_count": generated_token_count,

        "hit_max_new_tokens": (
            generated_token_count
            >= FINAL_GENERATION_CONFIG["max_new_tokens"]
        ),
    }

    with open(
        FINAL_BASE_PROGRESS_PATH,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            json.dumps(
                result,
                ensure_ascii=False,
            )
            + "\n"
        )

    completed_final[row["id"]] = result


model.config.use_cache = old_use_cache


# ============================================================
# Reconstruct exact final order
# ============================================================

final_base_outputs_nr4 = [
    completed_final[row["id"]]
    for row in reloaded_alpaca_subset
]

assert len(final_base_outputs_nr4) == 300


with open(
    FINAL_BASE_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_base_outputs_nr4,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# Diagnostics
# ============================================================

def repeated_ngram_fraction(text, n=4):
    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]

    from collections import Counter

    counts = Counter(ngrams)

    repeated_occurrences = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return repeated_occurrences / len(ngrams)


repeat_scores_final = [
    repeated_ngram_fraction(
        row["output"],
        n=4,
    )
    for row in final_base_outputs_nr4
]


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


final_base_sha256 = sha256_file(
    FINAL_BASE_PATH
)


print("\n===== FINAL BASE GENERATION COMPLETE =====")

print("Examples:", len(final_base_outputs_nr4))

print(
    "Hit max_new_tokens:",
    sum(
        row["hit_max_new_tokens"]
        for row in final_base_outputs_nr4
    ),
)

print(
    "Average generated tokens:",
    round(
        sum(
            row["generated_token_count"]
            for row in final_base_outputs_nr4
        )
        / len(final_base_outputs_nr4),
        2,
    ),
)

print(
    "Mean repeated 4-gram fraction:",
    round(
        sum(repeat_scores_final)
        / len(repeat_scores_final),
        4,
    ),
)

print(
    "Outputs with >20% repeated 4-grams:",
    sum(
        score > 0.20
        for score in repeat_scores_final
    ),
)

print(
    "Outputs with >40% repeated 4-grams:",
    sum(
        score > 0.40
        for score in repeat_scores_final
    ),
)

print("\nFinal file:")
print(FINAL_BASE_PATH)

print("\nSHA256:")
print(final_base_sha256)

In [ ]:
%%writefile sft_training.py

from __future__ import annotations

import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from alignment_core import build_sft_batch_impl


# ============================================================
# SMALL FILE UTILITIES
# ============================================================

def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
        )


# ============================================================
# BUILD ONE SFT MICRO-BATCH
# ============================================================

def prepare_sft_batch_cpu(
    tokenizer,
    prompts,
    responses,
    max_length,
):
    """
    Uses the already-tested assignment implementation to produce:
      input_ids
      attention_mask
      labels

    Then removes only trailing all-padding columns.

    Returned tensors remain on CPU.
    """

    batch = build_sft_batch_impl(
        tokenizer=tokenizer,
        prompts=prompts,
        responses=responses,
        max_length=max_length,
    )

    if batch["input_ids"].shape[0] == 0:
        raise ValueError("Empty SFT batch.")

    # Longest real sequence in this micro-batch.
    used_length = int(
        batch["attention_mask"]
        .sum(dim=1)
        .max()
        .item()
    )

    # Causal LM needs at least two positions to form a shift.
    used_length = max(used_length, 2)

    batch = {
        key: value[:, :used_length].contiguous()
        for key, value in batch.items()
    }

    # After causal shifting, these are the actual response
    # targets contributing to the loss.
    response_token_count = int(
        batch["labels"][:, 1:]
        .ne(-100)
        .sum()
        .item()
    )

    if response_token_count <= 0:
        raise RuntimeError(
            "SFT micro-batch has no scorable response tokens."
        )

    return batch, response_token_count


def move_batch_to_device(batch, device):
    return {
        key: value.to(
            device,
            non_blocking=True,
        )
        for key, value in batch.items()
    }


# ============================================================
# SUMMED RESPONSE NLL
# ============================================================

def response_nll_sum(
    model,
    batch,
):
    """
    Return SUM of negative log probabilities over response
    targets only.

    labels already contain -100 for:
      - prompt tokens
      - padding

    We explicitly apply the causal one-token shift.
    """

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        use_cache=False,
    )

    logits = outputs.logits

    shift_logits = logits[:, :-1, :]
    shift_labels = batch["labels"][:, 1:]

    nll_sum = F.cross_entropy(
        shift_logits.reshape(
            -1,
            shift_logits.shape[-1],
        ),
        shift_labels.reshape(-1),
        ignore_index=-100,
        reduction="sum",
    )

    return nll_sum


# ============================================================
# VALIDATION
# ============================================================

@torch.inference_mode()
def evaluate_sft(
    model,
    tokenizer,
    dataloader,
    max_length,
    device,
    max_batches=None,
):
    """
    Exact response-token-mean validation NLL.
    """

    was_training = model.training

    model.eval()

    total_nll = 0.0
    total_response_tokens = 0
    total_examples = 0
    batches_evaluated = 0

    try:
        for batch_index, batch in enumerate(dataloader):

            if (
                max_batches is not None
                and batch_index >= max_batches
            ):
                break

            cpu_batch, token_count = (
                prepare_sft_batch_cpu(
                    tokenizer=tokenizer,
                    prompts=batch["prompts"],
                    responses=batch["responses"],
                    max_length=max_length,
                )
            )

            gpu_batch = move_batch_to_device(
                cpu_batch,
                device,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                nll_sum = response_nll_sum(
                    model,
                    gpu_batch,
                )

            total_nll += float(
                nll_sum.detach().float().item()
            )

            total_response_tokens += token_count
            total_examples += len(batch["prompts"])
            batches_evaluated += 1

            del gpu_batch
            del cpu_batch
            del nll_sum

    finally:
        if was_training:
            model.train()

    if total_response_tokens <= 0:
        raise RuntimeError(
            "Validation contained no response tokens."
        )

    validation_loss = (
        total_nll
        / total_response_tokens
    )

    # Should be easily finite for this run.
    perplexity = (
        math.exp(validation_loss)
        if validation_loss < 50
        else float("inf")
    )

    return {
        "validation_loss": validation_loss,
        "perplexity": perplexity,
        "examples": total_examples,
        "response_tokens": total_response_tokens,
        "batches": batches_evaluated,
    }


# ============================================================
# CHECKPOINT SAVING
# ============================================================

def save_sft_checkpoint(
    model,
    tokenizer,
    checkpoint_dir,
    metadata,
):
    checkpoint_dir = Path(checkpoint_dir)

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        checkpoint_dir,
        safe_serialization=True,
    )

    tokenizer.save_pretrained(
        checkpoint_dir,
    )

    write_json(
        checkpoint_dir
        / "checkpoint_metadata.json",
        metadata,
    )


# ============================================================
# FULL ONE-EPOCH SFT TRAINING LOOP
# ============================================================

def train_sft(
    *,
    model,
    tokenizer,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    scaler,
    config,
    run_dir,
    device,
):
    run_dir = Path(run_dir)

    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        run_dir / "metrics.jsonl"
    )

    summary_path = (
        run_dir / "training_summary.json"
    )

    best_checkpoint_dir = (
        run_dir / "best_checkpoint"
    )

    final_checkpoint_dir = (
        run_dir / "final_checkpoint"
    )

    # Avoid accidentally mixing metrics from two complete runs.
    if metrics_path.exists():
        raise FileExistsError(
            f"{metrics_path} already exists. "
            "Use a new run directory or explicitly remove it "
            "before beginning a fresh training run."
        )

    accumulation_steps = int(
        config["gradient_accumulation_steps"]
    )

    max_length = int(
        config["context_length"]
    )

    clip_norm = float(
        config["gradient_clip_norm"]
    )

    eval_every = int(
        config["eval_every_updates"]
    )

    expected_updates = int(
        config["total_update_steps"]
    )

    model.config.use_cache = False

    optimizer.zero_grad(
        set_to_none=True,
    )

    # --------------------------------------------------------
    # Full validation at update 0.
    #
    # This is useful for the learning curve, but does NOT
    # participate in SFT checkpoint selection.
    # --------------------------------------------------------

    print("Running full pre-training validation...")

    initial_val = evaluate_sft(
        model=model,
        tokenizer=tokenizer,
        dataloader=val_loader,
        max_length=max_length,
        device=device,
    )

    append_jsonl(
        metrics_path,
        {
            "type": "validation",
            "update": 0,
            **initial_val,
        },
    )

    print(
        "Initial validation loss:",
        initial_val["validation_loss"],
    )

    print(
        "Initial validation perplexity:",
        initial_val["perplexity"],
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    best_validation_loss = float("inf")
    best_update = None

    update = 0
    skipped_optimizer_steps = 0

    total_train_nll = 0.0
    total_train_tokens = 0

    training_start_time = time.time()

    progress = tqdm(
        total=expected_updates,
        desc="SFT optimizer updates",
    )

    # We buffer up to 16 micro-batches so we know the exact
    # number of response tokens in the whole accumulation
    # window BEFORE computing its normalized gradient.
    window = []

    last_eval_update = 0

    for microbatch_index, raw_batch in enumerate(
        train_loader,
        start=1,
    ):

        cpu_batch, token_count = (
            prepare_sft_batch_cpu(
                tokenizer=tokenizer,
                prompts=raw_batch["prompts"],
                responses=raw_batch["responses"],
                max_length=max_length,
            )
        )

        window.append(
            (
                cpu_batch,
                token_count,
            )
        )

        is_last_microbatch = (
            microbatch_index
            == len(train_loader)
        )

        window_is_full = (
            len(window)
            == accumulation_steps
        )

        if not (
            window_is_full
            or is_last_microbatch
        ):
            continue

        # ====================================================
        # One optimizer update
        # ====================================================

        window_response_tokens = sum(
            token_count
            for _, token_count in window
        )

        window_nll_value = 0.0

        optimizer.zero_grad(
            set_to_none=True,
        )

        for cpu_microbatch, _ in window:

            gpu_microbatch = move_batch_to_device(
                cpu_microbatch,
                device,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                nll_sum = response_nll_sum(
                    model,
                    gpu_microbatch,
                )

                # IMPORTANT:
                # Normalize every microbatch by the number of
                # response tokens in the ENTIRE accumulation
                # window.
                #
                # Summing gradients across all microbatches
                # therefore gives exactly:
                #
                # total response NLL / total response tokens
                #
                loss_piece = (
                    nll_sum
                    / window_response_tokens
                )

            scaler.scale(
                loss_piece
            ).backward()

            window_nll_value += float(
                nll_sum.detach().float().item()
            )

            del gpu_microbatch
            del nll_sum
            del loss_piece

        # Convert scaled gradients back to ordinary gradients
        # before clipping.
        scaler.unscale_(
            optimizer
        )

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=clip_norm,
        )

        # LR actually used for this attempted update.
        lr_used = float(
            optimizer.param_groups[0]["lr"]
        )

        scale_before = scaler.get_scale()

        scaler.step(
            optimizer
        )

        scaler.update()

        scale_after = scaler.get_scale()

        # If GradScaler encountered inf/nan gradients, it skips
        # optimizer.step and reduces the scale.
        optimizer_step_succeeded = (
            scale_after >= scale_before
        )

        window_train_loss = (
            window_nll_value
            / window_response_tokens
        )

        if optimizer_step_succeeded:

            scheduler.step()

            update += 1

            total_train_nll += window_nll_value
            total_train_tokens += (
                window_response_tokens
            )

            record = {
                "type": "train",
                "update": update,
                "microbatch_end": (
                    microbatch_index
                ),
                "microbatches_in_window": (
                    len(window)
                ),
                "response_tokens": (
                    window_response_tokens
                ),
                "train_loss": (
                    window_train_loss
                ),
                "lr_used": lr_used,
                "lr_next": float(
                    optimizer
                    .param_groups[0]["lr"]
                ),
                "grad_norm_before_clip": float(
                    grad_norm.detach()
                    .float()
                    .item()
                ),
                "grad_scale": float(
                    scale_after
                ),
            }

            append_jsonl(
                metrics_path,
                record,
            )

            progress.update(1)

            progress.set_postfix(
                loss=f"{window_train_loss:.4f}",
                lr=f"{lr_used:.2e}",
            )

        else:

            skipped_optimizer_steps += 1

            append_jsonl(
                metrics_path,
                {
                    "type": "skipped_update",
                    "microbatch_end": (
                        microbatch_index
                    ),
                    "reason": (
                        "GradScaler detected "
                        "non-finite gradients"
                    ),
                    "scale_before": float(
                        scale_before
                    ),
                    "scale_after": float(
                        scale_after
                    ),
                },
            )

        # Free buffered CPU tensors.
        window.clear()

        # ----------------------------------------------------
        # Periodic validation
        # ----------------------------------------------------

        should_validate = (
            optimizer_step_succeeded
            and update > 0
            and update % eval_every == 0
        )

        if should_validate:

            validation = evaluate_sft(
                model=model,
                tokenizer=tokenizer,
                dataloader=val_loader,
                max_length=max_length,
                device=device,
            )

            append_jsonl(
                metrics_path,
                {
                    "type": "validation",
                    "update": update,
                    **validation,
                },
            )

            last_eval_update = update

            print(
                f"\nUpdate {update}: "
                f"validation loss = "
                f"{validation['validation_loss']:.6f}"
            )

            if (
                validation["validation_loss"]
                < best_validation_loss
            ):

                best_validation_loss = (
                    validation[
                        "validation_loss"
                    ]
                )

                best_update = update

                print(
                    "Saving new best checkpoint "
                    f"at update {update}..."
                )

                save_sft_checkpoint(
                    model=model,
                    tokenizer=tokenizer,
                    checkpoint_dir=(
                        best_checkpoint_dir
                    ),
                    metadata={
                        "checkpoint_type": "best",
                        "update": update,
                        "validation_loss": (
                            best_validation_loss
                        ),
                        "selection_criterion": (
                            "lowest validation loss"
                        ),
                    },
                )

            model.train()

    progress.close()

    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------

    if last_eval_update != update:

        print(
            "\nRunning final validation "
            f"at update {update}..."
        )

        final_validation = evaluate_sft(
            model=model,
            tokenizer=tokenizer,
            dataloader=val_loader,
            max_length=max_length,
            device=device,
        )

        append_jsonl(
            metrics_path,
            {
                "type": "validation",
                "update": update,
                **final_validation,
            },
        )

    else:
        # If the final update happened to coincide exactly
        # with an evaluation boundary, recover that validation
        # result by evaluating once more only for simplicity
        # and an explicit final summary.
        final_validation = evaluate_sft(
            model=model,
            tokenizer=tokenizer,
            dataloader=val_loader,
            max_length=max_length,
            device=device,
        )

        append_jsonl(
            metrics_path,
            {
                "type": "validation_final_repeat",
                "update": update,
                **final_validation,
            },
        )

    # The final checkpoint can itself be the best.
    if (
        final_validation["validation_loss"]
        < best_validation_loss
    ):

        best_validation_loss = (
            final_validation[
                "validation_loss"
            ]
        )

        best_update = update

        print(
            "Final checkpoint is also the "
            "new best checkpoint."
        )

        save_sft_checkpoint(
            model=model,
            tokenizer=tokenizer,
            checkpoint_dir=(
                best_checkpoint_dir
            ),
            metadata={
                "checkpoint_type": "best",
                "update": update,
                "validation_loss": (
                    best_validation_loss
                ),
                "selection_criterion": (
                    "lowest validation loss"
                ),
            },
        )

    # --------------------------------------------------------
    # ALWAYS SAVE FINAL CHECKPOINT
    # --------------------------------------------------------

    print("Saving final checkpoint...")

    save_sft_checkpoint(
        model=model,
        tokenizer=tokenizer,
        checkpoint_dir=(
            final_checkpoint_dir
        ),
        metadata={
            "checkpoint_type": "final",
            "update": update,
            "validation_loss": (
                final_validation[
                    "validation_loss"
                ]
            ),
        },
    )

    elapsed_seconds = (
        time.time()
        - training_start_time
    )

    overall_train_loss = (
        total_train_nll
        / total_train_tokens
    )

    summary = {
        "completed": True,
        "optimizer_updates": update,
        "expected_optimizer_updates": (
            expected_updates
        ),
        "skipped_optimizer_steps": (
            skipped_optimizer_steps
        ),
        "overall_training_loss": (
            overall_train_loss
        ),
        "initial_validation": initial_val,
        "final_validation": final_validation,
        "best_validation_loss": (
            best_validation_loss
        ),
        "best_update": best_update,
        "elapsed_seconds": elapsed_seconds,
        "elapsed_hours": (
            elapsed_seconds / 3600.0
        ),
        "best_checkpoint": str(
            best_checkpoint_dir
        ),
        "final_checkpoint": str(
            final_checkpoint_dir
        ),
        "metrics_file": str(
            metrics_path
        ),
    }

    write_json(
        summary_path,
        summary,
    )

    return summary

In [ ]:
import importlib
import json
import torch

import sft_training

sft_training = importlib.reload(
    sft_training
)

from sft_training import (
    prepare_sft_batch_cpu,
    evaluate_sft,
    train_sft,
)


# ------------------------------------------------------------
# Inspect one dynamically cropped batch
# ------------------------------------------------------------

test_raw_batch = next(
    iter(val_loader)
)

test_tensor_batch, test_response_tokens = (
    prepare_sft_batch_cpu(
        tokenizer=tokenizer,
        prompts=test_raw_batch["prompts"],
        responses=test_raw_batch["responses"],
        max_length=SFT_CONFIG[
            "context_length"
        ],
    )
)

print("===== DYNAMIC BATCH CHECK =====")
print(
    "input_ids shape:",
    tuple(
        test_tensor_batch[
            "input_ids"
        ].shape
    ),
)

print(
    "attention_mask shape:",
    tuple(
        test_tensor_batch[
            "attention_mask"
        ].shape
    ),
)

print(
    "labels shape:",
    tuple(
        test_tensor_batch[
            "labels"
        ].shape
    ),
)

print(
    "Response tokens:",
    test_response_tokens,
)


# ------------------------------------------------------------
# Non-destructive 10-batch validation test
# ------------------------------------------------------------

torch.cuda.reset_peak_memory_stats(0)

new_smoke_val = evaluate_sft(
    model=model,
    tokenizer=tokenizer,
    dataloader=val_loader,
    max_length=SFT_CONFIG[
        "context_length"
    ],
    device=SFT_DEVICE,
    max_batches=10,
)

print(
    "\n===== NEW VALIDATION SMOKE ====="
)

print(
    json.dumps(
        new_smoke_val,
        indent=2,
    )
)

print(
    "\nPeak GPU memory:",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

In [ ]:
import os
import shutil
import json
import torch
from pathlib import Path


print("===== SFT FINAL PREFLIGHT =====")


# ============================================================
# 1. MODEL STATE
# ============================================================

print("\n[MODEL]")

print(
    "Device:",
    next(model.parameters()).device,
)

print(
    "Parameter dtype:",
    next(model.parameters()).dtype,
)

print(
    "Training mode:",
    model.training,
)

print(
    "use_cache:",
    model.config.use_cache,
)

assert next(model.parameters()).device.type == "cuda"
assert next(model.parameters()).device.index == 0
assert next(model.parameters()).dtype == torch.float32


# ============================================================
# 2. OPTIMIZER / SCHEDULER / SCALER
# ============================================================

print("\n[OPTIMIZATION]")

print(
    "Current optimizer LR:",
    optimizer.param_groups[0]["lr"],
)

print(
    "Configured peak LR:",
    SFT_CONFIG["learning_rate"],
)

print(
    "Scheduler last_epoch:",
    scheduler.last_epoch,
)

print(
    "GradScaler scale:",
    scaler.get_scale(),
)

print(
    "GradScaler enabled:",
    scaler.is_enabled(),
)

assert scaler.is_enabled()


# ============================================================
# 3. DATA COUNTS
# ============================================================

print("\n[DATA]")

print(
    "Train examples:",
    len(saved_sft["train"]),
)

print(
    "Validation examples:",
    len(saved_sft["validation"]),
)

print(
    "Train micro-batches:",
    len(train_loader),
)

print(
    "Expected optimizer updates:",
    SFT_CONFIG["total_update_steps"],
)

assert len(saved_sft["train"]) == 30000
assert len(saved_sft["validation"]) == 1000
assert len(train_loader) == 15000
assert SFT_CONFIG["total_update_steps"] == 938


# ============================================================
# 4. OUTPUT PATH CHECK
# ============================================================

print("\n[OUTPUTS]")

METRICS_PATH = (
    SFT_RUN_DIR
    / "metrics.jsonl"
)

BEST_DIR = (
    SFT_RUN_DIR
    / "best_checkpoint"
)

FINAL_DIR = (
    SFT_RUN_DIR
    / "final_checkpoint"
)

SUMMARY_PATH = (
    SFT_RUN_DIR
    / "training_summary.json"
)

print("Run directory:")
print(SFT_RUN_DIR)

print(
    "metrics.jsonl exists:",
    METRICS_PATH.exists(),
)

print(
    "best_checkpoint exists:",
    BEST_DIR.exists(),
)

print(
    "final_checkpoint exists:",
    FINAL_DIR.exists(),
)

print(
    "training_summary exists:",
    SUMMARY_PATH.exists(),
)


# We have not trained yet, so metrics must not already exist.
assert not METRICS_PATH.exists(), (
    "metrics.jsonl already exists. "
    "Do NOT start until we inspect why."
)


# ============================================================
# 5. DISK SPACE
# ============================================================

disk = shutil.disk_usage(
    "/kaggle/working"
)

print("\n[DISK]")

print(
    "Free space:",
    round(
        disk.free / 1024**3,
        2,
    ),
    "GB",
)

print(
    "Used space:",
    round(
        disk.used / 1024**3,
        2,
    ),
    "GB",
)


# Conservative minimum because we will save:
#   best checkpoint
#   final checkpoint
# and other later DPO artifacts.
assert (
    disk.free / 1024**3
) >= 6.0, (
    "Less than 6 GB free. "
    "Do not begin training yet."
)


# ============================================================
# 6. GPU MEMORY
# ============================================================

print("\n[GPU]")

for gpu_id in range(torch.cuda.device_count()):
    free_bytes, total_bytes = torch.cuda.mem_get_info(
        gpu_id
    )

    print(
        f"GPU {gpu_id}:",
        f"{free_bytes / 1024**3:.2f} GB free /",
        f"{total_bytes / 1024**3:.2f} GB total",
    )


# ============================================================
# 7. BASELINE ARTIFACT
# ============================================================

print("\n[BASE EVALUATION]")

print(
    "Final base file exists:",
    FINAL_BASE_PATH.exists(),
)

print(
    "Final base SHA256:",
    sha256_file(FINAL_BASE_PATH),
)

assert FINAL_BASE_PATH.exists()

assert (
    sha256_file(FINAL_BASE_PATH)
    ==
    "1467f8f6165a68d7a26794fdd5c3476b2b6c74db05007f5877db383b755c6538"
)


print("\n✓ PRE-FLIGHT PASSED")
print("No model update has been performed yet.")

In [ ]:
import json
import time
import torch

from torch.utils.data import DataLoader

from sft_training import train_sft


# ============================================================
# RECREATE THE SHUFFLED TRAIN LOADER FROM A FRESH SEED
# ============================================================

fresh_train_generator = torch.Generator()
fresh_train_generator.manual_seed(
    SFT_CONFIG["seed"]
)

train_loader = DataLoader(
    saved_sft["train"],
    batch_size=SFT_CONFIG[
        "per_device_batch_size"
    ],
    shuffle=True,
    generator=fresh_train_generator,
    collate_fn=sft_collate_fn,
    num_workers=0,
    drop_last=False,
)


assert len(train_loader) == 15000


print("===== STARTING REAL SFT TRAINING =====")
print("Training examples:", len(saved_sft["train"]))
print("Validation examples:", len(saved_sft["validation"]))
print("Micro-batches:", len(train_loader))
print(
    "Expected optimizer updates:",
    SFT_CONFIG["total_update_steps"],
)
print(
    "Effective batch size:",
    SFT_CONFIG[
        "nominal_effective_batch_size"
    ],
)
print("Run directory:")
print(SFT_RUN_DIR)


# ============================================================
# MEMORY STATS BEFORE TRAINING
# ============================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

start_wall_time = time.time()


# ============================================================
# ACTUAL TRAINING
# ============================================================

sft_summary = train_sft(
    model=model,
    tokenizer=tokenizer,

    train_loader=train_loader,
    val_loader=val_loader,

    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,

    config=SFT_CONFIG,

    run_dir=SFT_RUN_DIR,
    device=SFT_DEVICE,
)


# ============================================================
# SUMMARY
# ============================================================

elapsed_wall = (
    time.time()
    - start_wall_time
)

print("\n" + "=" * 80)
print("===== SFT TRAINING FINISHED =====")

print(
    json.dumps(
        sft_summary,
        indent=2,
    )
)

print(
    "\nWall-clock hours:",
    round(
        elapsed_wall / 3600,
        3,
    ),
)

print(
    "Peak GPU 0 memory:",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print("\nBest checkpoint:")
print(
    sft_summary["best_checkpoint"]
)

print("\nFinal checkpoint:")
print(
    sft_summary["final_checkpoint"]
)

In [ ]:
import gc
import json
import hashlib
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

from data_utils import format_instruction_prompt


SFT_BEST_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/sft_qwen2.5_0.5b_ultrachat_seed42/"
    "best_checkpoint"
)

SFT_FINAL_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/sft_qwen2.5_0.5b_ultrachat_seed42/"
    "final_checkpoint"
)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ============================================================
# VERIFY CHECKPOINT FILES
# ============================================================

best_model_file = (
    SFT_BEST_DIR / "model.safetensors"
)

final_model_file = (
    SFT_FINAL_DIR / "model.safetensors"
)

best_metadata_file = (
    SFT_BEST_DIR
    / "checkpoint_metadata.json"
)

final_metadata_file = (
    SFT_FINAL_DIR
    / "checkpoint_metadata.json"
)


assert best_model_file.exists()
assert final_model_file.exists()
assert best_metadata_file.exists()
assert final_metadata_file.exists()


with open(
    best_metadata_file,
    "r",
    encoding="utf-8",
) as f:
    best_metadata = json.load(f)

with open(
    final_metadata_file,
    "r",
    encoding="utf-8",
) as f:
    final_metadata = json.load(f)


print("===== SFT CHECKPOINT VERIFICATION =====")

print("\nBest metadata:")
print(
    json.dumps(
        best_metadata,
        indent=2,
    )
)

print("\nFinal metadata:")
print(
    json.dumps(
        final_metadata,
        indent=2,
    )
)


print("\nComputing model hashes...")

best_model_sha256 = sha256_file(
    best_model_file
)

final_model_sha256 = sha256_file(
    final_model_file
)


print("\nBest model SHA256:")
print(best_model_sha256)

print("\nFinal model SHA256:")
print(final_model_sha256)

print(
    "\nModel weights identical:",
    best_model_sha256
    == final_model_sha256,
)

assert best_metadata["update"] == 938
assert final_metadata["update"] == 938

# Since the final model was also the best model,
# these weight files should be identical.
assert (
    best_model_sha256
    == final_model_sha256
)


# ============================================================
# RELEASE TRAINING-ONLY MEMORY
# ============================================================

del model
del optimizer
del scheduler
del scaler

gc.collect()
torch.cuda.empty_cache()


print(
    "\nGPU 0 allocated after cleanup:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


# ============================================================
# RELOAD THE BEST CHECKPOINT FROM DISK
# ============================================================

sft_tokenizer = AutoTokenizer.from_pretrained(
    SFT_BEST_DIR,
    use_fast=True,
)

sft_model = AutoModelForCausalLM.from_pretrained(
    SFT_BEST_DIR,
    dtype=torch.float32,
)

sft_model = sft_model.to(
    SFT_DEVICE
)

sft_model.eval()
sft_model.config.use_cache = True


print("\n===== RELOADED SFT MODEL =====")
print(
    "Class:",
    type(sft_model).__name__,
)

print(
    "Device:",
    next(sft_model.parameters()).device,
)

print(
    "Dtype:",
    next(sft_model.parameters()).dtype,
)

print(
    "EOS:",
    sft_tokenizer.eos_token_id,
)

print(
    "PAD:",
    sft_tokenizer.pad_token_id,
)

print(
    "Parameters:",
    f"{sum(p.numel() for p in sft_model.parameters()):,}",
)

In [ ]:
from collections import Counter


SFT_GENERATION_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 256,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,
}


def repeated_ngram_fraction(
    text,
    n=4,
):
    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens) - n + 1
        )
    ]

    counts = Counter(ngrams)

    repeated_occurrences = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return (
        repeated_occurrences
        / len(ngrams)
    )


def generate_sft_one(
    instruction,
):
    prompt = format_instruction_prompt(
        instruction
    )

    encoded = sft_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded[
        "input_ids"
    ].to(SFT_DEVICE)

    attention_mask = encoded[
        "attention_mask"
    ].to(SFT_DEVICE)

    prompt_length = (
        input_ids.shape[1]
    )

    with torch.inference_mode():
        generated = sft_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_new_tokens=(
                SFT_GENERATION_CONFIG[
                    "max_new_tokens"
                ]
            ),

            do_sample=False,

            repetition_penalty=(
                SFT_GENERATION_CONFIG[
                    "repetition_penalty"
                ]
            ),

            no_repeat_ngram_size=(
                SFT_GENERATION_CONFIG[
                    "no_repeat_ngram_size"
                ]
            ),

            pad_token_id=(
                sft_tokenizer.pad_token_id
            ),

            eos_token_id=(
                sft_tokenizer.eos_token_id
            ),

            use_cache=True,
        )

    new_tokens = generated[
        0,
        prompt_length:,
    ]

    output = sft_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return {
        "output": output,
        "generated_tokens": len(
            new_tokens
        ),
        "repeat_4gram_fraction": (
            repeated_ngram_fraction(
                output,
                n=4,
            )
        ),
    }


print(
    "===== SFT GENERATION SANITY CHECK ====="
)

for row in reloaded_alpaca_subset[:5]:

    result = generate_sft_one(
        row["instruction"]
    )

    print("\n" + "=" * 80)

    print("ID:", row["id"])

    print("\nINSTRUCTION:")
    print(row["instruction"])

    print("\nSFT OUTPUT:")
    print(result["output"])

    print(
        "\nGenerated tokens:",
        result["generated_tokens"],
    )

    print(
        "Repeated 4-gram fraction:",
        round(
            result[
                "repeat_4gram_fraction"
            ],
            4,
        ),
    )

In [ ]:
print("===== TOKENIZER RELOAD EQUIVALENCE CHECK =====")

print("\nOriginal tokenizer class:")
print(type(tokenizer).__name__)

print("\nReloaded tokenizer class:")
print(type(sft_tokenizer).__name__)

print("\nOriginal vocab size:")
print(len(tokenizer))

print("\nReloaded vocab size:")
print(len(sft_tokenizer))

print("\nOriginal EOS / PAD:")
print(
    tokenizer.eos_token_id,
    tokenizer.pad_token_id,
)

print("\nReloaded EOS / PAD:")
print(
    sft_tokenizer.eos_token_id,
    sft_tokenizer.pad_token_id,
)


# ------------------------------------------------------------
# Compare multiple representative strings.
# ------------------------------------------------------------

test_texts = []

# Our exact prompt format.
for row in reloaded_alpaca_subset[:10]:
    test_texts.append(
        format_instruction_prompt(
            row["instruction"]
        )
    )

# Some training-style responses too.
for row in saved_sft["validation"].select(
    range(10)
):
    test_texts.append(
        row["response"]
    )

# Extra strings that exercise punctuation, whitespace,
# Unicode, contractions, numbers, etc.
test_texts.extend([
    "Hello, world!",
    "It's a test — with punctuation.",
    "123.45 + 67 = 190.45",
    "Multiple   spaces\nand\nnewlines.",
    "### Instruction:\nExplain AI.\n\n### Response:\n",
    "Urdu: پاکستان",
])


all_equal = True

for i, text in enumerate(test_texts):
    original_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    reloaded_ids = sft_tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    equal = (
        original_ids
        == reloaded_ids
    )

    if not equal:
        all_equal = False

        print(
            f"\nMismatch at test {i}"
        )

        print("TEXT:")
        print(repr(text[:500]))

        print(
            "Original length:",
            len(original_ids),
        )

        print(
            "Reloaded length:",
            len(reloaded_ids),
        )

        print(
            "Original first IDs:",
            original_ids[:50],
        )

        print(
            "Reloaded first IDs:",
            reloaded_ids[:50],
        )


print("\nAll tested tokenizations identical:", all_equal)

assert all_equal, (
    "Reloaded checkpoint tokenizer differs from "
    "the tokenizer used for training."
)

print(
    "\n✓ Saved/reloaded tokenizer matches "
    "the original tokenizer on all tests."
)

In [ ]:
import json
import hashlib
from pathlib import Path
from tqdm.auto import tqdm
import torch
from collections import Counter

from data_utils import format_instruction_prompt


# ============================================================
# OUTPUT PATHS
# ============================================================

SFT_EVAL_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/eval/sft_qwen2.5_0.5b"
)

SFT_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SFT_PROGRESS_PATH = (
    SFT_EVAL_DIR
    / "alpaca_eval_300_seed42_sft_progress.jsonl"
)

SFT_FINAL_EVAL_PATH = (
    SFT_EVAL_DIR
    / "alpaca_eval_300_seed42_sft.json"
)

SFT_GENERATION_CONFIG_PATH = (
    SFT_EVAL_DIR
    / "generation_config.json"
)


# ============================================================
# FROZEN GENERATION SETTINGS
# ============================================================

SFT_EVAL_GENERATION_CONFIG = {
    "checkpoint": str(SFT_BEST_DIR),

    "checkpoint_update": 938,

    "checkpoint_validation_loss": (
        1.3143231261241624
    ),

    "checkpoint_model_sha256": (
        best_model_sha256
    ),

    "generator": (
        "qwen2.5-0.5b-sft-ultrachat"
    ),

    "evaluation_set": (
        "eval_sets/alpaca_eval_300_seed42.json"
    ),

    "evaluation_set_sha256": (
        "0e5763f8986c2149b4f690fe73225dcb"
        "71f0b125f36de6355200284276b772e8"
    ),

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    # EXACT same final decoding policy as base.
    "do_sample": False,
    "max_new_tokens": 256,

    "temperature": None,
    "top_p": None,

    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,

    "pad_token_id": (
        sft_tokenizer.pad_token_id
    ),

    "eos_token_id": (
        sft_tokenizer.eos_token_id
    ),
}


SFT_GENERATION_CONFIG_PATH.write_text(
    json.dumps(
        SFT_EVAL_GENERATION_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# RESUME SUPPORT
# ============================================================

completed_sft = {}

if SFT_PROGRESS_PATH.exists():
    with open(
        SFT_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                completed_sft[row["id"]] = row


print(
    "Already completed:",
    len(completed_sft),
)


# ============================================================
# GENERATION
# ============================================================

sft_model.eval()
sft_model.config.use_cache = True


for row in tqdm(
    reloaded_alpaca_subset,
    desc="Generating SFT AlpacaEval",
):

    if row["id"] in completed_sft:
        continue

    prompt = format_instruction_prompt(
        row["instruction"]
    )

    encoded = sft_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded[
        "input_ids"
    ].to(SFT_DEVICE)

    attention_mask = encoded[
        "attention_mask"
    ].to(SFT_DEVICE)

    prompt_token_count = (
        input_ids.shape[1]
    )

    with torch.inference_mode():
        generated = sft_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_new_tokens=(
                SFT_EVAL_GENERATION_CONFIG[
                    "max_new_tokens"
                ]
            ),

            do_sample=False,

            repetition_penalty=(
                SFT_EVAL_GENERATION_CONFIG[
                    "repetition_penalty"
                ]
            ),

            no_repeat_ngram_size=(
                SFT_EVAL_GENERATION_CONFIG[
                    "no_repeat_ngram_size"
                ]
            ),

            pad_token_id=(
                sft_tokenizer.pad_token_id
            ),

            eos_token_id=(
                sft_tokenizer.eos_token_id
            ),

            use_cache=True,
        )

    new_tokens = generated[
        0,
        prompt_token_count:,
    ]

    output = sft_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    generated_token_count = len(
        new_tokens
    )

    result = {
        "id": row["id"],
        "instruction": row["instruction"],
        "output": output,

        "generator": (
            "qwen2.5-0.5b-sft-ultrachat"
        ),

        "dataset": "alpaca_eval",

        "source_index": (
            row["source_index"]
        ),

        "subset_position": (
            row["subset_position"]
        ),

        "prompt_token_count": (
            prompt_token_count
        ),

        "generated_token_count": (
            generated_token_count
        ),

        "hit_max_new_tokens": (
            generated_token_count
            >= SFT_EVAL_GENERATION_CONFIG[
                "max_new_tokens"
            ]
        ),
    }

    # Save every completed generation immediately.
    with open(
        SFT_PROGRESS_PATH,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            json.dumps(
                result,
                ensure_ascii=False,
            )
            + "\n"
        )

    completed_sft[row["id"]] = result


# ============================================================
# REBUILD IN EXACT FIXED PROMPT ORDER
# ============================================================

final_sft_outputs = [
    completed_sft[row["id"]]
    for row in reloaded_alpaca_subset
]

assert len(final_sft_outputs) == 300

assert [
    row["id"]
    for row in final_sft_outputs
] == [
    row["id"]
    for row in reloaded_alpaca_subset
]


with open(
    SFT_FINAL_EVAL_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_sft_outputs,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# REPETITION DIAGNOSTICS
# ============================================================

def repeated_ngram_fraction(
    text,
    n=4,
):
    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens) - n + 1
        )
    ]

    counts = Counter(ngrams)

    repeated_occurrences = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return (
        repeated_occurrences
        / len(ngrams)
    )


repeat_scores_sft = [
    repeated_ngram_fraction(
        row["output"],
        n=4,
    )
    for row in final_sft_outputs
]


# ============================================================
# SHA256
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


sft_eval_sha256 = sha256_file(
    SFT_FINAL_EVAL_PATH
)


# ============================================================
# FINAL DIAGNOSTICS
# ============================================================

hit_cap = sum(
    row["hit_max_new_tokens"]
    for row in final_sft_outputs
)

average_tokens = (
    sum(
        row["generated_token_count"]
        for row in final_sft_outputs
    )
    / len(final_sft_outputs)
)


print(
    "\n===== SFT GENERATION COMPLETE ====="
)

print(
    "Examples:",
    len(final_sft_outputs),
)

print(
    "Hit max_new_tokens:",
    hit_cap,
)

print(
    "Hit max percentage:",
    round(
        100
        * hit_cap
        / len(final_sft_outputs),
        2,
    ),
    "%",
)

print(
    "Average generated tokens:",
    round(
        average_tokens,
        2,
    ),
)

print(
    "Mean repeated 4-gram fraction:",
    round(
        sum(repeat_scores_sft)
        / len(repeat_scores_sft),
        4,
    ),
)

print(
    "Outputs with >20% repeated 4-grams:",
    sum(
        score > 0.20
        for score in repeat_scores_sft
    ),
)

print(
    "Outputs with >40% repeated 4-grams:",
    sum(
        score > 0.40
        for score in repeat_scores_sft
    ),
)

print("\nFinal file:")
print(
    SFT_FINAL_EVAL_PATH
)

print("\nSHA256:")
print(
    sft_eval_sha256
)

In [ ]:
from datasets import load_dataset


HH_DATASET_NAME = "Anthropic/hh-rlhf"


# ============================================================
# LOAD DATASET
# ============================================================

hh_raw = load_dataset(
    HH_DATASET_NAME
)


print("===== HH-RLHF DATASET =====")
print(hh_raw)

print("\nSplit names:")
print(list(hh_raw.keys()))


# ============================================================
# SCHEMA
# ============================================================

for split_name in hh_raw.keys():
    print(
        f"\n===== {split_name.upper()} ====="
    )

    print(
        "Rows:",
        len(hh_raw[split_name]),
    )

    print(
        "Features:",
        hh_raw[split_name].features,
    )


# ============================================================
# FIRST 5 TRAIN EXAMPLES
# ============================================================

print(
    "\n===== FIRST 5 TRAIN EXAMPLES ====="
)

for i in range(5):
    row = hh_raw["train"][i]

    print("\n" + "=" * 100)
    print("ROW:", i)

    print("\nKEYS:")
    print(row.keys())

    print("\nCHOSEN:")
    print(repr(row["chosen"]))

    print("\nREJECTED:")
    print(repr(row["rejected"]))

In [ ]:
# ============================================================
# HH-RLHF SINGLE-TURN EXTRACTION
# ============================================================

def extract_hh_single_turn(example):
    """
    Convert one Anthropic HH-RLHF row into:

        prompt
        chosen
        rejected

    Only accepts STRICT single-turn conversations:

        Human: <prompt>
        Assistant: <response>

    for both chosen and rejected, with the exact same Human
    prompt on both sides.

    Multi-turn rows are rejected.
    """

    chosen_text = example.get("chosen")
    rejected_text = example.get("rejected")

    if not isinstance(chosen_text, str):
        return None

    if not isinstance(rejected_text, str):
        return None


    human_marker = "\n\nHuman:"
    assistant_marker = "\n\nAssistant:"


    # --------------------------------------------------------
    # Strictly require one Human turn and one Assistant turn.
    # --------------------------------------------------------

    if chosen_text.count(human_marker) != 1:
        return None

    if rejected_text.count(human_marker) != 1:
        return None

    if chosen_text.count(assistant_marker) != 1:
        return None

    if rejected_text.count(assistant_marker) != 1:
        return None


    # --------------------------------------------------------
    # Both transcripts must begin with the Human marker.
    # --------------------------------------------------------

    if not chosen_text.startswith(human_marker):
        return None

    if not rejected_text.startswith(human_marker):
        return None


    # --------------------------------------------------------
    # Split Human prompt from Assistant response.
    # --------------------------------------------------------

    chosen_before, chosen_sep, chosen_response = (
        chosen_text.partition(
            assistant_marker
        )
    )

    rejected_before, rejected_sep, rejected_response = (
        rejected_text.partition(
            assistant_marker
        )
    )

    if not chosen_sep or not rejected_sep:
        return None


    chosen_prompt = chosen_before[
        len(human_marker):
    ].strip()

    rejected_prompt = rejected_before[
        len(human_marker):
    ].strip()

    chosen_response = chosen_response.strip()
    rejected_response = rejected_response.strip()


    # --------------------------------------------------------
    # Preference pair must share EXACTLY the same prompt.
    # --------------------------------------------------------

    if chosen_prompt != rejected_prompt:
        return None


    if not chosen_prompt:
        return None

    if not chosen_response:
        return None

    if not rejected_response:
        return None


    # Degenerate pair gives no preference signal.
    if chosen_response == rejected_response:
        return None


    return {
        "instruction": chosen_prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }

In [ ]:
%%writefile -a data_utils.py


# ============================================================
# HH-RLHF SINGLE-TURN EXTRACTION
# ============================================================

def extract_hh_single_turn(example):
    """
    Convert one Anthropic HH-RLHF row into:

        instruction
        chosen
        rejected

    Only accepts strict single-turn conversations:

        Human: <prompt>
        Assistant: <response>

    for both chosen and rejected, with exactly the same Human
    prompt on both sides.

    Multi-turn rows are rejected.
    """

    chosen_text = example.get("chosen")
    rejected_text = example.get("rejected")

    if not isinstance(chosen_text, str):
        return None

    if not isinstance(rejected_text, str):
        return None


    human_marker = "\n\nHuman:"
    assistant_marker = "\n\nAssistant:"


    # Require exactly one Human turn.
    if chosen_text.count(human_marker) != 1:
        return None

    if rejected_text.count(human_marker) != 1:
        return None


    # Require exactly one Assistant turn.
    if chosen_text.count(assistant_marker) != 1:
        return None

    if rejected_text.count(assistant_marker) != 1:
        return None


    # Both transcripts must begin with Human.
    if not chosen_text.startswith(human_marker):
        return None

    if not rejected_text.startswith(human_marker):
        return None


    # Split prompt and response.
    chosen_before, chosen_sep, chosen_response = (
        chosen_text.partition(assistant_marker)
    )

    rejected_before, rejected_sep, rejected_response = (
        rejected_text.partition(assistant_marker)
    )

    if not chosen_sep or not rejected_sep:
        return None


    chosen_prompt = chosen_before[
        len(human_marker):
    ].strip()

    rejected_prompt = rejected_before[
        len(human_marker):
    ].strip()

    chosen_response = chosen_response.strip()
    rejected_response = rejected_response.strip()


    # Both preference alternatives must use exactly
    # the same prompt.
    if chosen_prompt != rejected_prompt:
        return None


    if not chosen_prompt:
        return None

    if not chosen_response:
        return None

    if not rejected_response:
        return None


    # No useful preference signal if responses are identical.
    if chosen_response == rejected_response:
        return None


    return {
        "instruction": chosen_prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }

In [ ]:
import importlib
import data_utils

data_utils = importlib.reload(data_utils)

from data_utils import (
    extract_hh_single_turn,
    format_instruction_prompt,
)

print("✓ HH parser imported successfully")

In [ ]:
print("===== HH PARSER TEST =====")

for i in range(5):
    parsed = extract_hh_single_turn(
        hh_raw["train"][i]
    )

    print("\n" + "=" * 80)
    print("ROW:", i)

    if parsed is None:
        print("FILTERED OUT")

    else:
        print("ACCEPTED")

        print("\nINSTRUCTION:")
        print(parsed["instruction"])

        print("\nCHOSEN:")
        print(parsed["chosen"])

        print("\nREJECTED:")
        print(parsed["rejected"])

In [ ]:
import json
import numpy as np
from tqdm.auto import tqdm


DPO_CONTEXT_LENGTH = 512
DPO_MAX_PROMPT_LENGTH = 256
TOKENIZE_BATCH_SIZE = 512


def describe_lengths(values):
    arr = np.asarray(
        values,
        dtype=np.float64,
    )

    if len(arr) == 0:
        return {}

    return {
        "count": int(len(arr)),
        "mean": round(float(arr.mean()), 2),
        "median": round(float(np.median(arr)), 2),
        "p90": round(float(np.percentile(arr, 90)), 2),
        "p95": round(float(np.percentile(arr, 95)), 2),
        "p99": round(float(np.percentile(arr, 99)), 2),
        "max": int(arr.max()),
    }


def analyze_hh_split(
    dataset,
    split_name,
):
    # --------------------------------------------------------
    # 1. Parse strict single-turn rows
    # --------------------------------------------------------

    parsed_rows = []

    invalid_count = 0

    for source_index, example in enumerate(
        tqdm(
            dataset,
            desc=f"Parsing {split_name}",
        )
    ):
        parsed = extract_hh_single_turn(
            example
        )

        if parsed is None:
            invalid_count += 1
            continue

        parsed_rows.append({
            "source_index": source_index,
            **parsed,
        })


    print(
        f"\n{split_name}: "
        f"{len(parsed_rows)} strict single-turn rows"
    )


    # --------------------------------------------------------
    # 2. Length bookkeeping
    # --------------------------------------------------------

    prompt_lengths = []
    chosen_response_lengths = []
    rejected_response_lengths = []

    chosen_total_lengths = []
    rejected_total_lengths = []


    counts = {
        "source_rows": len(dataset),

        "single_turn_valid": len(
            parsed_rows
        ),

        "multi_or_invalid": (
            invalid_count
        ),

        "prompt_over_256": 0,

        "chosen_full_fit": 0,
        "rejected_full_fit": 0,

        "both_full_fit": 0,

        "usable_full_fit_and_prompt_limit": 0,
    }


    # We'll also save all rows that satisfy BOTH:
    #
    # prompt <= 256
    # chosen total <= 512
    # rejected total <= 512
    #
    # We are NOT sampling the final subset yet.
    usable_candidates = []


    eos_extra = (
        1
        if sft_tokenizer.eos_token_id
        is not None
        else 0
    )


    # --------------------------------------------------------
    # 3. Tokenize in chunks for speed
    # --------------------------------------------------------

    for start in tqdm(
        range(
            0,
            len(parsed_rows),
            TOKENIZE_BATCH_SIZE,
        ),
        desc=f"Tokenizing {split_name}",
    ):

        chunk = parsed_rows[
            start:
            start + TOKENIZE_BATCH_SIZE
        ]


        prompts = [
            format_instruction_prompt(
                row["instruction"]
            )
            for row in chunk
        ]

        chosen_texts = [
            row["chosen"]
            for row in chunk
        ]

        rejected_texts = [
            row["rejected"]
            for row in chunk
        ]


        prompt_encoded = sft_tokenizer(
            prompts,
            add_special_tokens=False,
            padding=False,
            truncation=False,
        )

        chosen_encoded = sft_tokenizer(
            chosen_texts,
            add_special_tokens=False,
            padding=False,
            truncation=False,
        )

        rejected_encoded = sft_tokenizer(
            rejected_texts,
            add_special_tokens=False,
            padding=False,
            truncation=False,
        )


        for row, p_ids, c_ids, r_ids in zip(
            chunk,
            prompt_encoded["input_ids"],
            chosen_encoded["input_ids"],
            rejected_encoded["input_ids"],
        ):

            prompt_len = len(p_ids)

            chosen_response_len = (
                len(c_ids)
                + eos_extra
            )

            rejected_response_len = (
                len(r_ids)
                + eos_extra
            )


            chosen_total = (
                prompt_len
                + chosen_response_len
            )

            rejected_total = (
                prompt_len
                + rejected_response_len
            )


            prompt_lengths.append(
                prompt_len
            )

            chosen_response_lengths.append(
                chosen_response_len
            )

            rejected_response_lengths.append(
                rejected_response_len
            )

            chosen_total_lengths.append(
                chosen_total
            )

            rejected_total_lengths.append(
                rejected_total
            )


            # ------------------------------
            # Prompt limit
            # ------------------------------

            prompt_ok = (
                prompt_len
                <= DPO_MAX_PROMPT_LENGTH
            )

            if not prompt_ok:
                counts[
                    "prompt_over_256"
                ] += 1


            # ------------------------------
            # Context limits
            # ------------------------------

            chosen_fit = (
                chosen_total
                <= DPO_CONTEXT_LENGTH
            )

            rejected_fit = (
                rejected_total
                <= DPO_CONTEXT_LENGTH
            )


            if chosen_fit:
                counts[
                    "chosen_full_fit"
                ] += 1

            if rejected_fit:
                counts[
                    "rejected_full_fit"
                ] += 1

            if (
                chosen_fit
                and rejected_fit
            ):
                counts[
                    "both_full_fit"
                ] += 1


            # ------------------------------
            # Fully usable
            # ------------------------------

            if (
                prompt_ok
                and chosen_fit
                and rejected_fit
            ):

                counts[
                    "usable_full_fit_and_prompt_limit"
                ] += 1

                usable_candidates.append({
                    "source_index": (
                        row["source_index"]
                    ),

                    "instruction": (
                        row["instruction"]
                    ),

                    "chosen": (
                        row["chosen"]
                    ),

                    "rejected": (
                        row["rejected"]
                    ),

                    "prompt_tokens": (
                        prompt_len
                    ),

                    "chosen_response_tokens": (
                        chosen_response_len
                    ),

                    "rejected_response_tokens": (
                        rejected_response_len
                    ),

                    "chosen_total_tokens": (
                        chosen_total
                    ),

                    "rejected_total_tokens": (
                        rejected_total
                    ),
                })


    analysis = {
        "counts": counts,

        "prompt_lengths": (
            describe_lengths(
                prompt_lengths
            )
        ),

        "chosen_response_lengths": (
            describe_lengths(
                chosen_response_lengths
            )
        ),

        "rejected_response_lengths": (
            describe_lengths(
                rejected_response_lengths
            )
        ),

        "chosen_total_lengths": (
            describe_lengths(
                chosen_total_lengths
            )
        ),

        "rejected_total_lengths": (
            describe_lengths(
                rejected_total_lengths
            )
        ),
    }


    return (
        analysis,
        usable_candidates,
    )


# ============================================================
# TRAIN SPLIT
# ============================================================

print(
    "===== ANALYZING HH TRAIN ====="
)

(
    hh_train_analysis,
    hh_train_candidates,
) = analyze_hh_split(
    hh_raw["train"],
    "train",
)

print(
    "\n===== HH TRAIN ANALYSIS ====="
)

print(
    json.dumps(
        hh_train_analysis,
        indent=2,
    )
)


# ============================================================
# TEST SPLIT
# ============================================================

print(
    "\n===== ANALYZING HH TEST ====="
)

(
    hh_test_analysis,
    hh_test_candidates,
) = analyze_hh_split(
    hh_raw["test"],
    "test",
)

print(
    "\n===== HH TEST ANALYSIS ====="
)

print(
    json.dumps(
        hh_test_analysis,
        indent=2,
    )
)


# ============================================================
# SIMPLE FEASIBILITY CHECK
# ============================================================

print(
    "\n===== DPO SUBSET FEASIBILITY ====="
)

print(
    "Usable train candidates:",
    len(hh_train_candidates),
)

print(
    "Need:",
    15000,
)

print(
    "\nUsable validation candidates:",
    len(hh_test_candidates),
)

print(
    "Need:",
    1000,
)

print(
    "\nEnough train:",
    len(hh_train_candidates) >= 15000,
)

print(
    "Enough validation:",
    len(hh_test_candidates) >= 1000,
)

In [ ]:
import json
import random
import hashlib
from pathlib import Path


HH_SEED = 42
HH_TRAIN_N = 15_000
HH_VAL_N = 1_000


HH_FIXED_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "data/hh_rlhf_fixed"
)

HH_FIXED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


HH_TRAIN_PATH = (
    HH_FIXED_DIR
    / "hh_train_15000_seed42.jsonl"
)

HH_VAL_PATH = (
    HH_FIXED_DIR
    / "hh_val_1000_seed42.jsonl"
)

HH_METADATA_PATH = (
    HH_FIXED_DIR
    / "selection_metadata.json"
)


# ============================================================
# DETERMINISTIC SAMPLING
# ============================================================

train_rng = random.Random(HH_SEED)
val_rng = random.Random(HH_SEED)


train_selected_indices = train_rng.sample(
    range(len(hh_train_candidates)),
    HH_TRAIN_N,
)

val_selected_indices = val_rng.sample(
    range(len(hh_test_candidates)),
    HH_VAL_N,
)


hh_train_subset = [
    {
        "id": f"hh_train_{hh_train_candidates[i]['source_index']:06d}",

        **hh_train_candidates[i],
    }
    for i in train_selected_indices
]


hh_val_subset = [
    {
        "id": f"hh_test_{hh_test_candidates[i]['source_index']:06d}",

        **hh_test_candidates[i],
    }
    for i in val_selected_indices
]


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(hh_train_subset) == 15_000
assert len(hh_val_subset) == 1_000


# Every selected example must satisfy our frozen filtering rule.
for row in hh_train_subset:
    assert row["prompt_tokens"] <= 256
    assert row["chosen_total_tokens"] <= 512
    assert row["rejected_total_tokens"] <= 512

for row in hh_val_subset:
    assert row["prompt_tokens"] <= 256
    assert row["chosen_total_tokens"] <= 512
    assert row["rejected_total_tokens"] <= 512


# No duplicate source rows within either selected split.
assert len({
    row["source_index"]
    for row in hh_train_subset
}) == 15_000

assert len({
    row["source_index"]
    for row in hh_val_subset
}) == 1_000


# ============================================================
# SAVE JSONL
# ============================================================

def write_jsonl(path, rows):
    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


write_jsonl(
    HH_TRAIN_PATH,
    hh_train_subset,
)

write_jsonl(
    HH_VAL_PATH,
    hh_val_subset,
)


# ============================================================
# HASHES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


hh_train_sha256 = sha256_file(
    HH_TRAIN_PATH
)

hh_val_sha256 = sha256_file(
    HH_VAL_PATH
)


# ============================================================
# METADATA
# ============================================================

hh_selection_metadata = {
    "source_dataset": "Anthropic/hh-rlhf",

    "train_source_split": "train",
    "validation_source_split": "test",

    "seed": HH_SEED,

    "train_examples": HH_TRAIN_N,
    "validation_examples": HH_VAL_N,

    "prompt_format": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "single_turn_rule": (
        "Accept only transcripts containing exactly one "
        "Human turn and one Assistant turn on both chosen "
        "and rejected sides, with identical Human prompts."
    ),

    "multi_turn_policy": "filtered out",

    "max_prompt_tokens": 256,
    "context_length": 512,

    "length_filter": (
        "Formatted prompt must be <=256 tokens and both "
        "prompt+chosen+EOS and prompt+rejected+EOS must "
        "fit completely within 512 tokens."
    ),

    "candidate_counts": {
        "train": len(hh_train_candidates),
        "validation": len(hh_test_candidates),
    },

    "source_analysis": {
        "train": hh_train_analysis["counts"],
        "validation": hh_test_analysis["counts"],
    },

    "sampling_method": (
        "random.Random(42).sample(candidate_indices, n), "
        "with candidate order inherited from the official "
        "source split."
    ),

    "train_file": str(HH_TRAIN_PATH),
    "validation_file": str(HH_VAL_PATH),

    "train_sha256": hh_train_sha256,
    "validation_sha256": hh_val_sha256,
}


with open(
    HH_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        hh_selection_metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# PRINT SUMMARY
# ============================================================

print("===== FIXED HH-RLHF SUBSETS =====")

print("Train examples:", len(hh_train_subset))
print("Validation examples:", len(hh_val_subset))

print("\nTrain file:")
print(HH_TRAIN_PATH)

print("\nValidation file:")
print(HH_VAL_PATH)

print("\nTrain SHA256:")
print(hh_train_sha256)

print("\nValidation SHA256:")
print(hh_val_sha256)

print("\nFirst 10 selected TRAIN source indices:")
print([
    row["source_index"]
    for row in hh_train_subset[:10]
])

print("\nFirst 10 selected VAL source indices:")
print([
    row["source_index"]
    for row in hh_val_subset[:10]
])

print("\nFile sizes:")

for path in [
    HH_TRAIN_PATH,
    HH_VAL_PATH,
    HH_METADATA_PATH,
]:
    print(
        path.name,
        round(
            path.stat().st_size / 1024**2,
            2,
        ),
        "MB",
    )

In [ ]:
def read_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                rows.append(
                    json.loads(line)
                )

    return rows


hh_train_fixed = read_jsonl(
    HH_TRAIN_PATH
)

hh_val_fixed = read_jsonl(
    HH_VAL_PATH
)


assert len(hh_train_fixed) == 15_000
assert len(hh_val_fixed) == 1_000

assert (
    sha256_file(HH_TRAIN_PATH)
    == hh_train_sha256
)

assert (
    sha256_file(HH_VAL_PATH)
    == hh_val_sha256
)


print(
    "✓ Exact HH 15,000 / 1,000 subsets "
    "reload successfully"
)

In [ ]:
import math
import json
import random
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import DataLoader

from data_utils import format_instruction_prompt


# ============================================================
# HH-DPO CONFIG
# ============================================================

HH_DPO_CONFIG = {
    "run_name": "hh_rlhf_dpo_seed42",

    "initial_policy": str(SFT_BEST_DIR),
    "reference_model": str(SFT_BEST_DIR),

    "train_examples": 15_000,
    "validation_examples": 1_000,

    "context_length": 512,
    "max_prompt_length": 256,

    "per_device_batch_size": 1,
    "gradient_accumulation_steps": 32,
    "effective_batch_size": 32,

    "learning_rate": 1e-6,
    "lr_schedule": "fixed",

    "optimizer": "AdamW",
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_epsilon": 1e-8,
    "weight_decay": 0.0,

    "beta": 0.1,

    "gradient_clip_norm": 1.0,
    "epochs": 1,

    "precision_policy": "fp16_autocast_with_fp32_weights",
    "precision_reference": "fp16_frozen",

    "seed": 42,

    # Our monitoring choice; the manual does not prescribe
    # a validation interval.
    "eval_every_updates": 100,

    "checkpoint_selection": (
        "highest validation reward accuracy"
    ),

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "train_data_sha256": (
        "d28ef3ded817949b7d8e08753cb9a0dd"
        "bcc7afa5ecb6403ac394325a98e30946"
    ),

    "validation_data_sha256": (
        "f8cf92b00c1dba254045fc6086070632"
        "d1562d9a0f76ab0d664e2e4225cb9e47"
    ),
}


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(
    HH_DPO_CONFIG["seed"]
)

np.random.seed(
    HH_DPO_CONFIG["seed"]
)

torch.manual_seed(
    HH_DPO_CONFIG["seed"]
)

torch.cuda.manual_seed_all(
    HH_DPO_CONFIG["seed"]
)


# ============================================================
# COLLATOR
# ============================================================

def hh_dpo_collate_fn(rows):
    """
    Keep strings here.

    Tokenization and response masking will happen in the
    DPO utilities using the already-tested alignment core.
    """

    prompts = [
        format_instruction_prompt(
            row["instruction"]
        )
        for row in rows
    ]

    chosen = [
        row["chosen"]
        for row in rows
    ]

    rejected = [
        row["rejected"]
        for row in rows
    ]

    ids = [
        row["id"]
        for row in rows
    ]

    return {
        "ids": ids,
        "prompts": prompts,
        "chosen": chosen,
        "rejected": rejected,
    }


# ============================================================
# DATALOADERS
# ============================================================

hh_train_generator = torch.Generator()

hh_train_generator.manual_seed(
    HH_DPO_CONFIG["seed"]
)


hh_train_loader = DataLoader(
    hh_train_fixed,

    batch_size=HH_DPO_CONFIG[
        "per_device_batch_size"
    ],

    shuffle=True,

    generator=hh_train_generator,

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,

    drop_last=False,
)


hh_val_loader = DataLoader(
    hh_val_fixed,

    batch_size=HH_DPO_CONFIG[
        "per_device_batch_size"
    ],

    shuffle=False,

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,

    drop_last=False,
)


# ============================================================
# OPTIMIZER-STEP COUNTS
# ============================================================

hh_micro_batches_per_epoch = len(
    hh_train_loader
)

hh_total_update_steps = math.ceil(
    hh_micro_batches_per_epoch
    / HH_DPO_CONFIG[
        "gradient_accumulation_steps"
    ]
)


HH_DPO_CONFIG[
    "micro_batches_per_epoch"
] = hh_micro_batches_per_epoch

HH_DPO_CONFIG[
    "total_update_steps"
] = hh_total_update_steps


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

HH_DPO_RUN_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/hh_rlhf_dpo_seed42"
)

HH_DPO_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HH_DPO_CONFIG_PATH = (
    HH_DPO_RUN_DIR
    / "config.json"
)

HH_DPO_CONFIG_PATH.write_text(
    json.dumps(
        HH_DPO_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# SUMMARY
# ============================================================

print("===== HH-DPO CONFIG =====")

print(
    json.dumps(
        HH_DPO_CONFIG,
        indent=2,
    )
)

print("\n===== DATA =====")

print(
    "Train examples:",
    len(hh_train_fixed),
)

print(
    "Validation examples:",
    len(hh_val_fixed),
)

print(
    "Train micro-batches:",
    len(hh_train_loader),
)

print(
    "Validation micro-batches:",
    len(hh_val_loader),
)

print(
    "Total optimizer updates:",
    hh_total_update_steps,
)

print(
    "Effective batch size:",
    HH_DPO_CONFIG[
        "effective_batch_size"
    ],
)

print("\nConfig saved to:")
print(HH_DPO_CONFIG_PATH)

In [ ]:
import importlib
import alignment_core

alignment_core = importlib.reload(
    alignment_core
)

from alignment_core import (
    build_dpo_batch_impl,
)


hh_test_raw_batch = next(
    iter(hh_val_loader)
)


hh_test_batch = build_dpo_batch_impl(
    tokenizer=sft_tokenizer,

    prompts=hh_test_raw_batch[
        "prompts"
    ],

    chosen_responses=hh_test_raw_batch[
        "chosen"
    ],

    rejected_responses=hh_test_raw_batch[
        "rejected"
    ],

    max_length=HH_DPO_CONFIG[
        "context_length"
    ],
)


print("===== HH DPO BATCH CHECK =====")

print(
    "ID:",
    hh_test_raw_batch["ids"][0],
)

print("\nPROMPT:")
print(
    hh_test_raw_batch[
        "prompts"
    ][0]
)

print("\nCHOSEN:")
print(
    hh_test_raw_batch[
        "chosen"
    ][0]
)

print("\nREJECTED:")
print(
    hh_test_raw_batch[
        "rejected"
    ][0]
)


print("\n===== CHOSEN TENSORS =====")

for key, value in (
    hh_test_batch["chosen"].items()
):
    print(
        key,
        tuple(value.shape),
        value.dtype,
    )


print("\n===== REJECTED TENSORS =====")

for key, value in (
    hh_test_batch["rejected"].items()
):
    print(
        key,
        tuple(value.shape),
        value.dtype,
    )


chosen_scored_tokens = (
    hh_test_batch["chosen"][
        "labels"
    ][:, 1:]
    .ne(-100)
    .sum()
    .item()
)

rejected_scored_tokens = (
    hh_test_batch["rejected"][
        "labels"
    ][:, 1:]
    .ne(-100)
    .sum()
    .item()
)


print(
    "\nChosen response tokens scored:",
    chosen_scored_tokens,
)

print(
    "Rejected response tokens scored:",
    rejected_scored_tokens,
)


# Verify prompt labels are masked.
chosen_labels = (
    hh_test_batch["chosen"][
        "labels"
    ][0]
)

rejected_labels = (
    hh_test_batch["rejected"][
        "labels"
    ][0]
)


chosen_first_target = int(
    (
        chosen_labels != -100
    )
    .nonzero(as_tuple=False)[0]
    .item()
)

rejected_first_target = int(
    (
        rejected_labels != -100
    )
    .nonzero(as_tuple=False)[0]
    .item()
)


print(
    "\nFirst chosen response-label position:",
    chosen_first_target,
)

print(
    "First rejected response-label position:",
    rejected_first_target,
)


prompt_token_count = len(
    sft_tokenizer.encode(
        hh_test_raw_batch[
            "prompts"
        ][0],
        add_special_tokens=False,
    )
)

print(
    "Prompt token count:",
    prompt_token_count,
)


assert (
    chosen_first_target
    == prompt_token_count
)

assert (
    rejected_first_target
    == prompt_token_count
)


print(
    "\n✓ HH DPO batch has correct "
    "response-only masking"
)

In [ ]:
HH_DPO_CONFIG["precision_reference"] = (
    "fp16_autocast_with_fp32_frozen_weights"
)

HH_DPO_CONFIG_PATH.write_text(
    json.dumps(
        HH_DPO_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Reference precision:",
    HH_DPO_CONFIG["precision_reference"],
)

In [ ]:
%%writefile dpo_training.py

from __future__ import annotations

import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from alignment_core import (
    build_dpo_batch_impl,
    compute_dpo_loss_from_logps_impl,
)


# ============================================================
# FILE HELPERS
# ============================================================

def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
        )


# ============================================================
# PREPARE ONE PREFERENCE PAIR
# ============================================================

def prepare_dpo_pair_cpu(
    tokenizer,
    prompt,
    chosen,
    rejected,
    max_length,
):
    """
    Prepare one preference pair using the already-tested
    assignment batch implementation.

    The returned joint tensors have two rows:

        row 0 = chosen
        row 1 = rejected

    Trailing columns that are padding for BOTH sequences are
    removed for efficiency.
    """

    dpo_batch = build_dpo_batch_impl(
        tokenizer=tokenizer,
        prompts=[prompt],
        chosen_responses=[chosen],
        rejected_responses=[rejected],
        max_length=max_length,
    )

    chosen_batch = dpo_batch["chosen"]
    rejected_batch = dpo_batch["rejected"]


    chosen_used_length = int(
        chosen_batch[
            "attention_mask"
        ]
        .sum(dim=1)
        .max()
        .item()
    )

    rejected_used_length = int(
        rejected_batch[
            "attention_mask"
        ]
        .sum(dim=1)
        .max()
        .item()
    )


    joint_length = max(
        chosen_used_length,
        rejected_used_length,
        2,
    )


    # Both source batches were originally padded to max_length,
    # so slicing both to joint_length leaves valid padding on
    # whichever response is shorter.
    joint_batch = {}

    for key in (
        "input_ids",
        "attention_mask",
        "labels",
    ):
        chosen_tensor = (
            chosen_batch[key]
            [:, :joint_length]
            .contiguous()
        )

        rejected_tensor = (
            rejected_batch[key]
            [:, :joint_length]
            .contiguous()
        )

        joint_batch[key] = torch.cat(
            [
                chosen_tensor,
                rejected_tensor,
            ],
            dim=0,
        )


    chosen_response_tokens = int(
        joint_batch["labels"][
            0,
            1:,
        ]
        .ne(-100)
        .sum()
        .item()
    )

    rejected_response_tokens = int(
        joint_batch["labels"][
            1,
            1:,
        ]
        .ne(-100)
        .sum()
        .item()
    )


    if chosen_response_tokens <= 0:
        raise RuntimeError(
            "Chosen response has no scorable tokens."
        )

    if rejected_response_tokens <= 0:
        raise RuntimeError(
            "Rejected response has no scorable tokens."
        )


    return {
        "batch": joint_batch,
        "chosen_response_tokens": (
            chosen_response_tokens
        ),
        "rejected_response_tokens": (
            rejected_response_tokens
        ),
    }


def move_batch_to_device(
    batch,
    device,
):
    return {
        key: value.to(
            device,
            non_blocking=True,
        )
        for key, value in batch.items()
    }


# ============================================================
# RESPONSE LOG PROBABILITIES FOR A JOINT BATCH
# ============================================================

def response_logps_from_joint_batch(
    model,
    batch,
):
    """
    Return one response-only sequence log probability per row.

    For a normal DPO pair:
        result[0] = chosen log p
        result[1] = rejected log p
    """

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch[
            "attention_mask"
        ],
        use_cache=False,
    )

    logits = outputs.logits


    # Causal one-token shift.
    shift_logits = logits[
        :,
        :-1,
        :,
    ]

    shift_labels = batch[
        "labels"
    ][:, 1:]


    vocab_size = (
        shift_logits.shape[-1]
    )


    # reduction='none' gives token NLL.
    # Tokens with label -100 contribute exactly zero.
    token_nll = F.cross_entropy(
        shift_logits.reshape(
            -1,
            vocab_size,
        ),
        shift_labels.reshape(-1),

        ignore_index=-100,
        reduction="none",
    )


    token_nll = token_nll.reshape(
        shift_labels.shape
    )


    sequence_logps = (
        -token_nll.sum(dim=1)
    )


    return sequence_logps


# ============================================================
# ONE POLICY/REFERENCE PAIR COMPUTATION
# ============================================================

def compute_pair_quantities(
    *,
    policy_model,
    reference_model,
    tokenizer,
    prompt,
    chosen,
    rejected,
    max_length,
    beta,
    policy_device,
    reference_device,
    require_policy_grad,
):
    """
    Compute:
      policy chosen/rejected logps
      reference chosen/rejected logps
      DPO loss
      implicit rewards
      reward margin
      reward correctness
    """

    prepared = prepare_dpo_pair_cpu(
        tokenizer=tokenizer,
        prompt=prompt,
        chosen=chosen,
        rejected=rejected,
        max_length=max_length,
    )

    cpu_batch = prepared["batch"]


    # --------------------------------------------------------
    # POLICY
    # --------------------------------------------------------

    policy_batch = (
        move_batch_to_device(
            cpu_batch,
            policy_device,
        )
    )


    grad_context = (
        torch.enable_grad()
        if require_policy_grad
        else torch.no_grad()
    )


    with grad_context:
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            policy_logps = (
                response_logps_from_joint_batch(
                    policy_model,
                    policy_batch,
                )
            )


    policy_chosen = (
        policy_logps[0]
    )

    policy_rejected = (
        policy_logps[1]
    )


    # --------------------------------------------------------
    # REFERENCE
    # --------------------------------------------------------

    reference_batch = (
        move_batch_to_device(
            cpu_batch,
            reference_device,
        )
    )


    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            reference_logps = (
                response_logps_from_joint_batch(
                    reference_model,
                    reference_batch,
                )
            )


    # Bring frozen scalar values to policy GPU.
    reference_logps = reference_logps.to(
        policy_device
    )


    reference_chosen = (
        reference_logps[0]
    )

    reference_rejected = (
        reference_logps[1]
    )


    # --------------------------------------------------------
    # DPO
    # --------------------------------------------------------

    dpo_loss = (
        compute_dpo_loss_from_logps_impl(
            policy_chosen_logps=(
                policy_chosen.unsqueeze(0)
            ),
            policy_rejected_logps=(
                policy_rejected.unsqueeze(0)
            ),
            reference_chosen_logps=(
                reference_chosen.unsqueeze(0)
            ),
            reference_rejected_logps=(
                reference_rejected.unsqueeze(0)
            ),
            beta=beta,
        )
    )


    reward_chosen = (
        beta
        * (
            policy_chosen
            - reference_chosen
        )
    )

    reward_rejected = (
        beta
        * (
            policy_rejected
            - reference_rejected
        )
    )

    reward_margin = (
        reward_chosen
        - reward_rejected
    )


    # Strict >, exactly as specified by the manual.
    reward_correct = (
        reward_chosen
        > reward_rejected
    )


    return {
        "loss": dpo_loss,

        "policy_chosen_logp": (
            policy_chosen
        ),

        "policy_rejected_logp": (
            policy_rejected
        ),

        "reference_chosen_logp": (
            reference_chosen
        ),

        "reference_rejected_logp": (
            reference_rejected
        ),

        "reward_chosen": (
            reward_chosen
        ),

        "reward_rejected": (
            reward_rejected
        ),

        "reward_margin": (
            reward_margin
        ),

        "reward_correct": (
            reward_correct
        ),

        "chosen_response_tokens": (
            prepared[
                "chosen_response_tokens"
            ]
        ),

        "rejected_response_tokens": (
            prepared[
                "rejected_response_tokens"
            ]
        ),
    }


# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def evaluate_dpo(
    *,
    policy_model,
    reference_model,
    tokenizer,
    dataloader,
    max_length,
    beta,
    policy_device,
    reference_device,
    max_batches=None,
):
    """
    Validation metrics required by the assignment:

      - mean DPO loss
      - reward accuracy, strict rw > rl
      - mean reward margin
    """

    policy_was_training = (
        policy_model.training
    )

    reference_was_training = (
        reference_model.training
    )


    policy_model.eval()
    reference_model.eval()


    total_loss = 0.0
    correct = 0
    total_margin = 0.0

    total_examples = 0


    try:
        for batch_index, raw_batch in enumerate(
            dataloader
        ):

            if (
                max_batches is not None
                and batch_index >= max_batches
            ):
                break


            # Our reference DPO setup has pair batch size 1.
            if len(
                raw_batch["prompts"]
            ) != 1:
                raise ValueError(
                    "evaluate_dpo expects "
                    "per-device pair batch size 1."
                )


            q = compute_pair_quantities(
                policy_model=policy_model,
                reference_model=reference_model,
                tokenizer=tokenizer,

                prompt=(
                    raw_batch["prompts"][0]
                ),

                chosen=(
                    raw_batch["chosen"][0]
                ),

                rejected=(
                    raw_batch["rejected"][0]
                ),

                max_length=max_length,
                beta=beta,

                policy_device=(
                    policy_device
                ),

                reference_device=(
                    reference_device
                ),

                require_policy_grad=False,
            )


            total_loss += float(
                q["loss"]
                .detach()
                .float()
                .item()
            )

            correct += int(
                q["reward_correct"]
                .detach()
                .item()
            )

            total_margin += float(
                q["reward_margin"]
                .detach()
                .float()
                .item()
            )

            total_examples += 1


    finally:
        if policy_was_training:
            policy_model.train()

        if reference_was_training:
            reference_model.train()


    if total_examples == 0:
        raise RuntimeError(
            "No DPO validation examples evaluated."
        )


    return {
        "dpo_loss": (
            total_loss
            / total_examples
        ),

        "reward_accuracy": (
            correct
            / total_examples
        ),

        "reward_margin": (
            total_margin
            / total_examples
        ),

        "examples": (
            total_examples
        ),
    }

In [ ]:
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


POLICY_DEVICE = torch.device(
    "cuda:0"
)

REFERENCE_DEVICE = torch.device(
    "cuda:1"
)


# ============================================================
# RELEASE SFT GENERATION MODEL
# ============================================================

del sft_model

gc.collect()

torch.cuda.empty_cache()


print(
    "GPU 0 allocated after cleanup:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)


# ============================================================
# TOKENIZER
# ============================================================

dpo_tokenizer = AutoTokenizer.from_pretrained(
    SFT_BEST_DIR,
    use_fast=True,
)


# ============================================================
# TRAINABLE POLICY: cuda:0, FP32 weights
# ============================================================

policy_model = (
    AutoModelForCausalLM.from_pretrained(
        SFT_BEST_DIR,
        dtype=torch.float32,
    )
)

policy_model = policy_model.to(
    POLICY_DEVICE
)

policy_model.config.use_cache = False

policy_model.train()


# ============================================================
# FROZEN REFERENCE: cuda:1, FP32 weights
# ============================================================

reference_model = (
    AutoModelForCausalLM.from_pretrained(
        SFT_BEST_DIR,
        dtype=torch.float32,
    )
)

reference_model = reference_model.to(
    REFERENCE_DEVICE
)

reference_model.config.use_cache = False

reference_model.eval()


for parameter in reference_model.parameters():
    parameter.requires_grad_(False)


# ============================================================
# VERIFY
# ============================================================

print("\n===== DPO MODELS LOADED =====")

print(
    "Policy device:",
    next(
        policy_model.parameters()
    ).device,
)

print(
    "Policy dtype:",
    next(
        policy_model.parameters()
    ).dtype,
)

print(
    "Policy trainable parameters:",
    f"{sum(
        p.numel()
        for p in policy_model.parameters()
        if p.requires_grad
    ):,}",
)


print(
    "\nReference device:",
    next(
        reference_model.parameters()
    ).device,
)

print(
    "Reference dtype:",
    next(
        reference_model.parameters()
    ).dtype,
)

print(
    "Reference trainable parameters:",
    f"{sum(
        p.numel()
        for p in reference_model.parameters()
        if p.requires_grad
    ):,}",
)


for gpu_id in [0, 1]:
    print(
        f"\nGPU {gpu_id} allocated:",
        round(
            torch.cuda.memory_allocated(
                gpu_id
            )
            / 1024**3,
            3,
        ),
        "GB",
    )

In [ ]:
import importlib
import math

import dpo_training

dpo_training = importlib.reload(
    dpo_training
)

from dpo_training import (
    compute_pair_quantities,
    evaluate_dpo,
)


raw_pair = next(
    iter(hh_val_loader)
)


pair_result = compute_pair_quantities(
    policy_model=policy_model,
    reference_model=reference_model,
    tokenizer=dpo_tokenizer,

    prompt=raw_pair["prompts"][0],
    chosen=raw_pair["chosen"][0],
    rejected=raw_pair[
        "rejected"
    ][0],

    max_length=HH_DPO_CONFIG[
        "context_length"
    ],

    beta=HH_DPO_CONFIG["beta"],

    policy_device=POLICY_DEVICE,
    reference_device=(
        REFERENCE_DEVICE
    ),

    require_policy_grad=True,
)


print(
    "===== INITIAL HH DPO PAIR CHECK ====="
)

print(
    "Policy chosen logp:",
    pair_result[
        "policy_chosen_logp"
    ].item(),
)

print(
    "Reference chosen logp:",
    pair_result[
        "reference_chosen_logp"
    ].item(),
)

print(
    "\nPolicy rejected logp:",
    pair_result[
        "policy_rejected_logp"
    ].item(),
)

print(
    "Reference rejected logp:",
    pair_result[
        "reference_rejected_logp"
    ].item(),
)


print(
    "\nDPO loss:",
    pair_result["loss"].item(),
)

print(
    "Expected log(2):",
    math.log(2),
)

print(
    "\nReward chosen:",
    pair_result[
        "reward_chosen"
    ].item(),
)

print(
    "Reward rejected:",
    pair_result[
        "reward_rejected"
    ].item(),
)

print(
    "Reward margin:",
    pair_result[
        "reward_margin"
    ].item(),
)

print(
    "Reward correct:",
    pair_result[
        "reward_correct"
    ].item(),
)


# Test policy gradients.
policy_model.zero_grad(
    set_to_none=True
)

pair_result[
    "loss"
].backward()


policy_has_grad = any(
    p.grad is not None
    and torch.isfinite(
        p.grad
    ).all()
    and p.grad.detach().abs().sum() > 0
    for p in policy_model.parameters()
)


reference_has_grad = any(
    p.grad is not None
    for p in reference_model.parameters()
)


print(
    "\nPolicy received gradient:",
    policy_has_grad,
)

print(
    "Reference received gradient:",
    reference_has_grad,
)


policy_model.zero_grad(
    set_to_none=True
)

In [ ]:
import gc
import torch


# The pair_result object retains the computation graph from
# our gradient sanity test until it is deleted.
cleanup_names = [
    "pair_result",
    "loss",
    "generated",
    "new_tokens",
    "input_ids",
    "attention_mask",
]

for name in cleanup_names:
    if name in globals():
        del globals()[name]


policy_model.zero_grad(
    set_to_none=True
)

gc.collect()

torch.cuda.empty_cache()


print("===== GPU MEMORY AFTER CLEANUP =====")

for gpu_id in [0, 1]:
    print(
        f"GPU {gpu_id} allocated:",
        round(
            torch.cuda.memory_allocated(gpu_id)
            / 1024**3,
            3,
        ),
        "GB",
    )

    print(
        f"GPU {gpu_id} reserved:",
        round(
            torch.cuda.memory_reserved(gpu_id)
            / 1024**3,
            3,
        ),
        "GB",
    )

In [ ]:
import json
import torch

from dpo_training import evaluate_dpo


torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)


initial_hh_smoke = evaluate_dpo(
    policy_model=policy_model,
    reference_model=reference_model,
    tokenizer=dpo_tokenizer,

    dataloader=hh_val_loader,

    max_length=HH_DPO_CONFIG[
        "context_length"
    ],

    beta=HH_DPO_CONFIG["beta"],

    policy_device=POLICY_DEVICE,
    reference_device=REFERENCE_DEVICE,

    max_batches=10,
)


print(
    "===== INITIAL HH DPO VALIDATION SMOKE ====="
)

print(
    json.dumps(
        initial_hh_smoke,
        indent=2,
    )
)


print("\n===== PEAK MEMORY =====")

print(
    "GPU 0:",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1:",
    round(
        torch.cuda.max_memory_allocated(1)
        / 1024**3,
        3,
    ),
    "GB",
)

In [ ]:
%%writefile -a dpo_training.py


# ============================================================
# CHECKPOINT SAVING
# ============================================================

def save_dpo_checkpoint(
    *,
    model,
    tokenizer,
    output_dir,
    metadata,
):
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        output_dir,
        safe_serialization=True,
    )

    tokenizer.save_pretrained(
        output_dir,
    )

    write_json(
        output_dir
        / "checkpoint_metadata.json",
        metadata,
    )


# ============================================================
# FULL FROM-SCRATCH DPO TRAINING LOOP
# ============================================================

def train_dpo(
    *,
    policy_model,
    reference_model,
    tokenizer,

    train_loader,
    val_loader,

    config,
    output_dir,

    policy_device,
    reference_device,
):
    """
    Full manual DPO training loop.

    Pair batch size = 1.
    Gradient accumulation gives effective pair batch size 32.

    The final partial accumulation window is normalized by its
    actual number of pairs rather than always dividing by 32.
    """

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    metrics_path = (
        output_dir
        / "metrics.jsonl"
    )

    summary_path = (
        output_dir
        / "summary.json"
    )

    best_dir = (
        output_dir
        / "best_checkpoint"
    )

    final_dir = (
        output_dir
        / "final_checkpoint"
    )


    # --------------------------------------------------------
    # Refuse to accidentally append a new run onto an old one.
    # --------------------------------------------------------

    if metrics_path.exists():
        raise RuntimeError(
            f"{metrics_path} already exists. "
            "Refusing to mix two training runs."
        )

    if summary_path.exists():
        raise RuntimeError(
            f"{summary_path} already exists."
        )

    if best_dir.exists():
        raise RuntimeError(
            f"{best_dir} already exists."
        )

    if final_dir.exists():
        raise RuntimeError(
            f"{final_dir} already exists."
        )


    # --------------------------------------------------------
    # MODELS
    # --------------------------------------------------------

    policy_model.train()

    reference_model.eval()

    reference_model.requires_grad_(
        False
    )


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        policy_model.parameters(),

        lr=config[
            "learning_rate"
        ],

        betas=(
            config["adam_beta1"],
            config["adam_beta2"],
        ),

        eps=config[
            "adam_epsilon"
        ],

        weight_decay=config[
            "weight_decay"
        ],
    )


    # FP32 trainable weights + FP16 autocast + gradient scaling.
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    accumulation_steps = config[
        "gradient_accumulation_steps"
    ]

    total_micro_batches = len(
        train_loader
    )

    expected_updates = math.ceil(
        total_micro_batches
        / accumulation_steps
    )


    # --------------------------------------------------------
    # METRIC STATE
    # --------------------------------------------------------

    update_index = 0

    skipped_optimizer_steps = 0

    total_train_loss_sum = 0.0
    total_train_examples = 0

    window_loss_sum = 0.0
    window_examples = 0


    best_reward_accuracy = (
        -float("inf")
    )

    best_reward_margin = (
        -float("inf")
    )

    best_update = None

    best_validation = None


    start_time = time.time()


    # --------------------------------------------------------
    # FULL INITIAL VALIDATION
    # --------------------------------------------------------

    print(
        "Running full initial validation..."
    )

    initial_validation = evaluate_dpo(
        policy_model=policy_model,
        reference_model=reference_model,
        tokenizer=tokenizer,

        dataloader=val_loader,

        max_length=config[
            "context_length"
        ],

        beta=config["beta"],

        policy_device=policy_device,
        reference_device=(
            reference_device
        ),
    )


    append_jsonl(
        metrics_path,
        {
            "type": (
                "initial_validation"
            ),

            "update": 0,

            **initial_validation,
        },
    )


    print(
        "Initial validation:",
        initial_validation,
    )


    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    progress = tqdm(
        total=expected_updates,
        desc="HH DPO updates",
    )


    for micro_index, raw_batch in enumerate(
        train_loader
    ):
        # micro_index is zero-based.


        if len(
            raw_batch["prompts"]
        ) != 1:
            raise ValueError(
                "Reference DPO recipe expects "
                "per-device pair batch size 1."
            )


        # ----------------------------------------------------
        # Determine actual size of THIS accumulation window.
        #
        # Normal windows = 32 pairs.
        # Final window = 24 pairs for 15,000 examples.
        # ----------------------------------------------------

        window_start = (
            micro_index
            // accumulation_steps
        ) * accumulation_steps

        current_window_size = min(
            accumulation_steps,
            total_micro_batches
            - window_start,
        )


        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        quantities = (
            compute_pair_quantities(
                policy_model=policy_model,
                reference_model=(
                    reference_model
                ),
                tokenizer=tokenizer,

                prompt=(
                    raw_batch[
                        "prompts"
                    ][0]
                ),

                chosen=(
                    raw_batch[
                        "chosen"
                    ][0]
                ),

                rejected=(
                    raw_batch[
                        "rejected"
                    ][0]
                ),

                max_length=config[
                    "context_length"
                ],

                beta=config["beta"],

                policy_device=(
                    policy_device
                ),

                reference_device=(
                    reference_device
                ),

                require_policy_grad=True,
            )
        )


        loss = quantities["loss"]


        if not torch.isfinite(
            loss
        ):
            raise RuntimeError(
                "Non-finite DPO loss at "
                f"micro-batch {micro_index}."
            )


        raw_loss_value = float(
            loss
            .detach()
            .float()
            .item()
        )


        total_train_loss_sum += (
            raw_loss_value
        )

        total_train_examples += 1

        window_loss_sum += (
            raw_loss_value
        )

        window_examples += 1


        # ----------------------------------------------------
        # BACKWARD
        #
        # Normalize by ACTUAL accumulation-window size.
        # ----------------------------------------------------

        scaled_loss = (
            loss
            / current_window_size
        )


        scaler.scale(
            scaled_loss
        ).backward()


        # ----------------------------------------------------
        # END OF ACCUMULATION WINDOW?
        # ----------------------------------------------------

        end_of_window = (
            (
                (micro_index + 1)
                % accumulation_steps
            )
            == 0
            or
            (micro_index + 1)
            == total_micro_batches
        )


        if not end_of_window:
            continue


        update_index += 1


        # ----------------------------------------------------
        # UNSCALE + CLIP
        # ----------------------------------------------------

        scaler.unscale_(
            optimizer
        )


        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                policy_model.parameters(),
                max_norm=config[
                    "gradient_clip_norm"
                ],
            )
        )


        grad_norm_value = float(
            grad_norm
            .detach()
            .float()
            .item()
        )


        # ----------------------------------------------------
        # OPTIMIZER STEP
        # ----------------------------------------------------

        old_scale = (
            scaler.get_scale()
        )


        scaler.step(
            optimizer
        )

        scaler.update()


        new_scale = (
            scaler.get_scale()
        )


        optimizer_step_skipped = (
            new_scale < old_scale
        )


        if optimizer_step_skipped:
            skipped_optimizer_steps += 1


        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # TRAIN METRIC FOR THIS OPTIMIZER WINDOW
        # ----------------------------------------------------

        mean_window_loss = (
            window_loss_sum
            / window_examples
        )


        train_record = {
            "type": "train_update",

            "update": update_index,

            "micro_batches_seen": (
                micro_index + 1
            ),

            "pairs_in_update": (
                window_examples
            ),

            "train_dpo_loss": (
                mean_window_loss
            ),

            "grad_norm_before_clip": (
                grad_norm_value
            ),

            "learning_rate": (
                optimizer.param_groups[
                    0
                ]["lr"]
            ),

            "grad_scale": (
                new_scale
            ),

            "optimizer_step_skipped": (
                optimizer_step_skipped
            ),
        }


        append_jsonl(
            metrics_path,
            train_record,
        )


        progress.update(1)

        progress.set_postfix(
            loss=(
                f"{mean_window_loss:.4f}"
            ),
            grad=(
                f"{grad_norm_value:.3f}"
            ),
        )


        # Reset accumulation-window metric state.
        window_loss_sum = 0.0
        window_examples = 0


        # ----------------------------------------------------
        # VALIDATION
        #
        # Every N updates and ALWAYS at final update.
        # ----------------------------------------------------

        should_validate = (
            update_index
            % config[
                "eval_every_updates"
            ]
            == 0

            or

            update_index
            == expected_updates
        )


        if not should_validate:
            continue


        validation = evaluate_dpo(
            policy_model=policy_model,
            reference_model=(
                reference_model
            ),
            tokenizer=tokenizer,

            dataloader=val_loader,

            max_length=config[
                "context_length"
            ],

            beta=config["beta"],

            policy_device=(
                policy_device
            ),

            reference_device=(
                reference_device
            ),
        )


        validation_record = {
            "type": "validation",

            "update": update_index,

            **validation,
        }


        append_jsonl(
            metrics_path,
            validation_record,
        )


        print(
            "\nValidation at update",
            update_index,
            ":",
            validation,
        )


        # ----------------------------------------------------
        # BEST CHECKPOINT
        #
        # Primary criterion:
        #   highest reward accuracy
        #
        # Tie-break:
        #   higher reward margin
        # ----------------------------------------------------

        accuracy = (
            validation[
                "reward_accuracy"
            ]
        )

        margin = (
            validation[
                "reward_margin"
            ]
        )


        is_better = (
            accuracy
            > best_reward_accuracy
        )


        if (
            accuracy
            == best_reward_accuracy
            and
            margin
            > best_reward_margin
        ):
            is_better = True


        if is_better:
            best_reward_accuracy = (
                accuracy
            )

            best_reward_margin = (
                margin
            )

            best_update = (
                update_index
            )

            best_validation = (
                validation
            )


            save_dpo_checkpoint(
                model=policy_model,
                tokenizer=tokenizer,

                output_dir=best_dir,

                metadata={
                    "checkpoint_type": (
                        "best"
                    ),

                    "update": (
                        update_index
                    ),

                    "selection_criterion": (
                        "highest validation "
                        "reward accuracy; "
                        "reward margin "
                        "used only as tie-break"
                    ),

                    **validation,
                },
            )


            print(
                "New best checkpoint:",
                best_update,
            )


    progress.close()


    # --------------------------------------------------------
    # SAFETY CHECK
    # --------------------------------------------------------

    if update_index != expected_updates:
        raise RuntimeError(
            "Unexpected optimizer update count: "
            f"{update_index} vs "
            f"{expected_updates}"
        )


    if total_train_examples != (
        total_micro_batches
    ):
        raise RuntimeError(
            "Not all training pairs were processed."
        )


    # --------------------------------------------------------
    # FINAL VALIDATION
    #
    # The final update was already validated above because
    # should_validate includes expected_updates.
    # Reuse that result rather than running another 1000 pairs.
    # --------------------------------------------------------

    final_validation = validation


    # --------------------------------------------------------
    # FINAL CHECKPOINT
    # --------------------------------------------------------

    save_dpo_checkpoint(
        model=policy_model,
        tokenizer=tokenizer,

        output_dir=final_dir,

        metadata={
            "checkpoint_type": "final",

            "update": (
                update_index
            ),

            **final_validation,
        },
    )


    elapsed_seconds = (
        time.time()
        - start_time
    )


    summary = {
        "completed": True,

        "optimizer_updates": (
            update_index
        ),

        "expected_optimizer_updates": (
            expected_updates
        ),

        "skipped_optimizer_steps": (
            skipped_optimizer_steps
        ),

        "overall_training_dpo_loss": (
            total_train_loss_sum
            / total_train_examples
        ),

        "initial_validation": (
            initial_validation
        ),

        "final_validation": (
            final_validation
        ),

        "best_validation": (
            best_validation
        ),

        "best_reward_accuracy": (
            best_reward_accuracy
        ),

        "best_reward_margin": (
            best_reward_margin
        ),

        "best_update": (
            best_update
        ),

        "elapsed_seconds": (
            elapsed_seconds
        ),

        "elapsed_hours": (
            elapsed_seconds
            / 3600.0
        ),

        "best_checkpoint": (
            str(best_dir)
        ),

        "final_checkpoint": (
            str(final_dir)
        ),

        "metrics_file": (
            str(metrics_path)
        ),
    }


    write_json(
        summary_path,
        summary,
    )


    print(
        "\n===== DPO TRAINING COMPLETE ====="
    )

    print(
        json.dumps(
            summary,
            indent=2,
        )
    )


    return {
        "summary": summary,
        "optimizer": optimizer,
        "scaler": scaler,
    }

In [ ]:
import importlib
import dpo_training

dpo_training = importlib.reload(
    dpo_training
)

from dpo_training import train_dpo


hh_train_generator = torch.Generator()

hh_train_generator.manual_seed(
    HH_DPO_CONFIG["seed"]
)


hh_train_loader = DataLoader(
    hh_train_fixed,

    batch_size=1,

    shuffle=True,

    generator=hh_train_generator,

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,

    drop_last=False,
)


print(
    "Train micro-batches:",
    len(hh_train_loader),
)

print(
    "Expected optimizer updates:",
    math.ceil(
        len(hh_train_loader)
        / 32
    ),
)


first_train_batch = next(
    iter(hh_train_loader)
)

print(
    "\nFirst shuffled pair ID:",
    first_train_batch["ids"][0],
)

In [ ]:
import gc
import torch

from dpo_training import (
    compute_pair_quantities,
)


# Fresh deterministic loader for this isolated preflight.
preflight_generator = torch.Generator()

preflight_generator.manual_seed(42)

hh_preflight_loader = DataLoader(
    hh_train_fixed,

    batch_size=1,
    shuffle=True,

    generator=preflight_generator,

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,
)


preflight_optimizer = torch.optim.AdamW(
    policy_model.parameters(),

    lr=HH_DPO_CONFIG[
        "learning_rate"
    ],

    betas=(
        HH_DPO_CONFIG["adam_beta1"],
        HH_DPO_CONFIG["adam_beta2"],
    ),

    eps=HH_DPO_CONFIG[
        "adam_epsilon"
    ],

    weight_decay=HH_DPO_CONFIG[
        "weight_decay"
    ],
)


preflight_scaler = (
    torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )
)


policy_model.zero_grad(
    set_to_none=True
)

torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)


preflight_losses = []


for i, raw_batch in enumerate(
    hh_preflight_loader
):
    if i >= 32:
        break

    q = compute_pair_quantities(
        policy_model=policy_model,
        reference_model=reference_model,
        tokenizer=dpo_tokenizer,

        prompt=raw_batch[
            "prompts"
        ][0],

        chosen=raw_batch[
            "chosen"
        ][0],

        rejected=raw_batch[
            "rejected"
        ][0],

        max_length=512,
        beta=0.1,

        policy_device=POLICY_DEVICE,
        reference_device=(
            REFERENCE_DEVICE
        ),

        require_policy_grad=True,
    )


    preflight_losses.append(
        float(
            q["loss"]
            .detach()
            .float()
            .item()
        )
    )


    preflight_scaler.scale(
        q["loss"] / 32.0
    ).backward()


preflight_scaler.unscale_(
    preflight_optimizer
)


preflight_grad_norm = (
    torch.nn.utils.clip_grad_norm_(
        policy_model.parameters(),
        max_norm=1.0,
    )
)


print(
    "===== 32-PAIR ACCUMULATION PREFLIGHT ====="
)

print(
    "Mean DPO loss:",
    sum(preflight_losses)
    / len(preflight_losses),
)

print(
    "Gradient norm before clipping:",
    float(preflight_grad_norm),
)

print(
    "Gradient scale:",
    preflight_scaler.get_scale(),
)


print("\nPeak allocated memory:")

print(
    "GPU 0:",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1:",
    round(
        torch.cuda.max_memory_allocated(1)
        / 1024**3,
        3,
    ),
    "GB",
)


# IMPORTANT:
# Do NOT step the optimizer. This is only a gradient/memory test.

policy_model.zero_grad(
    set_to_none=True
)

del q
del preflight_optimizer
del preflight_scaler
del hh_preflight_loader

gc.collect()

torch.cuda.empty_cache()


print(
    "\nGPU 0 allocated after preflight cleanup:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1 allocated after preflight cleanup:",
    round(
        torch.cuda.memory_allocated(1)
        / 1024**3,
        3,
    ),
    "GB",
)

In [ ]:
import gc
import torch

from dpo_training import (
    compute_pair_quantities,
)


TEST_GRAD_SCALE = 512.0


# ============================================================
# SAME DETERMINISTIC FIRST 32 PAIRS
# ============================================================

test_generator = torch.Generator()
test_generator.manual_seed(42)

hh_scale_test_loader = DataLoader(
    hh_train_fixed,

    batch_size=1,
    shuffle=True,

    generator=test_generator,

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,
)


test_optimizer = torch.optim.AdamW(
    policy_model.parameters(),

    lr=HH_DPO_CONFIG[
        "learning_rate"
    ],

    betas=(
        HH_DPO_CONFIG["adam_beta1"],
        HH_DPO_CONFIG["adam_beta2"],
    ),

    eps=HH_DPO_CONFIG[
        "adam_epsilon"
    ],

    weight_decay=HH_DPO_CONFIG[
        "weight_decay"
    ],
)


test_scaler = torch.amp.GradScaler(
    "cuda",

    init_scale=TEST_GRAD_SCALE,

    # The whole HH run is only 469 updates.
    # This prevents automatic scale growth during this run.
    growth_interval=2000,

    enabled=True,
)


policy_model.zero_grad(
    set_to_none=True
)


torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)


losses = []


# ============================================================
# ACCUMULATE EXACTLY 32 PAIRS
# ============================================================

for i, raw_batch in enumerate(
    hh_scale_test_loader
):
    if i >= 32:
        break

    q = compute_pair_quantities(
        policy_model=policy_model,
        reference_model=reference_model,
        tokenizer=dpo_tokenizer,

        prompt=raw_batch[
            "prompts"
        ][0],

        chosen=raw_batch[
            "chosen"
        ][0],

        rejected=raw_batch[
            "rejected"
        ][0],

        max_length=512,
        beta=0.1,

        policy_device=POLICY_DEVICE,
        reference_device=REFERENCE_DEVICE,

        require_policy_grad=True,
    )


    losses.append(
        float(
            q["loss"]
            .detach()
            .float()
            .item()
        )
    )


    test_scaler.scale(
        q["loss"] / 32.0
    ).backward()


# ============================================================
# UNSCALE
# ============================================================

test_scaler.unscale_(
    test_optimizer
)


# ============================================================
# EXPLICITLY CHECK EVERY GRADIENT BEFORE CLIPPING
# ============================================================

nonfinite_gradient_tensors = 0
gradient_tensors = 0
max_abs_gradient = 0.0


for parameter in policy_model.parameters():

    if parameter.grad is None:
        continue

    gradient_tensors += 1

    grad = parameter.grad.detach()

    if not torch.isfinite(grad).all():
        nonfinite_gradient_tensors += 1

    else:
        max_abs_gradient = max(
            max_abs_gradient,
            float(
                grad.abs().max().item()
            ),
        )


all_gradients_finite = (
    nonfinite_gradient_tensors == 0
)


print(
    "===== DPO GRAD-SCALE TEST ====="
)

print(
    "Test gradient scale:",
    test_scaler.get_scale(),
)

print(
    "Mean DPO loss:",
    sum(losses) / len(losses),
)

print(
    "Gradient tensors:",
    gradient_tensors,
)

print(
    "Non-finite gradient tensors:",
    nonfinite_gradient_tensors,
)

print(
    "All gradients finite:",
    all_gradients_finite,
)

print(
    "Max absolute unscaled gradient:",
    max_abs_gradient,
)


# ============================================================
# ONLY COMPUTE/CLIP NORM IF EVERYTHING IS FINITE
# ============================================================

if all_gradients_finite:

    grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            policy_model.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True,
        )
    )

    print(
        "Gradient norm before clipping:",
        float(
            grad_norm
            .detach()
            .float()
            .item()
        ),
    )

else:

    print(
        "Gradient norm not computed because "
        "non-finite gradients remain."
    )


print("\nPeak allocated memory:")

print(
    "GPU 0:",
    round(
        torch.cuda.max_memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1:",
    round(
        torch.cuda.max_memory_allocated(1)
        / 1024**3,
        3,
    ),
    "GB",
)


# ============================================================
# NO OPTIMIZER STEP — CLEAN EVERYTHING
# ============================================================

policy_model.zero_grad(
    set_to_none=True
)

del q
del test_optimizer
del test_scaler
del hh_scale_test_loader

gc.collect()
torch.cuda.empty_cache()


print(
    "\nGPU 0 allocated after cleanup:",
    round(
        torch.cuda.memory_allocated(0)
        / 1024**3,
        3,
    ),
    "GB",
)

print(
    "GPU 1 allocated after cleanup:",
    round(
        torch.cuda.memory_allocated(1)
        / 1024**3,
        3,
    ),
    "GB",
)

In [ ]:
from pathlib import Path
import json


# ============================================================
# RECORD THE NUMERICAL-STABILITY CHOICE
# ============================================================

HH_DPO_CONFIG["amp_init_scale"] = 512.0
HH_DPO_CONFIG["amp_growth_interval"] = 2000

HH_DPO_CONFIG["amp_scale_rationale"] = (
    "Default GradScaler init_scale=65536 produced non-finite "
    "gradients during a deterministic 32-pair DPO accumulation "
    "preflight. init_scale=512 produced finite gradients for "
    "all 290 gradient tensors; unscaled gradient norm was "
    "45.956459 and was handled by the configured clip norm 1.0."
)


HH_DPO_CONFIG_PATH.write_text(
    json.dumps(
        HH_DPO_CONFIG,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# PATCH dpo_training.py
# ============================================================

DPO_TRAINING_PATH = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "dpo_training.py"
)


text = DPO_TRAINING_PATH.read_text(
    encoding="utf-8"
)


old_scaler_block = '''    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )
'''


new_scaler_block = '''    scaler = torch.amp.GradScaler(
        "cuda",
        init_scale=config["amp_init_scale"],
        growth_interval=config["amp_growth_interval"],
        enabled=True,
    )
'''


assert old_scaler_block in text, (
    "Could not find the expected GradScaler block."
)


text = text.replace(
    old_scaler_block,
    new_scaler_block,
    1,
)


DPO_TRAINING_PATH.write_text(
    text,
    encoding="utf-8",
)


# ============================================================
# VERIFY PATCH
# ============================================================

patched_text = DPO_TRAINING_PATH.read_text(
    encoding="utf-8"
)


assert (
    'init_scale=config["amp_init_scale"]'
    in patched_text
)

assert (
    'growth_interval=config["amp_growth_interval"]'
    in patched_text
)


saved_config = json.loads(
    HH_DPO_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


assert saved_config[
    "amp_init_scale"
] == 512.0

assert saved_config[
    "amp_growth_interval"
] == 2000


print("===== HH-DPO AMP PATCH VERIFIED =====")

print(
    "AMP initial scale:",
    saved_config["amp_init_scale"],
)

print(
    "AMP growth interval:",
    saved_config["amp_growth_interval"],
)

print(
    "\nRationale:"
)

print(
    saved_config["amp_scale_rationale"]
)

print(
    "\n✓ dpo_training.py patched successfully"
)

In [ ]:
import gc
import json
import math
import time
import importlib
from pathlib import Path

import torch
from torch.utils.data import DataLoader

import dpo_training


# ============================================================
# RELOAD PATCHED TRAINING CODE
# ============================================================

dpo_training = importlib.reload(
    dpo_training
)

from dpo_training import train_dpo


# ============================================================
# VERIFY THIS IS A FRESH RUN
# ============================================================

HH_METRICS_PATH = (
    HH_DPO_RUN_DIR
    / "metrics.jsonl"
)

HH_SUMMARY_PATH = (
    HH_DPO_RUN_DIR
    / "summary.json"
)

HH_BEST_DIR = (
    HH_DPO_RUN_DIR
    / "best_checkpoint"
)

HH_FINAL_DIR = (
    HH_DPO_RUN_DIR
    / "final_checkpoint"
)


for path in [
    HH_METRICS_PATH,
    HH_SUMMARY_PATH,
    HH_BEST_DIR,
    HH_FINAL_DIR,
]:
    assert not path.exists(), (
        f"Run artifact already exists: {path}"
    )


# ============================================================
# FRESH TRAIN LOADER
#
# Important: this generator has never been iterated.
# ============================================================

final_hh_train_generator = (
    torch.Generator()
)

final_hh_train_generator.manual_seed(
    HH_DPO_CONFIG["seed"]
)


final_hh_train_loader = DataLoader(
    hh_train_fixed,

    batch_size=HH_DPO_CONFIG[
        "per_device_batch_size"
    ],

    shuffle=True,

    generator=(
        final_hh_train_generator
    ),

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,

    drop_last=False,
)


# Fresh deterministic validation loader too.
final_hh_val_loader = DataLoader(
    hh_val_fixed,

    batch_size=1,

    shuffle=False,

    collate_fn=hh_dpo_collate_fn,

    num_workers=0,

    drop_last=False,
)


assert len(
    final_hh_train_loader
) == 15000

assert len(
    final_hh_val_loader
) == 1000

assert math.ceil(
    len(final_hh_train_loader)
    / HH_DPO_CONFIG[
        "gradient_accumulation_steps"
    ]
) == 469


# ============================================================
# FINAL MODEL STATE CHECK
# ============================================================

policy_model.train()

reference_model.eval()

policy_model.zero_grad(
    set_to_none=True
)


assert next(
    policy_model.parameters()
).dtype == torch.float32

assert next(
    policy_model.parameters()
).device.type == "cuda"

assert next(
    policy_model.parameters()
).device.index == 0


assert next(
    reference_model.parameters()
).dtype == torch.float32

assert next(
    reference_model.parameters()
).device.type == "cuda"

assert next(
    reference_model.parameters()
).device.index == 1


assert not any(
    p.requires_grad
    for p in reference_model.parameters()
)


# ============================================================
# MEMORY CLEANUP / TRACKING
# ============================================================

gc.collect()
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)


print(
    "===== STARTING REAL HH-RLHF DPO ====="
)

print(
    "Train pairs:",
    len(final_hh_train_loader),
)

print(
    "Validation pairs:",
    len(final_hh_val_loader),
)

print(
    "Optimizer updates:",
    469,
)

print(
    "Gradient accumulation:",
    HH_DPO_CONFIG[
        "gradient_accumulation_steps"
    ],
)

print(
    "Learning rate:",
    HH_DPO_CONFIG[
        "learning_rate"
    ],
)

print(
    "Beta:",
    HH_DPO_CONFIG["beta"],
)

print(
    "AMP initial scale:",
    HH_DPO_CONFIG[
        "amp_init_scale"
    ],
)

print(
    "Gradient clip:",
    HH_DPO_CONFIG[
        "gradient_clip_norm"
    ],
)


print("\nStarting GPU memory:")

for gpu_id in [0, 1]:
    print(
        f"GPU {gpu_id}:",
        round(
            torch.cuda.memory_allocated(
                gpu_id
            )
            / 1024**3,
            3,
        ),
        "GB",
    )


# ============================================================
# ACTUAL TRAINING
# ============================================================

hh_dpo_wall_start = time.time()


hh_dpo_result = train_dpo(
    policy_model=policy_model,
    reference_model=reference_model,

    tokenizer=dpo_tokenizer,

    train_loader=(
        final_hh_train_loader
    ),

    val_loader=(
        final_hh_val_loader
    ),

    config=HH_DPO_CONFIG,

    output_dir=HH_DPO_RUN_DIR,

    policy_device=POLICY_DEVICE,

    reference_device=(
        REFERENCE_DEVICE
    ),
)


hh_dpo_wall_seconds = (
    time.time()
    - hh_dpo_wall_start
)


# ============================================================
# FINAL RUN INFORMATION
# ============================================================

print(
    "\n===== HH-DPO WALL CLOCK ====="
)

print(
    "Seconds:",
    round(
        hh_dpo_wall_seconds,
        2,
    ),
)

print(
    "Hours:",
    round(
        hh_dpo_wall_seconds
        / 3600,
        3,
    ),
)


print(
    "\n===== PEAK GPU MEMORY ====="
)

for gpu_id in [0, 1]:
    print(
        f"GPU {gpu_id}:",
        round(
            torch.cuda.max_memory_allocated(
                gpu_id
            )
            / 1024**3,
            3,
        ),
        "GB",
    )


print(
    "\n===== SAVED SUMMARY ====="
)

print(
    HH_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

In [ ]:
import json
import hashlib
from pathlib import Path


HH_BEST_DIR = (
    HH_DPO_RUN_DIR
    / "best_checkpoint"
)

HH_FINAL_DIR = (
    HH_DPO_RUN_DIR
    / "final_checkpoint"
)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ============================================================
# METADATA
# ============================================================

best_metadata = json.loads(
    (
        HH_BEST_DIR
        / "checkpoint_metadata.json"
    ).read_text(
        encoding="utf-8"
    )
)

final_metadata = json.loads(
    (
        HH_FINAL_DIR
        / "checkpoint_metadata.json"
    ).read_text(
        encoding="utf-8"
    )
)


print(
    "===== HH-DPO CHECKPOINT VERIFICATION ====="
)

print(
    "\nBest metadata:"
)

print(
    json.dumps(
        best_metadata,
        indent=2,
    )
)

print(
    "\nFinal metadata:"
)

print(
    json.dumps(
        final_metadata,
        indent=2,
    )
)


# ============================================================
# MODEL HASHES
# ============================================================

best_model_path = (
    HH_BEST_DIR
    / "model.safetensors"
)

final_model_path = (
    HH_FINAL_DIR
    / "model.safetensors"
)


assert best_model_path.exists()
assert final_model_path.exists()


print(
    "\nComputing model hashes..."
)

hh_best_model_sha256 = (
    sha256_file(
        best_model_path
    )
)

hh_final_model_sha256 = (
    sha256_file(
        final_model_path
    )
)


print(
    "\nBest model SHA256:",
    hh_best_model_sha256,
)

print(
    "Final model SHA256:",
    hh_final_model_sha256,
)

print(
    "Model weights identical:",
    (
        hh_best_model_sha256
        == hh_final_model_sha256
    ),
)


# ============================================================
# EXPECTED SELECTION
# ============================================================

assert (
    best_metadata["update"]
    == 469
)

assert (
    final_metadata["update"]
    == 469
)

assert abs(
    best_metadata[
        "reward_accuracy"
    ]
    - 0.633
) < 1e-12

assert abs(
    best_metadata[
        "reward_margin"
    ]
    - 0.23464530624076724
) < 1e-12


# Because the final update was also the best update,
# these two saves should contain the same model weights.
assert (
    hh_best_model_sha256
    == hh_final_model_sha256
)


# ============================================================
# METRICS-FILE SANITY
# ============================================================

with open(
    HH_METRICS_PATH,
    "r",
    encoding="utf-8",
) as f:
    hh_metric_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


metric_types = {}

for row in hh_metric_rows:
    metric_types[
        row["type"]
    ] = (
        metric_types.get(
            row["type"],
            0,
        )
        + 1
    )


print(
    "\nMetrics records:",
    len(hh_metric_rows),
)

print(
    "Metric types:",
    metric_types,
)


assert (
    metric_types[
        "train_update"
    ]
    == 469
)

assert (
    metric_types[
        "initial_validation"
    ]
    == 1
)

assert (
    metric_types[
        "validation"
    ]
    == 5
)


print(
    "\n✓ HH-RLHF DPO checkpoints "
    "and metrics verified successfully"
)

In [ ]:
import gc
import json
import torch
from pathlib import Path


# ============================================================
# RELEASE TRAINING-ONLY OBJECTS
# ============================================================

cleanup_names = [
    "hh_dpo_result",
    "policy_model",
    "reference_model",
    "final_hh_train_loader",
    "final_hh_val_loader",
    "optimizer",
    "scaler",
]

for name in cleanup_names:
    if name in globals():
        del globals()[name]


gc.collect()
torch.cuda.empty_cache()


print("===== GPU MEMORY AFTER HH TRAINING CLEANUP =====")

for gpu_id in [0, 1]:
    print(
        f"GPU {gpu_id} allocated:",
        round(
            torch.cuda.memory_allocated(gpu_id)
            / 1024**3,
            3,
        ),
        "GB",
    )


# ============================================================
# LOCATE PROVIDED HH EVALUATION SET
# ============================================================

HH_EVAL_PATH = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "eval_sets/hh_rlhf_eval_300_seed42.json"
)


print("\n===== HH EVALUATION FILE =====")

print("Path:")
print(HH_EVAL_PATH)

print(
    "Exists:",
    HH_EVAL_PATH.exists(),
)

assert HH_EVAL_PATH.exists(), (
    f"Could not find {HH_EVAL_PATH}"
)


# ============================================================
# LOAD
# ============================================================

with open(
    HH_EVAL_PATH,
    "r",
    encoding="utf-8",
) as f:
    hh_eval_raw = json.load(f)


print(
    "\nPython type:",
    type(hh_eval_raw).__name__,
)


if isinstance(hh_eval_raw, list):

    print(
        "Number of examples:",
        len(hh_eval_raw),
    )

    print(
        "\n===== FIRST 5 EXAMPLES ====="
    )

    for i, row in enumerate(
        hh_eval_raw[:5]
    ):
        print("\n" + "=" * 100)
        print("INDEX:", i)

        print("\nKEYS:")
        print(row.keys())

        print("\nFULL ROW:")
        print(
            json.dumps(
                row,
                indent=2,
                ensure_ascii=False,
            )
        )

elif isinstance(hh_eval_raw, dict):

    print(
        "\nTop-level keys:"
    )

    print(
        hh_eval_raw.keys()
    )

    print(
        "\nPreview:"
    )

    print(
        json.dumps(
            hh_eval_raw,
            indent=2,
            ensure_ascii=False,
        )[:5000]
    )

else:
    raise TypeError(
        "Unexpected HH evaluation JSON structure."
    )

In [ ]:
import gc
import torch
from collections import Counter

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

from data_utils import (
    format_instruction_prompt,
)


# ============================================================
# LOAD BEST HH-DPO CHECKPOINT
# ============================================================

HH_GENERATION_DEVICE = torch.device(
    "cuda:0"
)


gc.collect()
torch.cuda.empty_cache()


hh_dpo_tokenizer = (
    AutoTokenizer.from_pretrained(
        HH_BEST_DIR,
        use_fast=True,
    )
)


hh_dpo_model = (
    AutoModelForCausalLM.from_pretrained(
        HH_BEST_DIR,
        dtype=torch.float32,
    )
    .to(HH_GENERATION_DEVICE)
)


hh_dpo_model.eval()

hh_dpo_model.config.use_cache = True


print(
    "===== RELOADED HH-DPO MODEL ====="
)

print(
    "Class:",
    type(hh_dpo_model).__name__,
)

print(
    "Device:",
    next(
        hh_dpo_model.parameters()
    ).device,
)

print(
    "Dtype:",
    next(
        hh_dpo_model.parameters()
    ).dtype,
)

print(
    "EOS:",
    hh_dpo_tokenizer.eos_token_id,
)

print(
    "PAD:",
    hh_dpo_tokenizer.pad_token_id,
)

print(
    "Parameters:",
    f"{sum(
        p.numel()
        for p in hh_dpo_model.parameters()
    ):,}",
)


# ============================================================
# FROZEN DECODING POLICY
# ============================================================

HH_GENERATION_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 256,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,
}


# ============================================================
# REPETITION DIAGNOSTIC
# ============================================================

def repeated_ngram_fraction(
    text,
    n=4,
):
    tokens = text.split()

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens) - n + 1
        )
    ]

    counts = Counter(ngrams)

    repeated_occurrences = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return (
        repeated_occurrences
        / len(ngrams)
    )


# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_hh_dpo_one(
    instruction,
):
    prompt = (
        format_instruction_prompt(
            instruction
        )
    )

    encoded = hh_dpo_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded[
        "input_ids"
    ].to(
        HH_GENERATION_DEVICE
    )

    attention_mask = encoded[
        "attention_mask"
    ].to(
        HH_GENERATION_DEVICE
    )

    prompt_length = (
        input_ids.shape[1]
    )


    with torch.inference_mode():
        generated = (
            hh_dpo_model.generate(
                input_ids=input_ids,

                attention_mask=(
                    attention_mask
                ),

                do_sample=False,

                max_new_tokens=(
                    HH_GENERATION_CONFIG[
                        "max_new_tokens"
                    ]
                ),

                repetition_penalty=(
                    HH_GENERATION_CONFIG[
                        "repetition_penalty"
                    ]
                ),

                no_repeat_ngram_size=(
                    HH_GENERATION_CONFIG[
                        "no_repeat_ngram_size"
                    ]
                ),

                pad_token_id=(
                    hh_dpo_tokenizer.pad_token_id
                ),

                eos_token_id=(
                    hh_dpo_tokenizer.eos_token_id
                ),

                use_cache=True,
            )
        )


    new_tokens = generated[
        0,
        prompt_length:,
    ]


    output = (
        hh_dpo_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        )
        .strip()
    )


    return {
        "output": output,

        "prompt_tokens": (
            prompt_length
        ),

        "generated_tokens": (
            len(new_tokens)
        ),

        "hit_max_new_tokens": (
            len(new_tokens)
            >= HH_GENERATION_CONFIG[
                "max_new_tokens"
            ]
        ),

        "repeat_4gram_fraction": (
            repeated_ngram_fraction(
                output,
                n=4,
            )
        ),
    }


# ============================================================
# FIRST 5 HH EVALUATION PROMPTS
# ============================================================

print(
    "\n===== HH-DPO GENERATION SANITY CHECK ====="
)


for row in hh_eval_raw[:5]:

    result = generate_hh_dpo_one(
        row["instruction"]
    )


    print("\n" + "=" * 100)

    print(
        "ID:",
        row["id"],
    )

    print(
        "CATEGORY:",
        row["category"],
    )

    print(
        "\nEXPECTED BEHAVIOR:"
    )

    print(
        row["expected_behavior"]
    )

    print(
        "\nINSTRUCTION:"
    )

    print(
        row["instruction"]
    )

    print(
        "\nHH-DPO OUTPUT:"
    )

    print(
        result["output"]
    )

    print(
        "\nPrompt tokens:",
        result["prompt_tokens"],
    )

    print(
        "Generated tokens:",
        result["generated_tokens"],
    )

    print(
        "Hit max_new_tokens:",
        result["hit_max_new_tokens"],
    )

    print(
        "Repeated 4-gram fraction:",
        round(
            result[
                "repeat_4gram_fraction"
            ],
            4,
        ),
    )

In [ ]:
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


SFT_HH_DEVICE = torch.device(
    "cuda:1"
)


# ============================================================
# LOAD SFT BEST CHECKPOINT
# ============================================================

sft_hh_tokenizer = (
    AutoTokenizer.from_pretrained(
        SFT_BEST_DIR,
        use_fast=True,
    )
)


sft_hh_model = (
    AutoModelForCausalLM.from_pretrained(
        SFT_BEST_DIR,
        dtype=torch.float32,
    )
    .to(SFT_HH_DEVICE)
)


sft_hh_model.eval()
sft_hh_model.config.use_cache = True


print(
    "===== SFT MODEL FOR HH SANITY ====="
)

print(
    "Device:",
    next(
        sft_hh_model.parameters()
    ).device,
)

print(
    "Dtype:",
    next(
        sft_hh_model.parameters()
    ).dtype,
)


# ============================================================
# SAME GENERATION SETTINGS AS HH-DPO
# ============================================================

def generate_sft_hh_one(
    instruction,
):
    prompt = (
        format_instruction_prompt(
            instruction
        )
    )

    encoded = sft_hh_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded[
        "input_ids"
    ].to(
        SFT_HH_DEVICE
    )

    attention_mask = encoded[
        "attention_mask"
    ].to(
        SFT_HH_DEVICE
    )

    prompt_length = (
        input_ids.shape[1]
    )


    with torch.inference_mode():
        generated = (
            sft_hh_model.generate(
                input_ids=input_ids,

                attention_mask=(
                    attention_mask
                ),

                do_sample=False,

                max_new_tokens=256,

                repetition_penalty=1.0,

                no_repeat_ngram_size=4,

                pad_token_id=(
                    sft_hh_tokenizer.pad_token_id
                ),

                eos_token_id=(
                    sft_hh_tokenizer.eos_token_id
                ),

                use_cache=True,
            )
        )


    new_tokens = generated[
        0,
        prompt_length:,
    ]


    output = (
        sft_hh_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        )
        .strip()
    )


    return {
        "output": output,

        "generated_tokens": (
            len(new_tokens)
        ),

        "hit_max_new_tokens": (
            len(new_tokens) >= 256
        ),

        "repeat_4gram_fraction": (
            repeated_ngram_fraction(
                output,
                n=4,
            )
        ),
    }


# ============================================================
# SIDE-BY-SIDE 5-PROMPT CHECK
# ============================================================

print(
    "\n===== SFT VS HH-DPO SANITY COMPARISON ====="
)


for row in hh_eval_raw[:5]:

    sft_result = (
        generate_sft_hh_one(
            row["instruction"]
        )
    )

    dpo_result = (
        generate_hh_dpo_one(
            row["instruction"]
        )
    )


    print("\n" + "=" * 100)

    print(
        "ID:",
        row["id"],
    )

    print(
        "CATEGORY:",
        row["category"],
    )

    print(
        "\nEXPECTED BEHAVIOR:"
    )

    print(
        row["expected_behavior"]
    )

    print(
        "\nINSTRUCTION:"
    )

    print(
        row["instruction"]
    )


    print(
        "\n--- SFT OUTPUT ---"
    )

    print(
        sft_result["output"]
    )

    print(
        "\nSFT tokens:",
        sft_result[
            "generated_tokens"
        ],
    )

    print(
        "SFT repeated 4-gram:",
        round(
            sft_result[
                "repeat_4gram_fraction"
            ],
            4,
        ),
    )


    print(
        "\n--- HH-DPO OUTPUT ---"
    )

    print(
        dpo_result["output"]
    )

    print(
        "\nDPO tokens:",
        dpo_result[
            "generated_tokens"
        ],
    )

    print(
        "DPO repeated 4-gram:",
        round(
            dpo_result[
                "repeat_4gram_fraction"
            ],
            4,
        ),
    )

In [ ]:
import json
import hashlib
from pathlib import Path
from tqdm.auto import tqdm


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

HH_EVAL_OUTPUT_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/eval/hh_rlhf_seed42"
)

HH_EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SFT_HH_PROGRESS_PATH = (
    HH_EVAL_OUTPUT_DIR
    / "sft_progress.jsonl"
)

DPO_HH_PROGRESS_PATH = (
    HH_EVAL_OUTPUT_DIR
    / "hh_dpo_progress.jsonl"
)


SFT_HH_FINAL_PATH = (
    HH_EVAL_OUTPUT_DIR
    / "hh_eval_300_seed42_sft.json"
)

DPO_HH_FINAL_PATH = (
    HH_EVAL_OUTPUT_DIR
    / "hh_eval_300_seed42_hh_dpo.json"
)


HH_EVAL_GENERATION_CONFIG_PATH = (
    HH_EVAL_OUTPUT_DIR
    / "generation_config.json"
)


# ============================================================
# GENERATION CONFIG
# ============================================================

hh_eval_generation_config = {
    "evaluation_file": str(
        HH_EVAL_PATH
    ),

    "evaluation_examples": 300,

    "sft_checkpoint": str(
        SFT_BEST_DIR
    ),

    "hh_dpo_checkpoint": str(
        HH_BEST_DIR
    ),

    "hh_dpo_checkpoint_sha256": (
        "65d819ab026d41e980163a0f1271906e503c4c7991a0b6c9a8a0f1ce36b8d1a0"
    ),

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "do_sample": False,
    "max_new_tokens": 256,
    "temperature": None,
    "top_p": None,
    "repetition_penalty": 1.0,
    "no_repeat_ngram_size": 4,
}


HH_EVAL_GENERATION_CONFIG_PATH.write_text(
    json.dumps(
        hh_eval_generation_config,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# LOAD ANY EXISTING PROGRESS
# ============================================================

def load_progress(path):
    completed = {}

    if path.exists():
        with open(
            path,
            "r",
            encoding="utf-8",
        ) as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    completed[row["id"]] = row

    return completed


sft_completed = load_progress(
    SFT_HH_PROGRESS_PATH
)

dpo_completed = load_progress(
    DPO_HH_PROGRESS_PATH
)


print(
    "Existing SFT outputs:",
    len(sft_completed),
)

print(
    "Existing HH-DPO outputs:",
    len(dpo_completed),
)


# ============================================================
# GENERATE
# ============================================================

for row in tqdm(
    hh_eval_raw,
    desc="Generating HH SFT + DPO",
):

    # --------------------------------------------------------
    # SFT
    # --------------------------------------------------------

    if row["id"] not in sft_completed:

        result = generate_sft_hh_one(
            row["instruction"]
        )

        prompt_tokens = len(
            sft_hh_tokenizer.encode(
                format_instruction_prompt(
                    row["instruction"]
                ),
                add_special_tokens=False,
            )
        )

        saved = {
            "id": row["id"],
            "instruction": row["instruction"],

            "output": result["output"],

            "generator": (
                "qwen2.5-0.5b-sft-ultrachat"
            ),

            "dataset": row["dataset"],
            "category": row["category"],

            "expected_behavior": (
                row["expected_behavior"]
            ),

            "prompt_token_count": (
                prompt_tokens
            ),

            "generated_token_count": (
                result["generated_tokens"]
            ),

            "hit_max_new_tokens": (
                result[
                    "hit_max_new_tokens"
                ]
            ),
        }

        with open(
            SFT_HH_PROGRESS_PATH,
            "a",
            encoding="utf-8",
        ) as f:
            f.write(
                json.dumps(
                    saved,
                    ensure_ascii=False,
                )
                + "\n"
            )

        sft_completed[
            row["id"]
        ] = saved


    # --------------------------------------------------------
    # HH-DPO
    # --------------------------------------------------------

    if row["id"] not in dpo_completed:

        result = generate_hh_dpo_one(
            row["instruction"]
        )

        prompt_tokens = len(
            hh_dpo_tokenizer.encode(
                format_instruction_prompt(
                    row["instruction"]
                ),
                add_special_tokens=False,
            )
        )

        saved = {
            "id": row["id"],
            "instruction": row["instruction"],

            "output": result["output"],

            "generator": (
                "qwen2.5-0.5b-hh-rlhf-dpo"
            ),

            "dataset": row["dataset"],
            "category": row["category"],

            "expected_behavior": (
                row["expected_behavior"]
            ),

            "prompt_token_count": (
                prompt_tokens
            ),

            "generated_token_count": (
                result["generated_tokens"]
            ),

            "hit_max_new_tokens": (
                result[
                    "hit_max_new_tokens"
                ]
            ),
        }

        with open(
            DPO_HH_PROGRESS_PATH,
            "a",
            encoding="utf-8",
        ) as f:
            f.write(
                json.dumps(
                    saved,
                    ensure_ascii=False,
                )
                + "\n"
            )

        dpo_completed[
            row["id"]
        ] = saved


# ============================================================
# RESTORE EXACT EVALUATION ORDER
# ============================================================

sft_hh_outputs = [
    sft_completed[
        row["id"]
    ]
    for row in hh_eval_raw
]


dpo_hh_outputs = [
    dpo_completed[
        row["id"]
    ]
    for row in hh_eval_raw
]


assert len(
    sft_hh_outputs
) == 300

assert len(
    dpo_hh_outputs
) == 300


assert [
    row["id"]
    for row in sft_hh_outputs
] == [
    row["id"]
    for row in hh_eval_raw
]


assert [
    row["id"]
    for row in dpo_hh_outputs
] == [
    row["id"]
    for row in hh_eval_raw
]


# ============================================================
# SAVE FINAL JSON FILES
# ============================================================

with open(
    SFT_HH_FINAL_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        sft_hh_outputs,
        f,
        indent=2,
        ensure_ascii=False,
    )


with open(
    DPO_HH_FINAL_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        dpo_hh_outputs,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# SHA256
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


sft_hh_sha256 = sha256_file(
    SFT_HH_FINAL_PATH
)

dpo_hh_sha256 = sha256_file(
    DPO_HH_FINAL_PATH
)


# ============================================================
# DIAGNOSTICS
# ============================================================

def summarize_generation(rows):

    repeat_scores = [
        repeated_ngram_fraction(
            row["output"],
            n=4,
        )
        for row in rows
    ]

    capped = sum(
        row["hit_max_new_tokens"]
        for row in rows
    )

    average_tokens = (
        sum(
            row["generated_token_count"]
            for row in rows
        )
        / len(rows)
    )

    return {
        "examples": len(rows),

        "hit_max_new_tokens": capped,

        "hit_max_percentage": (
            100
            * capped
            / len(rows)
        ),

        "average_generated_tokens": (
            average_tokens
        ),

        "mean_repeated_4gram_fraction": (
            sum(repeat_scores)
            / len(repeat_scores)
        ),

        "outputs_gt_20pct_repeated_4gram": (
            sum(
                x > 0.20
                for x in repeat_scores
            )
        ),

        "outputs_gt_40pct_repeated_4gram": (
            sum(
                x > 0.40
                for x in repeat_scores
            )
        ),
    }


sft_hh_summary = (
    summarize_generation(
        sft_hh_outputs
    )
)

dpo_hh_summary = (
    summarize_generation(
        dpo_hh_outputs
    )
)


print(
    "\n===== FULL HH GENERATION COMPLETE ====="
)

print(
    "\nSFT:"
)

print(
    json.dumps(
        sft_hh_summary,
        indent=2,
    )
)


print(
    "\nHH-DPO:"
)

print(
    json.dumps(
        dpo_hh_summary,
        indent=2,
    )
)


print(
    "\nSFT file:"
)

print(
    SFT_HH_FINAL_PATH
)

print(
    "SFT SHA256:",
    sft_hh_sha256,
)


print(
    "\nHH-DPO file:"
)

print(
    DPO_HH_FINAL_PATH
)

print(
    "HH-DPO SHA256:",
    dpo_hh_sha256,
)

In [ ]:
import json
from pathlib import Path


SFT_RUN_DIR = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/sft_qwen2.5_0.5b_ultrachat_seed42"
)

SFT_SUMMARY_PATH = (
    SFT_RUN_DIR / "summary.json"
)


# Exact values from the completed SFT run.
sft_summary_reconstructed = {
    "completed": True,

    "optimizer_updates": 938,
    "expected_optimizer_updates": 938,

    "skipped_optimizer_steps": 0,

    "overall_training_loss": (
        1.3364279199522695
    ),

    "initial_validation": {
        "validation_loss": (
            1.3498976668730491
        ),

        "perplexity": (
            3.857030808477362
        ),

        "examples": 1000,

        "response_tokens": 213375,

        "batches": 500,
    },

    "final_validation": {
        "validation_loss": (
            1.3143231261241624
        ),

        "perplexity": (
            3.7222306516854995
        ),

        "examples": 1000,

        "response_tokens": 213375,

        "batches": 500,
    },

    "best_validation_loss": (
        1.3143231261241624
    ),

    "best_update": 938,

    "elapsed_seconds": (
        4242.975714206696
    ),

    "elapsed_hours": (
        1.1786043650574154
    ),

    "best_checkpoint": str(
        SFT_RUN_DIR
        / "best_checkpoint"
    ),

    "final_checkpoint": str(
        SFT_RUN_DIR
        / "final_checkpoint"
    ),

    "metrics_file": str(
        SFT_RUN_DIR
        / "metrics.jsonl"
    ),
}


SFT_SUMMARY_PATH.write_text(
    json.dumps(
        sft_summary_reconstructed,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "===== SFT SUMMARY RESTORED ====="
)

print(
    "Path:",
    SFT_SUMMARY_PATH,
)

print(
    "Exists:",
    SFT_SUMMARY_PATH.exists(),
)

print(
    "\nContents:"
)

print(
    SFT_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

In [ ]:
from pathlib import Path


REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)


expected_paths = [
    "alignment_core.py",
    "data_utils.py",
    "dpo_training.py",
    "submission_adapter.py",
    "sft_training.py",

    "data/sft_fixed/"
    "ultrachat_train_30000_seed42.jsonl",

    "data/sft_fixed/"
    "ultrachat_val_1000_seed42.jsonl",

    "data/hh_rlhf_fixed/"
    "hh_train_15000_seed42.jsonl",

    "data/hh_rlhf_fixed/"
    "hh_val_1000_seed42.jsonl",

    "eval_sets/"
    "alpaca_eval_300_seed42.json",

    "eval_sets/"
    "hh_rlhf_eval_300_seed42.json",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "config.json",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "metrics.jsonl",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "summary.json",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "best_checkpoint/model.safetensors",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "final_checkpoint/model.safetensors",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "config.json",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "metrics.jsonl",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "summary.json",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "best_checkpoint/model.safetensors",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "final_checkpoint/model.safetensors",

    "outputs/eval/"
    "base_qwen2.5_0.5b/"
    "alpaca_eval_300_seed42_base_nr4.json",

    "outputs/eval/"
    "sft_qwen2.5_0.5b/"
    "alpaca_eval_300_seed42_sft.json",

    "outputs/eval/"
    "hh_rlhf_seed42/"
    "hh_eval_300_seed42_sft.json",

    "outputs/eval/"
    "hh_rlhf_seed42/"
    "hh_eval_300_seed42_hh_dpo.json",
]


missing = []

for relative_path in expected_paths:

    path = (
        REPO_ROOT
        / relative_path
    )

    exists = path.exists()

    print(
        f"{'✓' if exists else '✗'} "
        f"{relative_path}"
    )

    if not exists:
        missing.append(
            relative_path
        )


print(
    "\n===== RESULT ====="
)

print(
    "Expected files:",
    len(expected_paths),
)

print(
    "Missing files:",
    len(missing),
)


if missing:
    print(
        "\nMissing:"
    )

    for path in missing:
        print(
            "-",
            path,
        )

else:
    print(
        "\n✓ All expected PA2 artifacts are present."
    )

In [ ]:
import json
import hashlib
import shutil
from pathlib import Path
from datetime import datetime, timezone


REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

RESUME_STATE_PATH = (
    REPO_ROOT / "PA2_RESUME_STATE.json"
)

RESUME_NOTES_PATH = (
    REPO_ROOT / "PA2_RESUME_INSTRUCTIONS.md"
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path):
    path = Path(path)

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def artifact_info(relative_path):
    path = REPO_ROOT / relative_path

    if not path.exists():
        raise FileNotFoundError(path)

    return {
        "path": relative_path,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


# ============================================================
# CRITICAL ARTIFACT LIST
# ============================================================

critical_paths = [
    # Code
    "alignment_core.py",
    "data_utils.py",
    "dpo_training.py",
    "submission_adapter.py",
    "sft_training.py",

    # Fixed data
    "data/sft_fixed/"
    "ultrachat_train_30000_seed42.jsonl",

    "data/sft_fixed/"
    "ultrachat_val_1000_seed42.jsonl",

    "data/hh_rlhf_fixed/"
    "hh_train_15000_seed42.jsonl",

    "data/hh_rlhf_fixed/"
    "hh_val_1000_seed42.jsonl",

    # Evaluation sets
    "eval_sets/"
    "alpaca_eval_300_seed42.json",

    "eval_sets/"
    "hh_rlhf_eval_300_seed42.json",

    # SFT run
    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "config.json",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "metrics.jsonl",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "summary.json",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "best_checkpoint/model.safetensors",

    "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "final_checkpoint/model.safetensors",

    # HH-DPO run
    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "config.json",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "metrics.jsonl",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "summary.json",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "best_checkpoint/model.safetensors",

    "outputs/"
    "hh_rlhf_dpo_seed42/"
    "final_checkpoint/model.safetensors",

    # Evaluation generations
    "outputs/eval/"
    "base_qwen2.5_0.5b/"
    "alpaca_eval_300_seed42_base_nr4.json",

    "outputs/eval/"
    "sft_qwen2.5_0.5b/"
    "alpaca_eval_300_seed42_sft.json",

    "outputs/eval/"
    "hh_rlhf_seed42/"
    "hh_eval_300_seed42_sft.json",

    "outputs/eval/"
    "hh_rlhf_seed42/"
    "hh_eval_300_seed42_hh_dpo.json",
]


print(
    "Hashing critical PA2 artifacts..."
)


critical_files = {}

for relative_path in critical_paths:

    print(
        "  ",
        relative_path,
    )

    critical_files[
        relative_path
    ] = artifact_info(
        relative_path
    )


# ============================================================
# RESUME STATE
# ============================================================

resume_state = {
    "assignment": (
        "CS 5326 PA2 - Small Model Alignment"
    ),

    "manifest_version": 1,

    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "seed": 42,

    "current_state": (
        "SFT and HH-RLHF from-scratch DPO are complete. "
        "Base/SFT AlpacaEval outputs and SFT/HH-DPO "
        "HH evaluation outputs are complete."
    ),

    "next_stage": (
        "UltraFeedback from-scratch DPO."
    ),

    "critical_resume_rule": (
        "UltraFeedback DPO MUST initialize from the "
        "original SFT best checkpoint, not from the "
        "HH-RLHF DPO checkpoint."
    ),

    "prompt_format": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "evaluation_decoding": {
        "do_sample": False,
        "max_new_tokens": 256,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 4,
    },

    "sft": {
        "train_examples": 30000,
        "validation_examples": 1000,

        "best_update": 938,

        "best_validation_loss": (
            1.3143231261241624
        ),

        "checkpoint": (
            "outputs/"
            "sft_qwen2.5_0.5b_ultrachat_seed42/"
            "best_checkpoint"
        ),

        "model_sha256": (
            "cf40f50587c5287c6dea232603e3976f"
            "e6bf4480fe1b6145f0273da90e8f70ac"
        ),
    },

    "hh_dpo": {
        "train_pairs": 15000,
        "validation_pairs": 1000,

        "best_update": 469,

        "best_reward_accuracy": 0.633,

        "best_reward_margin": (
            0.23464530624076724
        ),

        "checkpoint": (
            "outputs/"
            "hh_rlhf_dpo_seed42/"
            "best_checkpoint"
        ),

        "model_sha256": (
            "65d819ab026d41e980163a0f1271906e"
            "503c4c7991a0b6c9a8a0f1ce36b8d1a0"
        ),

        "amp_init_scale": 512.0,

        "amp_growth_interval": 2000,

        "amp_note": (
            "Default GradScaler scale 65536 produced "
            "non-finite gradients during the deterministic "
            "32-pair preflight. Scale 512 produced finite "
            "gradients and was used for the full run."
        ),
    },

    "known_generation_hashes": {
        "sft_alpaca": (
            "b89c69f7c78487b2e748232598530327"
            "b3ce90b57fb2c9660f015113d28179e7"
        ),

        "sft_hh": (
            "690089f0d0fed87acc32856f0b242e17"
            "6714a68fd6de156b0282fd8fd7aa59e7"
        ),

        "hh_dpo_hh": (
            "e65f52aafa0250a0258e3881203e2566"
            "d0ada22ad468ba2131d58503a9f3295f"
        ),
    },

    "critical_files": critical_files,
}


RESUME_STATE_PATH.write_text(
    json.dumps(
        resume_state,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# HUMAN-READABLE NOTE
# ============================================================

resume_notes = """# PA2 Resume Instructions

## Completed

- Core implementation + public tests
- Fixed AlpacaEval 300-prompt subset
- Base Qwen2.5-0.5B AlpacaEval generation
- SFT on 30,000 UltraChat examples
- SFT AlpacaEval generation
- Fixed HH-RLHF 15,000 train / 1,000 validation pairs
- HH-RLHF from-scratch DPO
- SFT and HH-DPO generation on all 300 HH evaluation prompts

## Next

Run the UltraFeedback from-scratch DPO experiment.

IMPORTANT: initialize both the UltraFeedback trainable policy
and frozen reference from the ORIGINAL SFT best checkpoint:

outputs/sft_qwen2.5_0.5b_ultrachat_seed42/best_checkpoint

Do NOT initialize UltraFeedback from the HH-DPO checkpoint.

## Prompt format

### Instruction:
{instruction}

### Response:

## Evaluation decoding

- greedy
- do_sample=False
- max_new_tokens=256
- repetition_penalty=1.0
- no_repeat_ngram_size=4

## HH-DPO result

- 15,000 train pairs
- 1,000 validation pairs
- 469 updates
- LR = 1e-6 fixed
- beta = 0.1
- accumulation = 32
- best update = 469
- reward accuracy = 0.633
- reward margin = 0.23464530624076724
- AMP initial scale = 512
- skipped optimizer steps = 0

PA2_RESUME_STATE.json contains SHA256 hashes for verification.
"""


RESUME_NOTES_PATH.write_text(
    resume_notes,
    encoding="utf-8",
)


# ============================================================
# VERIFY MANIFEST AGAINST DISK
# ============================================================

saved_state = json.loads(
    RESUME_STATE_PATH.read_text(
        encoding="utf-8"
    )
)


verification_failures = []


for relative_path, info in (
    saved_state[
        "critical_files"
    ].items()
):

    path = REPO_ROOT / relative_path

    if not path.exists():
        verification_failures.append(
            f"MISSING: {relative_path}"
        )
        continue

    actual_sha = sha256_file(
        path
    )

    if actual_sha != info["sha256"]:
        verification_failures.append(
            f"HASH MISMATCH: {relative_path}"
        )


assert not verification_failures, (
    verification_failures
)


# ============================================================
# SIZE AUDIT
# ============================================================

def directory_size(path):
    total = 0

    for item in path.rglob("*"):
        if item.is_file():
            total += item.stat().st_size

    return total


repo_size_bytes = directory_size(
    REPO_ROOT
)

working_disk = shutil.disk_usage(
    "/kaggle/working"
)


# Individual checkpoint directory sizes are useful because
# best/final are duplicates in both completed runs.
checkpoint_dirs = {
    "SFT best": (
        REPO_ROOT
        / "outputs/"
        "sft_qwen2.5_0.5b_ultrachat_seed42/"
        "best_checkpoint"
    ),

    "SFT final": (
        REPO_ROOT
        / "outputs/"
        "sft_qwen2.5_0.5b_ultrachat_seed42/"
        "final_checkpoint"
    ),

    "HH-DPO best": (
        REPO_ROOT
        / "outputs/"
        "hh_rlhf_dpo_seed42/"
        "best_checkpoint"
    ),

    "HH-DPO final": (
        REPO_ROOT
        / "outputs/"
        "hh_rlhf_dpo_seed42/"
        "final_checkpoint"
    ),
}


print(
    "\n===== PA2 RESUME MANIFEST COMPLETE ====="
)

print(
    "Manifest:",
    RESUME_STATE_PATH
)

print(
    "Resume notes:",
    RESUME_NOTES_PATH
)

print(
    "\nCritical files hashed:",
    len(critical_files)
)


print(
    "\n===== SIZE AUDIT ====="
)

print(
    "Whole repository:",
    round(
        repo_size_bytes / 1024**3,
        3,
    ),
    "GB",
)


for name, path in checkpoint_dirs.items():

    size = directory_size(path)

    print(
        f"{name}:",
        round(
            size / 1024**3,
            3,
        ),
        "GB",
    )


print(
    "\nFree /kaggle/working space:",
    round(
        working_disk.free / 1024**3,
        3,
    ),
    "GB",
)


print(
    "\nSFT best/final model weights identical:",
    (
        critical_files[
            "outputs/"
            "sft_qwen2.5_0.5b_ultrachat_seed42/"
            "best_checkpoint/model.safetensors"
        ]["sha256"]
        ==
        critical_files[
            "outputs/"
            "sft_qwen2.5_0.5b_ultrachat_seed42/"
            "final_checkpoint/model.safetensors"
        ]["sha256"]
    ),
)


print(
    "HH-DPO best/final model weights identical:",
    (
        critical_files[
            "outputs/"
            "hh_rlhf_dpo_seed42/"
            "best_checkpoint/model.safetensors"
        ]["sha256"]
        ==
        critical_files[
            "outputs/"
            "hh_rlhf_dpo_seed42/"
            "final_checkpoint/model.safetensors"
        ]["sha256"]
    ),
)


print(
    "\n✓ All critical files verified against "
    "the resume manifest."
)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/small-model-alignment-starter")

print("Resume manifest:", (root / "PA2_RESUME_STATE.json").exists())

print(
    "SFT model:",
    (
        root
        / "outputs/sft_qwen2.5_0.5b_ultrachat_seed42/"
        "best_checkpoint/model.safetensors"
    ).exists()
)

print(
    "HH-DPO model:",
    (
        root
        / "outputs/hh_rlhf_dpo_seed42/"
        "best_checkpoint/model.safetensors"
    ).exists()
)

In [1]:
from pathlib import Path

print("===== LOOKING FOR SAVED PA2 STATE =====")

manifests = list(
    Path("/kaggle/input").rglob(
        "PA2_RESUME_STATE.json"
    )
)

print(
    "Manifests found:",
    len(manifests)
)

for p in manifests:
    print(
        "\nFOUND:",
        p
    )

    print(
        "Saved repo directory:",
        p.parent
    )

===== LOOKING FOR SAVED PA2 STATE =====
Manifests found: 1

FOUND: /kaggle/input/notebooks/muhammadhashirsalman/adv-gen-ai-pa2/small-model-alignment-starter/PA2_RESUME_STATE.json
Saved repo directory: /kaggle/input/notebooks/muhammadhashirsalman/adv-gen-ai-pa2/small-model-alignment-starter


In [2]:
from pathlib import Path
import json
import hashlib
import shutil


# ============================================================
# LOCATE SAVED REPOSITORY
# ============================================================

MANIFEST_SOURCE = Path(
    "/kaggle/input/notebooks/"
    "muhammadhashirsalman/"
    "adv-gen-ai-pa2/"
    "small-model-alignment-starter/"
    "PA2_RESUME_STATE.json"
)

SAVED_REPO = MANIFEST_SOURCE.parent

WORKING_REPO = Path(
    "/kaggle/working/"
    "small-model-alignment-starter"
)


print("===== PA2 RESTORE =====")

print(
    "Saved repository:",
    SAVED_REPO
)

print(
    "Working destination:",
    WORKING_REPO
)


assert SAVED_REPO.exists()
assert MANIFEST_SOURCE.exists()


# ============================================================
# SAFETY CHECK
# ============================================================

if WORKING_REPO.exists():
    raise RuntimeError(
        f"{WORKING_REPO} already exists.\n"
        "Do NOT overwrite it yet. Tell me if you see "
        "this message."
    )


# ============================================================
# COPY SAVED OUTPUT TO WRITABLE WORKING DIRECTORY
# ============================================================

print(
    "\nCopying saved repository into /kaggle/working..."
)

shutil.copytree(
    SAVED_REPO,
    WORKING_REPO,
)


print(
    "✓ Repository copied"
)


# ============================================================
# LOAD RESUME MANIFEST
# ============================================================

RESUME_STATE_PATH = (
    WORKING_REPO
    / "PA2_RESUME_STATE.json"
)

resume_state = json.loads(
    RESUME_STATE_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "\nResume state:"
)

print(
    resume_state[
        "current_state"
    ]
)

print(
    "\nNext stage:"
)

print(
    resume_state[
        "next_stage"
    ]
)


# ============================================================
# HASH VERIFICATION
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


print(
    "\n===== VERIFYING SAVED ARTIFACTS ====="
)

failures = []


for relative_path, info in (
    resume_state[
        "critical_files"
    ].items()
):

    path = (
        WORKING_REPO
        / relative_path
    )

    if not path.exists():
        failures.append(
            f"MISSING: {relative_path}"
        )

        print(
            "✗",
            relative_path,
            "(missing)",
        )

        continue


    actual_sha = sha256_file(
        path
    )


    if actual_sha != info["sha256"]:

        failures.append(
            f"HASH MISMATCH: "
            f"{relative_path}"
        )

        print(
            "✗",
            relative_path,
            "(hash mismatch)",
        )

    else:

        print(
            "✓",
            relative_path,
        )


# ============================================================
# FINAL RESULT
# ============================================================

print(
    "\n===== RESTORE RESULT ====="
)

print(
    "Critical files:",
    len(
        resume_state[
            "critical_files"
        ]
    ),
)

print(
    "Failures:",
    len(failures),
)


if failures:

    print(
        "\nProblems:"
    )

    for failure in failures:
        print(
            "-",
            failure,
        )

    raise RuntimeError(
        "PA2 restore verification failed."
    )


print(
    "\n✓ FULL PA2 RESTORE VERIFIED"
)

print(
    "\nNext task:"
)

print(
    resume_state[
        "next_stage"
    ]
)

===== PA2 RESTORE =====
Saved repository: /kaggle/input/notebooks/muhammadhashirsalman/adv-gen-ai-pa2/small-model-alignment-starter
Working destination: /kaggle/working/small-model-alignment-starter

Copying saved repository into /kaggle/working...
✓ Repository copied

Resume state:
SFT and HH-RLHF from-scratch DPO are complete. Base/SFT AlpacaEval outputs and SFT/HH-DPO HH evaluation outputs are complete.

Next stage:
UltraFeedback from-scratch DPO.

===== VERIFYING SAVED ARTIFACTS =====
✓ alignment_core.py
✓ data_utils.py
✓ dpo_training.py
✓ submission_adapter.py
✓ sft_training.py
✓ data/sft_fixed/ultrachat_train_30000_seed42.jsonl
✓ data/sft_fixed/ultrachat_val_1000_seed42.jsonl
✓ data/hh_rlhf_fixed/hh_train_15000_seed42.jsonl
✓ data/hh_rlhf_fixed/hh_val_1000_seed42.jsonl
✓ eval_sets/alpaca_eval_300_seed42.json
✓ eval_sets/hh_rlhf_eval_300_seed42.json
✓ outputs/sft_qwen2.5_0.5b_ultrachat_seed42/config.json
✓ outputs/sft_qwen2.5_0.5b_ultrachat_seed42/metrics.jsonl
✓ outputs/sft_qwen2

In [3]:
from datasets import load_dataset


UF_DATASET_NAME = (
    "HuggingFaceH4/ultrafeedback_binarized"
)


print("===== LOADING ULTRAFEEDBACK =====")

uf_raw = load_dataset(
    UF_DATASET_NAME
)


print("\n===== ULTRAFEEDBACK DATASET =====")
print(uf_raw)


print("\nSplit names:")
print(list(uf_raw.keys()))


# ============================================================
# SPLIT SCHEMA
# ============================================================

for split_name in uf_raw.keys():

    print(
        f"\n===== {split_name.upper()} ====="
    )

    print(
        "Rows:",
        len(uf_raw[split_name]),
    )

    print(
        "Features:"
    )

    print(
        uf_raw[split_name].features
    )


# ============================================================
# INSPECT FIRST 5 ROWS OF EACH RELEVANT SPLIT
# ============================================================

for split_name in uf_raw.keys():

    print(
        f"\n\n{'#' * 100}"
    )

    print(
        f"FIRST 5 ROWS: {split_name}"
    )

    print(
        f"{'#' * 100}"
    )

    for i in range(
        min(5, len(uf_raw[split_name]))
    ):

        row = uf_raw[
            split_name
        ][i]

        print(
            "\n" + "=" * 100
        )

        print(
            "ROW:",
            i
        )

        print(
            "\nKEYS:"
        )

        print(
            row.keys()
        )


        # Print selected fields cleanly if present.
        for key in [
            "prompt",
            "chosen",
            "rejected",
            "score_chosen",
            "score_rejected",
        ]:

            if key in row:

                print(
                    f"\n{key.upper()}:"
                )

                print(
                    repr(
                        row[key]
                    )
                )

===== LOADING ULTRAFEEDBACK =====


README.md:   0%|          | 0.00/6.53k [00:00<?, ?B/s]

data/train_prefs-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  226MB            

data/train_prefs-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test_prefs-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.29MB            

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]


===== ULTRAFEEDBACK DATASET =====
DatasetDict({
    train_prefs: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 61135
    })
    train_sft: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 61135
    })
    test_prefs: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 2000
    })
    test_sft: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 1000
    })
    train_gen: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
        num_rows: 61135
    })
    test_gen: Dataset({
        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 

In [4]:
from pathlib import Path
import os
import importlib


REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

os.chdir(REPO_ROOT)

DATA_UTILS_PATH = (
    REPO_ROOT / "data_utils.py"
)


uf_parser_code = r'''

# ============================================================
# ULTRAFEEDBACK SINGLE-TURN EXTRACTION
# ============================================================

def extract_ultrafeedback_single_turn(example):
    """
    Extract a strict single-turn UltraFeedback preference pair.

    Expected structure:

        chosen = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": chosen_response},
        ]

        rejected = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": rejected_response},
        ]

    The explicit example["prompt"] must agree with the user
    message on both sides.
    """

    prompt = example.get("prompt")
    chosen = example.get("chosen")
    rejected = example.get("rejected")


    if not isinstance(prompt, str):
        return None

    if not isinstance(chosen, list):
        return None

    if not isinstance(rejected, list):
        return None


    # Strict single-turn structure.
    if len(chosen) != 2:
        return None

    if len(rejected) != 2:
        return None


    # Roles must be user -> assistant.
    if (
        chosen[0].get("role") != "user"
        or chosen[1].get("role") != "assistant"
    ):
        return None

    if (
        rejected[0].get("role") != "user"
        or rejected[1].get("role") != "assistant"
    ):
        return None


    prompt = prompt.strip()

    chosen_prompt = (
        chosen[0].get("content", "").strip()
    )

    rejected_prompt = (
        rejected[0].get("content", "").strip()
    )

    chosen_response = (
        chosen[1].get("content", "").strip()
    )

    rejected_response = (
        rejected[1].get("content", "").strip()
    )


    if not prompt:
        return None

    if not chosen_response:
        return None

    if not rejected_response:
        return None


    # The explicit prompt and both message-list prompts
    # must describe exactly the same instruction.
    if prompt != chosen_prompt:
        return None

    if prompt != rejected_prompt:
        return None


    # Degenerate pair gives no usable preference signal.
    if chosen_response == rejected_response:
        return None


    return {
        "instruction": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
        "prompt_id": example.get("prompt_id"),
        "score_chosen": example.get("score_chosen"),
        "score_rejected": example.get("score_rejected"),
    }
'''


existing_text = DATA_UTILS_PATH.read_text(
    encoding="utf-8"
)


if "def extract_ultrafeedback_single_turn(" not in existing_text:

    with open(
        DATA_UTILS_PATH,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            uf_parser_code
        )

    print(
        "✓ UltraFeedback parser appended to data_utils.py"
    )

else:

    print(
        "✓ UltraFeedback parser already present"
    )


import data_utils

data_utils = importlib.reload(
    data_utils
)

from data_utils import (
    extract_ultrafeedback_single_turn,
    format_instruction_prompt,
)

print(
    "✓ UltraFeedback parser imported successfully"
)

✓ UltraFeedback parser appended to data_utils.py
✓ UltraFeedback parser imported successfully


In [5]:
print(
    "===== ULTRAFEEDBACK PARSER TEST ====="
)

for i in range(5):

    parsed = (
        extract_ultrafeedback_single_turn(
            uf_raw["train_prefs"][i]
        )
    )

    print(
        "\n" + "=" * 80
    )

    print(
        "ROW:",
        i
    )

    if parsed is None:

        print(
            "FILTERED OUT"
        )

    else:

        print(
            "ACCEPTED"
        )

        print(
            "\nINSTRUCTION:"
        )

        print(
            parsed["instruction"][:500]
        )

        print(
            "\nCHOSEN:"
        )

        print(
            parsed["chosen"][:500]
        )

        print(
            "\nREJECTED:"
        )

        print(
            parsed["rejected"][:500]
        )

        print(
            "\nSCORES:",
            parsed["score_chosen"],
            parsed["score_rejected"],
        )

===== ULTRAFEEDBACK PARSER TEST =====

ROW: 0
ACCEPTED

INSTRUCTION:
how can i develop a habit of drawing daily

CHOSEN:
Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:

1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.
2. Set a specific duration: Determine the amount of time you want to spend on drawing each day

REJECTED:
As an AI language model, I cannot personally develop habits for you. But, here are some tips for developing a habit of drawing daily:

1. Start small: Start with simple drawings or doodles and gradually increase the complexity of your drawings.

2. Set a schedule: Set a specific time every day to draw, and try to stick to it as much as possible.

3. Make it fun: Do

In [6]:
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

from transformers import AutoTokenizer

from data_utils import (
    extract_ultrafeedback_single_turn,
    format_instruction_prompt,
)


# ============================================================
# TOKENIZER
# ============================================================

REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

SFT_BEST_DIR = (
    REPO_ROOT
    / "outputs/"
    "sft_qwen2.5_0.5b_ultrachat_seed42/"
    "best_checkpoint"
)


uf_tokenizer = AutoTokenizer.from_pretrained(
    SFT_BEST_DIR,
    use_fast=True,
)


DPO_CONTEXT_LENGTH = 512
DPO_MAX_PROMPT_LENGTH = 256
TOKENIZE_BATCH_SIZE = 512


# ============================================================
# LENGTH SUMMARY
# ============================================================

def describe_lengths(values):

    arr = np.asarray(
        values,
        dtype=np.float64,
    )

    if len(arr) == 0:
        return {}

    return {
        "count": int(len(arr)),

        "mean": round(
            float(arr.mean()),
            2,
        ),

        "median": round(
            float(np.median(arr)),
            2,
        ),

        "p90": round(
            float(
                np.percentile(
                    arr,
                    90,
                )
            ),
            2,
        ),

        "p95": round(
            float(
                np.percentile(
                    arr,
                    95,
                )
            ),
            2,
        ),

        "p99": round(
            float(
                np.percentile(
                    arr,
                    99,
                )
            ),
            2,
        ),

        "max": int(
            arr.max()
        ),
    }


# ============================================================
# ANALYZE ONE SPLIT
# ============================================================

def analyze_uf_split(
    dataset,
    split_name,
):

    parsed_rows = []

    invalid_count = 0

    score_equal = 0
    score_chosen_higher = 0
    score_rejected_higher = 0
    score_missing = 0


    # --------------------------------------------------------
    # STRUCTURE
    # --------------------------------------------------------

    for source_index, example in enumerate(
        tqdm(
            dataset,
            desc=f"Parsing {split_name}",
        )
    ):

        parsed = (
            extract_ultrafeedback_single_turn(
                example
            )
        )

        if parsed is None:

            invalid_count += 1
            continue


        parsed_rows.append({
            "source_index": source_index,
            **parsed,
        })


        sc = parsed[
            "score_chosen"
        ]

        sr = parsed[
            "score_rejected"
        ]


        if sc is None or sr is None:

            score_missing += 1

        elif sc == sr:

            score_equal += 1

        elif sc > sr:

            score_chosen_higher += 1

        else:

            score_rejected_higher += 1


    print(
        f"\n{split_name}: "
        f"{len(parsed_rows)} valid strict "
        f"single-turn rows"
    )


    # --------------------------------------------------------
    # LENGTH STATISTICS
    # --------------------------------------------------------

    prompt_lengths = []

    chosen_response_lengths = []
    rejected_response_lengths = []

    chosen_total_lengths = []
    rejected_total_lengths = []


    counts = {
        "source_rows": (
            len(dataset)
        ),

        "single_turn_valid": (
            len(parsed_rows)
        ),

        "invalid_or_non_single_turn": (
            invalid_count
        ),

        "score_equal": (
            score_equal
        ),

        "score_chosen_higher": (
            score_chosen_higher
        ),

        "score_rejected_higher": (
            score_rejected_higher
        ),

        "score_missing": (
            score_missing
        ),

        "prompt_over_256": 0,

        "chosen_full_fit": 0,

        "rejected_full_fit": 0,

        "both_full_fit": 0,

        "usable_full_fit_and_prompt_limit": 0,
    }


    usable_candidates = []


    eos_extra = (
        1
        if uf_tokenizer.eos_token_id
        is not None
        else 0
    )


    # --------------------------------------------------------
    # BATCH TOKENIZATION
    # --------------------------------------------------------

    for start in tqdm(
        range(
            0,
            len(parsed_rows),
            TOKENIZE_BATCH_SIZE,
        ),
        desc=f"Tokenizing {split_name}",
    ):

        chunk = parsed_rows[
            start:
            start + TOKENIZE_BATCH_SIZE
        ]


        prompts = [
            format_instruction_prompt(
                row["instruction"]
            )
            for row in chunk
        ]


        chosen_texts = [
            row["chosen"]
            for row in chunk
        ]


        rejected_texts = [
            row["rejected"]
            for row in chunk
        ]


        prompt_encoded = (
            uf_tokenizer(
                prompts,

                add_special_tokens=False,

                padding=False,

                truncation=False,
            )
        )


        chosen_encoded = (
            uf_tokenizer(
                chosen_texts,

                add_special_tokens=False,

                padding=False,

                truncation=False,
            )
        )


        rejected_encoded = (
            uf_tokenizer(
                rejected_texts,

                add_special_tokens=False,

                padding=False,

                truncation=False,
            )
        )


        for row, p_ids, c_ids, r_ids in zip(
            chunk,

            prompt_encoded[
                "input_ids"
            ],

            chosen_encoded[
                "input_ids"
            ],

            rejected_encoded[
                "input_ids"
            ],
        ):

            prompt_len = len(
                p_ids
            )


            chosen_response_len = (
                len(c_ids)
                + eos_extra
            )


            rejected_response_len = (
                len(r_ids)
                + eos_extra
            )


            chosen_total = (
                prompt_len
                + chosen_response_len
            )


            rejected_total = (
                prompt_len
                + rejected_response_len
            )


            prompt_lengths.append(
                prompt_len
            )

            chosen_response_lengths.append(
                chosen_response_len
            )

            rejected_response_lengths.append(
                rejected_response_len
            )

            chosen_total_lengths.append(
                chosen_total
            )

            rejected_total_lengths.append(
                rejected_total
            )


            # --------------------------------------------
            # LIMITS
            # --------------------------------------------

            prompt_ok = (
                prompt_len
                <= DPO_MAX_PROMPT_LENGTH
            )


            if not prompt_ok:

                counts[
                    "prompt_over_256"
                ] += 1


            chosen_fit = (
                chosen_total
                <= DPO_CONTEXT_LENGTH
            )


            rejected_fit = (
                rejected_total
                <= DPO_CONTEXT_LENGTH
            )


            if chosen_fit:

                counts[
                    "chosen_full_fit"
                ] += 1


            if rejected_fit:

                counts[
                    "rejected_full_fit"
                ] += 1


            if (
                chosen_fit
                and rejected_fit
            ):

                counts[
                    "both_full_fit"
                ] += 1


            # --------------------------------------------
            # FULLY USABLE CANDIDATE
            # --------------------------------------------

            if (
                prompt_ok
                and chosen_fit
                and rejected_fit
            ):

                counts[
                    "usable_full_fit_and_prompt_limit"
                ] += 1


                usable_candidates.append({
                    "source_index": (
                        row[
                            "source_index"
                        ]
                    ),

                    "prompt_id": (
                        row[
                            "prompt_id"
                        ]
                    ),

                    "instruction": (
                        row[
                            "instruction"
                        ]
                    ),

                    "chosen": (
                        row[
                            "chosen"
                        ]
                    ),

                    "rejected": (
                        row[
                            "rejected"
                        ]
                    ),

                    "score_chosen": (
                        row[
                            "score_chosen"
                        ]
                    ),

                    "score_rejected": (
                        row[
                            "score_rejected"
                        ]
                    ),

                    "prompt_tokens": (
                        prompt_len
                    ),

                    "chosen_response_tokens": (
                        chosen_response_len
                    ),

                    "rejected_response_tokens": (
                        rejected_response_len
                    ),

                    "chosen_total_tokens": (
                        chosen_total
                    ),

                    "rejected_total_tokens": (
                        rejected_total
                    ),
                })


    analysis = {
        "counts": counts,

        "prompt_lengths": (
            describe_lengths(
                prompt_lengths
            )
        ),

        "chosen_response_lengths": (
            describe_lengths(
                chosen_response_lengths
            )
        ),

        "rejected_response_lengths": (
            describe_lengths(
                rejected_response_lengths
            )
        ),

        "chosen_total_lengths": (
            describe_lengths(
                chosen_total_lengths
            )
        ),

        "rejected_total_lengths": (
            describe_lengths(
                rejected_total_lengths
            )
        ),
    }


    return (
        analysis,
        usable_candidates,
    )


# ============================================================
# TRAIN PREFS
# ============================================================

print(
    "===== ANALYZING ULTRAFEEDBACK TRAIN_PREFS ====="
)


(
    uf_train_analysis,
    uf_train_candidates,
) = analyze_uf_split(
    uf_raw["train_prefs"],
    "train_prefs",
)


print(
    "\n===== UF TRAIN ANALYSIS ====="
)

print(
    json.dumps(
        uf_train_analysis,
        indent=2,
    )
)


# ============================================================
# TEST PREFS
# ============================================================

print(
    "\n===== ANALYZING ULTRAFEEDBACK TEST_PREFS ====="
)


(
    uf_test_analysis,
    uf_test_candidates,
) = analyze_uf_split(
    uf_raw["test_prefs"],
    "test_prefs",
)


print(
    "\n===== UF TEST ANALYSIS ====="
)

print(
    json.dumps(
        uf_test_analysis,
        indent=2,
    )
)


# ============================================================
# FEASIBILITY
# ============================================================

print(
    "\n===== ULTRAFEEDBACK DPO FEASIBILITY ====="
)

print(
    "Usable train candidates:",
    len(
        uf_train_candidates
    ),
)

print(
    "Need:",
    15000,
)


print(
    "\nUsable validation candidates:",
    len(
        uf_test_candidates
    ),
)

print(
    "Need:",
    1000,
)


print(
    "\nEnough train:",
    len(
        uf_train_candidates
    ) >= 15000,
)

print(
    "Enough validation:",
    len(
        uf_test_candidates
    ) >= 1000,
)

===== ANALYZING ULTRAFEEDBACK TRAIN_PREFS =====


Parsing train_prefs:   0%|          | 0/61135 [00:00<?, ?it/s]


train_prefs: 60630 valid strict single-turn rows


Tokenizing train_prefs:   0%|          | 0/119 [00:00<?, ?it/s]


===== UF TRAIN ANALYSIS =====
{
  "counts": {
    "source_rows": 61135,
    "single_turn_valid": 60630,
    "invalid_or_non_single_turn": 505,
    "score_equal": 7286,
    "score_chosen_higher": 53344,
    "score_rejected_higher": 0,
    "score_missing": 0,
    "prompt_over_256": 11221,
    "chosen_full_fit": 40157,
    "rejected_full_fit": 43536,
    "both_full_fit": 36135,
    "usable_full_fit_and_prompt_limit": 33377
  },
  "prompt_lengths": {
    "count": 60630,
    "mean": 163.43,
    "median": 82.5,
    "p90": 420.0,
    "p95": 623.0,
    "p99": 1128.0,
    "max": 5636
  },
  "chosen_response_lengths": {
    "count": 60630,
    "mean": 278.17,
    "median": 226.0,
    "p90": 620.0,
    "p95": 740.0,
    "p99": 907.0,
    "max": 1483
  },
  "rejected_response_lengths": {
    "count": 60630,
    "mean": 243.56,
    "median": 181.0,
    "p90": 561.0,
    "p95": 701.55,
    "p99": 862.0,
    "max": 2073
  },
  "chosen_total_lengths": {
    "count": 60630,
    "mean": 441.6,
    "med

Parsing test_prefs:   0%|          | 0/2000 [00:00<?, ?it/s]


test_prefs: 1985 valid strict single-turn rows


Tokenizing test_prefs:   0%|          | 0/4 [00:00<?, ?it/s]


===== UF TEST ANALYSIS =====
{
  "counts": {
    "source_rows": 2000,
    "single_turn_valid": 1985,
    "invalid_or_non_single_turn": 15,
    "score_equal": 270,
    "score_chosen_higher": 1715,
    "score_rejected_higher": 0,
    "score_missing": 0,
    "prompt_over_256": 329,
    "chosen_full_fit": 1296,
    "rejected_full_fit": 1435,
    "both_full_fit": 1179,
    "usable_full_fit_and_prompt_limit": 1116
  },
  "prompt_lengths": {
    "count": 1985,
    "mean": 154.17,
    "median": 78.0,
    "p90": 389.0,
    "p95": 600.0,
    "p99": 1075.64,
    "max": 2066
  },
  "chosen_response_lengths": {
    "count": 1985,
    "mean": 283.29,
    "median": 232.0,
    "p90": 624.2,
    "p95": 743.0,
    "p99": 878.32,
    "max": 1218
  },
  "rejected_response_lengths": {
    "count": 1985,
    "mean": 242.95,
    "median": 185.0,
    "p90": 557.2,
    "p95": 682.6,
    "p99": 845.76,
    "max": 1105
  },
  "chosen_total_lengths": {
    "count": 1985,
    "mean": 437.45,
    "median": 391.0,


In [7]:
import json
import random
import hashlib
from pathlib import Path


# ============================================================
# CONFIG
# ============================================================

SEED = 42
N_TRAIN = 15000
N_VAL = 1000

REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

OUT_DIR = (
    REPO_ROOT
    / "data"
    / "ultrafeedback_fixed"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TRAIN_PATH = (
    OUT_DIR
    / "ultrafeedback_train_15000_seed42.jsonl"
)

VAL_PATH = (
    OUT_DIR
    / "ultrafeedback_val_1000_seed42.jsonl"
)

META_PATH = (
    OUT_DIR
    / "selection_metadata.json"
)


# ============================================================
# CHECK TRAIN / TEST OVERLAP
# ============================================================

train_prompt_ids = {
    row["prompt_id"]
    for row in uf_train_candidates
    if row["prompt_id"] is not None
}

test_prompt_ids = {
    row["prompt_id"]
    for row in uf_test_candidates
    if row["prompt_id"] is not None
}


prompt_id_overlap = (
    train_prompt_ids
    & test_prompt_ids
)


train_instructions = {
    row["instruction"]
    for row in uf_train_candidates
}

test_instructions = {
    row["instruction"]
    for row in uf_test_candidates
}


instruction_overlap = (
    train_instructions
    & test_instructions
)


print("===== OVERLAP CHECK =====")

print(
    "Train usable candidates:",
    len(uf_train_candidates),
)

print(
    "Test usable candidates:",
    len(uf_test_candidates),
)

print(
    "Prompt-ID overlap:",
    len(prompt_id_overlap),
)

print(
    "Exact instruction overlap:",
    len(instruction_overlap),
)


# If official train/test unexpectedly overlap,
# stop before freezing anything.
if (
    len(prompt_id_overlap) > 0
    or len(instruction_overlap) > 0
):

    raise RuntimeError(
        "Unexpected UltraFeedback train/test overlap found. "
        "Stop here and send me the output."
    )


# ============================================================
# DETERMINISTIC SELECTION
# ============================================================

train_rng = random.Random(
    SEED
)

val_rng = random.Random(
    SEED
)


train_selected = train_rng.sample(
    uf_train_candidates,
    N_TRAIN,
)

val_selected = val_rng.sample(
    uf_test_candidates,
    N_VAL,
)


assert len(train_selected) == N_TRAIN
assert len(val_selected) == N_VAL


# ============================================================
# WRITE JSONL
# ============================================================

def write_jsonl(
    path,
    rows,
):

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:

        for row in rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


write_jsonl(
    TRAIN_PATH,
    train_selected,
)

write_jsonl(
    VAL_PATH,
    val_selected,
)


# ============================================================
# SHA256
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


train_sha = sha256_file(
    TRAIN_PATH
)

val_sha = sha256_file(
    VAL_PATH
)


# ============================================================
# METADATA
# ============================================================

metadata = {
    "dataset": (
        "HuggingFaceH4/"
        "ultrafeedback_binarized"
    ),

    "train_source_split": (
        "train_prefs"
    ),

    "validation_source_split": (
        "test_prefs"
    ),

    "seed": SEED,

    "train_target_size": (
        N_TRAIN
    ),

    "validation_target_size": (
        N_VAL
    ),

    "context_length": 512,

    "max_prompt_length": 256,

    "filtering": {
        "strict_single_turn": True,
        "prompt_must_match_both_sides": True,
        "chosen_and_rejected_must_differ": True,
        "full_chosen_sequence_must_fit": True,
        "full_rejected_sequence_must_fit": True,
        "no_response_truncation": True,
        "score_ties_retained": True,
        "preference_order_source": (
            "dataset chosen/rejected fields"
        ),
    },

    "usable_candidates": {
        "train": (
            len(
                uf_train_candidates
            )
        ),

        "validation": (
            len(
                uf_test_candidates
            )
        ),
    },

    "overlap_checks": {
        "prompt_id_overlap": (
            len(
                prompt_id_overlap
            )
        ),

        "exact_instruction_overlap": (
            len(
                instruction_overlap
            )
        ),
    },

    "train_first_10_source_indices": [
        row["source_index"]
        for row in train_selected[:10]
    ],

    "validation_first_10_source_indices": [
        row["source_index"]
        for row in val_selected[:10]
    ],

    "files": {
        TRAIN_PATH.name: {
            "sha256": train_sha,
        },

        VAL_PATH.name: {
            "sha256": val_sha,
        },
    },
}


META_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# RELOAD VERIFICATION
# ============================================================

def read_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            rows.append(
                json.loads(
                    line
                )
            )

    return rows


train_reload = read_jsonl(
    TRAIN_PATH
)

val_reload = read_jsonl(
    VAL_PATH
)


assert len(train_reload) == N_TRAIN
assert len(val_reload) == N_VAL

assert (
    sha256_file(TRAIN_PATH)
    == train_sha
)

assert (
    sha256_file(VAL_PATH)
    == val_sha
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n===== ULTRAFEEDBACK FIXED DATA COMPLETE ====="
)

print(
    "Train rows:",
    len(train_reload),
)

print(
    "Validation rows:",
    len(val_reload),
)

print(
    "\nTrain first 10 source indices:"
)

print(
    metadata[
        "train_first_10_source_indices"
    ]
)

print(
    "\nValidation first 10 source indices:"
)

print(
    metadata[
        "validation_first_10_source_indices"
    ]
)

print(
    "\nTrain SHA256:"
)

print(
    train_sha
)

print(
    "\nValidation SHA256:"
)

print(
    val_sha
)

print(
    "\nSaved:"
)

print(
    TRAIN_PATH
)

print(
    VAL_PATH
)

print(
    META_PATH
)

print(
    "\n✓ UltraFeedback fixed subsets verified"
)

===== OVERLAP CHECK =====
Train usable candidates: 33377
Test usable candidates: 1116
Prompt-ID overlap: 0
Exact instruction overlap: 0

===== ULTRAFEEDBACK FIXED DATA COMPLETE =====
Train rows: 15000
Validation rows: 1000

Train first 10 source indices:
[13258, 2968, 32923, 29312, 26579, 16580, 12237, 10314, 50540, 3800]

Validation first 10 source indices:
[437, 106, 1054, 921, 837, 533, 397, 344, 1584, 126]

Train SHA256:
eb726a47be09821959086c24aa0f6588ab1a578e4c4a22fab367358fd67e8397

Validation SHA256:
a6e4c6db82092363252e32c5adec8c897015fd3a5fbe9cd549cfa832820aafea

Saved:
/kaggle/working/small-model-alignment-starter/data/ultrafeedback_fixed/ultrafeedback_train_15000_seed42.jsonl
/kaggle/working/small-model-alignment-starter/data/ultrafeedback_fixed/ultrafeedback_val_1000_seed42.jsonl
/kaggle/working/small-model-alignment-starter/data/ultrafeedback_fixed/selection_metadata.json

✓ UltraFeedback fixed subsets verified


In [8]:
import os
import json
import inspect
import importlib
from pathlib import Path

REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

os.chdir(REPO_ROOT)


# ============================================================
# LOAD EXISTING DPO MODULE
# ============================================================

import dpo_training

dpo_training = importlib.reload(
    dpo_training
)


print("===== DPO TRAINING FUNCTIONS =====")

for name in [
    "prepare_dpo_pair_cpu",
    "response_logps_from_joint_batch",
    "compute_pair_quantities",
    "evaluate_dpo",
    "save_dpo_checkpoint",
    "train_dpo",
]:

    obj = getattr(
        dpo_training,
        name,
        None,
    )

    print(
        "\n",
        name,
        ":",
        sep="",
    )

    if obj is None:

        print(
            "NOT FOUND"
        )

    else:

        print(
            inspect.signature(
                obj
            )
        )


# ============================================================
# READ PREVIOUS HH CONFIG
# ============================================================

HH_CONFIG_PATH = (
    REPO_ROOT
    / "outputs"
    / "hh_rlhf_dpo_seed42"
    / "config.json"
)


print(
    "\n===== PREVIOUS HH-RLHF DPO CONFIG ====="
)

hh_config = json.loads(
    HH_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        hh_config,
        indent=2,
    )
)

===== DPO TRAINING FUNCTIONS =====

prepare_dpo_pair_cpu:
(tokenizer, prompt, chosen, rejected, max_length)

response_logps_from_joint_batch:
(model, batch)

compute_pair_quantities:
(*, policy_model, reference_model, tokenizer, prompt, chosen, rejected, max_length, beta, policy_device, reference_device, require_policy_grad)

evaluate_dpo:
(*, policy_model, reference_model, tokenizer, dataloader, max_length, beta, policy_device, reference_device, max_batches=None)

save_dpo_checkpoint:
(*, model, tokenizer, output_dir, metadata)

train_dpo:
(*, policy_model, reference_model, tokenizer, train_loader, val_loader, config, output_dir, policy_device, reference_device)

===== PREVIOUS HH-RLHF DPO CONFIG =====
{
  "run_name": "hh_rlhf_dpo_seed42",
  "initial_policy": "/kaggle/working/small-model-alignment-starter/outputs/sft_qwen2.5_0.5b_ultrachat_seed42/best_checkpoint",
  "reference_model": "/kaggle/working/small-model-alignment-starter/outputs/sft_qwen2.5_0.5b_ultrachat_seed42/best_checkpo

In [9]:
import json
import math
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

SFT_BEST_DIR = (
    REPO_ROOT
    / "outputs"
    / "sft_qwen2.5_0.5b_ultrachat_seed42"
    / "best_checkpoint"
)

UF_TRAIN_PATH = (
    REPO_ROOT
    / "data"
    / "ultrafeedback_fixed"
    / "ultrafeedback_train_15000_seed42.jsonl"
)

UF_VAL_PATH = (
    REPO_ROOT
    / "data"
    / "ultrafeedback_fixed"
    / "ultrafeedback_val_1000_seed42.jsonl"
)

UF_OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "ultrafeedback_dpo_seed42"
)

UF_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# FIXED DATA HASHES
# ============================================================

UF_TRAIN_SHA = (
    "eb726a47be09821959086c24aa0f6588"
    "ab1a578e4c4a22fab367358fd67e8397"
)

UF_VAL_SHA = (
    "a6e4c6db82092363252e32c5adec8c8"
    "97015fd3a5fbe9cd549cfa832820aafea"
)


# ============================================================
# CONFIG
# ============================================================

TRAIN_EXAMPLES = 15000
VAL_EXAMPLES = 1000

GRAD_ACCUM = 32

TOTAL_UPDATES = math.ceil(
    TRAIN_EXAMPLES / GRAD_ACCUM
)


uf_dpo_config = {
    "run_name": (
        "ultrafeedback_dpo_seed42"
    ),

    "initial_policy": str(
        SFT_BEST_DIR
    ),

    "reference_model": str(
        SFT_BEST_DIR
    ),

    "train_examples": (
        TRAIN_EXAMPLES
    ),

    "validation_examples": (
        VAL_EXAMPLES
    ),

    "context_length": 512,

    "max_prompt_length": 256,

    "per_device_batch_size": 1,

    "gradient_accumulation_steps": (
        GRAD_ACCUM
    ),

    "effective_batch_size": 32,

    "learning_rate": 1e-6,

    "lr_schedule": "fixed",

    "optimizer": "AdamW",

    "adam_beta1": 0.9,

    "adam_beta2": 0.999,

    "adam_epsilon": 1e-8,

    "weight_decay": 0.0,

    "beta": 0.1,

    "gradient_clip_norm": 1.0,

    "epochs": 1,

    "precision_policy": (
        "fp16_autocast_with_fp32_weights"
    ),

    "precision_reference": (
        "fp16_autocast_with_fp32_frozen_weights"
    ),

    "seed": 42,

    "eval_every_updates": 100,

    "checkpoint_selection": (
        "highest validation reward accuracy"
    ),

    "prompt_template": (
        "### Instruction:\n"
        "{instruction}\n\n"
        "### Response:\n"
    ),

    "train_data_path": str(
        UF_TRAIN_PATH
    ),

    "validation_data_path": str(
        UF_VAL_PATH
    ),

    "train_data_sha256": (
        UF_TRAIN_SHA
    ),

    "validation_data_sha256": (
        UF_VAL_SHA
    ),

    "micro_batches_per_epoch": (
        TRAIN_EXAMPLES
    ),

    "total_update_steps": (
        TOTAL_UPDATES
    ),

    # Start with the numerically stable AMP setup
    # validated during the HH-RLHF run.
    # We will still preflight it separately on UltraFeedback.
    "amp_init_scale": 512.0,

    "amp_growth_interval": 2000,

    "amp_scale_rationale": (
        "Using init_scale=512 because the previous "
        "HH-RLHF DPO deterministic 32-pair preflight "
        "showed that the default GradScaler scale 65536 "
        "produced non-finite gradients, while 512 produced "
        "finite gradients. UltraFeedback will receive its "
        "own independent numerical preflight before training."
    ),
}


# ============================================================
# SANITY ASSERTIONS
# ============================================================

assert (
    Path(
        uf_dpo_config[
            "initial_policy"
        ]
    )
    == SFT_BEST_DIR
)

assert (
    Path(
        uf_dpo_config[
            "reference_model"
        ]
    )
    == SFT_BEST_DIR
)

assert (
    "hh_rlhf_dpo_seed42"
    not in uf_dpo_config[
        "initial_policy"
    ]
)

assert (
    uf_dpo_config[
        "total_update_steps"
    ]
    == 469
)


# ============================================================
# SAVE
# ============================================================

UF_CONFIG_PATH = (
    UF_OUTPUT_DIR
    / "config.json"
)

UF_CONFIG_PATH.write_text(
    json.dumps(
        uf_dpo_config,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "===== ULTRAFEEDBACK DPO CONFIG ====="
)

print(
    json.dumps(
        uf_dpo_config,
        indent=2,
    )
)

print(
    "\nSaved config:"
)

print(
    UF_CONFIG_PATH
)

print(
    "\n✓ Policy starts from SFT best"
)

print(
    "✓ Reference starts from SFT best"
)

print(
    "✓ HH-DPO checkpoint is NOT used"
)

print(
    "✓ Total optimizer updates:",
    TOTAL_UPDATES,
)

===== ULTRAFEEDBACK DPO CONFIG =====
{
  "run_name": "ultrafeedback_dpo_seed42",
  "initial_policy": "/kaggle/working/small-model-alignment-starter/outputs/sft_qwen2.5_0.5b_ultrachat_seed42/best_checkpoint",
  "reference_model": "/kaggle/working/small-model-alignment-starter/outputs/sft_qwen2.5_0.5b_ultrachat_seed42/best_checkpoint",
  "train_examples": 15000,
  "validation_examples": 1000,
  "context_length": 512,
  "max_prompt_length": 256,
  "per_device_batch_size": 1,
  "gradient_accumulation_steps": 32,
  "effective_batch_size": 32,
  "learning_rate": 1e-06,
  "lr_schedule": "fixed",
  "optimizer": "AdamW",
  "adam_beta1": 0.9,
  "adam_beta2": 0.999,
  "adam_epsilon": 1e-08,
  "weight_decay": 0.0,
  "beta": 0.1,
  "gradient_clip_norm": 1.0,
  "epochs": 1,
  "precision_policy": "fp16_autocast_with_fp32_weights",
  "precision_reference": "fp16_autocast_with_fp32_frozen_weights",
  "seed": 42,
  "eval_every_updates": 100,
  "checkpoint_selection": "highest validation reward accuracy"

In [10]:
import inspect
import dpo_training


print(
    "===== train_dpo SOURCE ====="
)

print(
    inspect.getsource(
        dpo_training.train_dpo
    )
)


print(
    "\n===== evaluate_dpo SOURCE ====="
)

print(
    inspect.getsource(
        dpo_training.evaluate_dpo
    )
)

===== train_dpo SOURCE =====
def train_dpo(
    *,
    policy_model,
    reference_model,
    tokenizer,

    train_loader,
    val_loader,

    config,
    output_dir,

    policy_device,
    reference_device,
):
    """
    Full manual DPO training loop.

    Pair batch size = 1.
    Gradient accumulation gives effective pair batch size 32.

    The final partial accumulation window is normalized by its
    actual number of pairs rather than always dividing by 32.
    """

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    metrics_path = (
        output_dir
        / "metrics.jsonl"
    )

    summary_path = (
        output_dir
        / "summary.json"
    )

    best_dir = (
        output_dir
        / "best_checkpoint"
    )

    final_dir = (
        output_dir
        / "final_checkpoint"
    )


    # --------------------------------------------------------
    # Refuse to accidentally append a new r

In [11]:
import json
import inspect
from pathlib import Path

import dpo_training


REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)


# ============================================================
# INSPECT THE EXACT PROMPT HANDLING
# ============================================================

print("===== prepare_dpo_pair_cpu SOURCE =====")

print(
    inspect.getsource(
        dpo_training.prepare_dpo_pair_cpu
    )
)


print(
    "\n===== compute_pair_quantities SOURCE ====="
)

print(
    inspect.getsource(
        dpo_training.compute_pair_quantities
    )
)


# ============================================================
# SHOW ONE PREVIOUS HH TRAIN ROW
# ============================================================

HH_TRAIN_PATH = (
    REPO_ROOT
    / "data"
    / "hh_rlhf_fixed"
    / "hh_train_15000_seed42.jsonl"
)

with open(
    HH_TRAIN_PATH,
    "r",
    encoding="utf-8",
) as f:
    hh_first = json.loads(
        f.readline()
    )


print(
    "\n===== FIRST SAVED HH ROW ====="
)

print(
    "Keys:",
    hh_first.keys()
)

for key in [
    "instruction",
    "prompt",
    "chosen",
    "rejected",
]:
    if key in hh_first:
        print(
            f"\n{key.upper()}:"
        )
        print(
            repr(
                hh_first[key][:500]
                if isinstance(
                    hh_first[key],
                    str,
                )
                else hh_first[key]
            )
        )


# ============================================================
# SHOW ONE NEW ULTRAFEEDBACK ROW
# ============================================================

UF_TRAIN_PATH = (
    REPO_ROOT
    / "data"
    / "ultrafeedback_fixed"
    / "ultrafeedback_train_15000_seed42.jsonl"
)

with open(
    UF_TRAIN_PATH,
    "r",
    encoding="utf-8",
) as f:
    uf_first = json.loads(
        f.readline()
    )


print(
    "\n===== FIRST SAVED ULTRAFEEDBACK ROW ====="
)

print(
    "Keys:",
    uf_first.keys()
)

for key in [
    "instruction",
    "prompt",
    "chosen",
    "rejected",
]:
    if key in uf_first:
        print(
            f"\n{key.upper()}:"
        )
        print(
            repr(
                uf_first[key][:500]
                if isinstance(
                    uf_first[key],
                    str,
                )
                else uf_first[key]
            )
        )

===== prepare_dpo_pair_cpu SOURCE =====
def prepare_dpo_pair_cpu(
    tokenizer,
    prompt,
    chosen,
    rejected,
    max_length,
):
    """
    Prepare one preference pair using the already-tested
    assignment batch implementation.

    The returned joint tensors have two rows:

        row 0 = chosen
        row 1 = rejected

    Trailing columns that are padding for BOTH sequences are
    removed for efficiency.
    """

    dpo_batch = build_dpo_batch_impl(
        tokenizer=tokenizer,
        prompts=[prompt],
        chosen_responses=[chosen],
        rejected_responses=[rejected],
        max_length=max_length,
    )

    chosen_batch = dpo_batch["chosen"]
    rejected_batch = dpo_batch["rejected"]


    chosen_used_length = int(
        chosen_batch[
            "attention_mask"
        ]
        .sum(dim=1)
        .max()
        .item()
    )

    rejected_used_length = int(
        rejected_batch[
            "attention_mask"
        ]
        .sum(dim=1)
        .max

In [12]:
import gc
import json
import math
import os
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

from data_utils import (
    format_instruction_prompt,
)

from dpo_training import (
    compute_pair_quantities,
)


# ============================================================
# PATHS / DEVICES
# ============================================================

REPO_ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

SFT_BEST_DIR = (
    REPO_ROOT
    / "outputs"
    / "sft_qwen2.5_0.5b_ultrachat_seed42"
    / "best_checkpoint"
)

UF_TRAIN_PATH = (
    REPO_ROOT
    / "data"
    / "ultrafeedback_fixed"
    / "ultrafeedback_train_15000_seed42.jsonl"
)


assert torch.cuda.device_count() >= 2

POLICY_DEVICE = torch.device(
    "cuda:0"
)

REFERENCE_DEVICE = torch.device(
    "cuda:1"
)


print("===== DEVICES =====")

print(
    "Policy:",
    POLICY_DEVICE,
    torch.cuda.get_device_name(0),
)

print(
    "Reference:",
    REFERENCE_DEVICE,
    torch.cuda.get_device_name(1),
)


# ============================================================
# LOAD FIRST 32 FIXED ULTRAFEEDBACK PAIRS
# ============================================================

uf_preflight_rows = []

with open(
    UF_TRAIN_PATH,
    "r",
    encoding="utf-8",
) as f:

    for _ in range(32):

        uf_preflight_rows.append(
            json.loads(
                next(f)
            )
        )


assert len(
    uf_preflight_rows
) == 32


print(
    "\nPreflight pairs:",
    len(uf_preflight_rows),
)


# ============================================================
# TOKENIZER
# ============================================================

uf_dpo_tokenizer = (
    AutoTokenizer.from_pretrained(
        SFT_BEST_DIR,
        use_fast=True,
    )
)


# ============================================================
# VERIFY PROMPT FORMATTING AGAINST SAVED LENGTH METADATA
# ============================================================

first_row = (
    uf_preflight_rows[0]
)

formatted_first_prompt = (
    format_instruction_prompt(
        first_row["instruction"]
    )
)

formatted_prompt_len = len(
    uf_dpo_tokenizer(
        formatted_first_prompt,
        add_special_tokens=False,
    )["input_ids"]
)


print(
    "\n===== PROMPT FORMAT CHECK ====="
)

print(
    "Stored prompt_tokens:",
    first_row[
        "prompt_tokens"
    ],
)

print(
    "Recomputed formatted prompt tokens:",
    formatted_prompt_len,
)

assert (
    formatted_prompt_len
    == first_row[
        "prompt_tokens"
    ]
)

print(
    "✓ Prompt formatting matches frozen data metadata"
)


# ============================================================
# CLEAN GPU MEMORY
# ============================================================

gc.collect()

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats(
    POLICY_DEVICE
)

torch.cuda.reset_peak_memory_stats(
    REFERENCE_DEVICE
)


# ============================================================
# LOAD POLICY FROM ORIGINAL SFT BEST
# ============================================================

print(
    "\n===== LOADING POLICY ====="
)

uf_policy_model = (
    AutoModelForCausalLM.from_pretrained(
        SFT_BEST_DIR,
        torch_dtype=torch.float32,
    )
)

uf_policy_model.to(
    POLICY_DEVICE
)

uf_policy_model.train()


# ============================================================
# LOAD FROZEN REFERENCE FROM SAME ORIGINAL SFT BEST
# ============================================================

print(
    "===== LOADING REFERENCE ====="
)

uf_reference_model = (
    AutoModelForCausalLM.from_pretrained(
        SFT_BEST_DIR,
        torch_dtype=torch.float32,
    )
)

uf_reference_model.to(
    REFERENCE_DEVICE
)

uf_reference_model.eval()

uf_reference_model.requires_grad_(
    False
)


# ============================================================
# CONFIRM REFERENCE REALLY IS FROZEN
# ============================================================

policy_trainable = any(
    p.requires_grad
    for p in uf_policy_model.parameters()
)

reference_trainable = any(
    p.requires_grad
    for p in uf_reference_model.parameters()
)


print(
    "\nPolicy has trainable parameters:",
    policy_trainable,
)

print(
    "Reference has trainable parameters:",
    reference_trainable,
)


assert policy_trainable is True
assert reference_trainable is False


# ============================================================
# INITIAL IDENTICAL-MODEL PAIR CHECK
# ============================================================

row0 = (
    uf_preflight_rows[0]
)

q0 = compute_pair_quantities(
    policy_model=uf_policy_model,
    reference_model=uf_reference_model,
    tokenizer=uf_dpo_tokenizer,

    prompt=format_instruction_prompt(
        row0["instruction"]
    ),

    chosen=row0["chosen"],
    rejected=row0["rejected"],

    max_length=512,
    beta=0.1,

    policy_device=POLICY_DEVICE,
    reference_device=REFERENCE_DEVICE,

    require_policy_grad=False,
)


print(
    "\n===== INITIAL IDENTICAL MODEL CHECK ====="
)

for key in [
    "policy_chosen_logp",
    "reference_chosen_logp",
    "policy_rejected_logp",
    "reference_rejected_logp",
    "loss",
    "reward_chosen",
    "reward_rejected",
    "reward_margin",
    "reward_correct",
]:

    value = q0[key]

    if torch.is_tensor(value):
        value = value.detach().float().item()

    print(
        f"{key}:",
        value,
    )


assert torch.allclose(
    q0["policy_chosen_logp"],
    q0["reference_chosen_logp"],
    atol=0.0,
    rtol=0.0,
)

assert torch.allclose(
    q0["policy_rejected_logp"],
    q0["reference_rejected_logp"],
    atol=0.0,
    rtol=0.0,
)

assert float(
    q0["reward_margin"]
    .detach()
    .float()
    .item()
) == 0.0

assert bool(
    q0["reward_correct"]
    .detach()
    .item()
) is False


print(
    "✓ Initial policy/reference behavior is exactly identical"
)


# ============================================================
# 32-PAIR GRADIENT ACCUMULATION PREFLIGHT
# ============================================================

print(
    "\n===== 32-PAIR AMP PREFLIGHT ====="
)


preflight_optimizer = (
    torch.optim.AdamW(
        uf_policy_model.parameters(),

        lr=1e-6,

        betas=(
            0.9,
            0.999,
        ),

        eps=1e-8,

        weight_decay=0.0,
    )
)


preflight_scaler = (
    torch.amp.GradScaler(
        "cuda",

        init_scale=512.0,

        growth_interval=2000,

        enabled=True,
    )
)


preflight_optimizer.zero_grad(
    set_to_none=True
)


loss_sum = 0.0


for i, row in enumerate(
    uf_preflight_rows
):

    q = compute_pair_quantities(
        policy_model=uf_policy_model,
        reference_model=uf_reference_model,
        tokenizer=uf_dpo_tokenizer,

        prompt=format_instruction_prompt(
            row["instruction"]
        ),

        chosen=row["chosen"],
        rejected=row["rejected"],

        max_length=512,
        beta=0.1,

        policy_device=POLICY_DEVICE,
        reference_device=REFERENCE_DEVICE,

        require_policy_grad=True,
    )


    loss = q["loss"]


    if not torch.isfinite(
        loss
    ):

        raise RuntimeError(
            f"Non-finite loss at pair {i}"
        )


    loss_sum += float(
        loss
        .detach()
        .float()
        .item()
    )


    preflight_scaler.scale(
        loss / 32.0
    ).backward()


# ============================================================
# UNSCALE AND INSPECT GRADIENTS
# ============================================================

preflight_scaler.unscale_(
    preflight_optimizer
)


gradient_tensors = 0
nonfinite_gradient_tensors = 0
max_abs_gradient = 0.0


for param in (
    uf_policy_model.parameters()
):

    if param.grad is None:
        continue


    gradient_tensors += 1


    grad = (
        param.grad.detach()
    )


    if not torch.isfinite(
        grad
    ).all():

        nonfinite_gradient_tensors += 1


    if grad.numel() > 0:

        max_abs_gradient = max(
            max_abs_gradient,

            float(
                grad
                .abs()
                .max()
                .float()
                .item()
            ),
        )


all_gradients_finite = (
    nonfinite_gradient_tensors
    == 0
)


grad_norm_before_clip = (
    torch.nn.utils.clip_grad_norm_(
        uf_policy_model.parameters(),
        max_norm=1.0,
    )
)


grad_norm_before_clip = float(
    grad_norm_before_clip
    .detach()
    .float()
    .item()
)


# ============================================================
# MEMORY
# ============================================================

policy_peak_gb = (
    torch.cuda.max_memory_allocated(
        POLICY_DEVICE
    )
    / (1024 ** 3)
)

reference_peak_gb = (
    torch.cuda.max_memory_allocated(
        REFERENCE_DEVICE
    )
    / (1024 ** 3)
)


# ============================================================
# RESULTS
# ============================================================

print(
    "\n===== PREFLIGHT RESULT ====="
)

print(
    "Mean DPO loss:",
    loss_sum / 32.0,
)

print(
    "Gradient tensors:",
    gradient_tensors,
)

print(
    "Non-finite gradient tensors:",
    nonfinite_gradient_tensors,
)

print(
    "All gradients finite:",
    all_gradients_finite,
)

print(
    "Max abs unscaled gradient:",
    max_abs_gradient,
)

print(
    "Gradient norm before clipping:",
    grad_norm_before_clip,
)

print(
    "GradScaler scale:",
    preflight_scaler.get_scale(),
)

print(
    "Policy GPU peak GB:",
    policy_peak_gb,
)

print(
    "Reference GPU peak GB:",
    reference_peak_gb,
)


if not all_gradients_finite:

    raise RuntimeError(
        "UltraFeedback AMP preflight failed: "
        "non-finite gradients."
    )


if not math.isfinite(
    grad_norm_before_clip
):

    raise RuntimeError(
        "UltraFeedback AMP preflight failed: "
        "gradient norm is non-finite."
    )


print(
    "\n✓ ULTRAFEEDBACK AMP PREFLIGHT PASSED"
)


# ============================================================
# IMPORTANT:
# DO NOT optimizer.step().
# This model remains exactly at the SFT initialization.
# ============================================================

preflight_optimizer.zero_grad(
    set_to_none=True
)

print(
    "✓ No optimizer step was performed"
)

===== DEVICES =====
Policy: cuda:0 Tesla T4
Reference: cuda:1 Tesla T4

Preflight pairs: 32


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



===== PROMPT FORMAT CHECK =====
Stored prompt_tokens: 41
Recomputed formatted prompt tokens: 41
✓ Prompt formatting matches frozen data metadata

===== LOADING POLICY =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

===== LOADING REFERENCE =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Policy has trainable parameters: True
Reference has trainable parameters: False

===== INITIAL IDENTICAL MODEL CHECK =====
policy_chosen_logp: -215.80233764648438
reference_chosen_logp: -215.80233764648438
policy_rejected_logp: -401.0673828125
reference_rejected_logp: -401.0673828125
loss: 0.6931471824645996
reward_chosen: 0.0
reward_rejected: 0.0
reward_margin: 0.0
reward_correct: 0.0
✓ Initial policy/reference behavior is exactly identical

===== 32-PAIR AMP PREFLIGHT =====

===== PREFLIGHT RESULT =====
Mean DPO loss: 0.6931471824645996
Gradient tensors: 290
Non-finite gradient tensors: 0
All gradients finite: True
Max abs unscaled gradient: 3.4931087493896484
Gradient norm before clipping: 61.619895935058594
GradScaler scale: 512.0
Policy GPU peak GB: 7.5375189781188965
Reference GPU peak GB: 3.2146735191345215

✓ ULTRAFEEDBACK AMP PREFLIGHT PASSED
✓ No optimizer step was performed


In [13]:
from pathlib import Path
import json
import hashlib

ROOT = Path(
    "/kaggle/working/small-model-alignment-starter"
)

MANIFEST = ROOT / "PA2_RESUME_STATE.json"


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


state = json.loads(
    MANIFEST.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# Re-hash ALL previously tracked files.
# This is necessary because data_utils.py was modified.
# ------------------------------------------------------------

for relative_path in list(
    state["critical_files"].keys()
):

    path = ROOT / relative_path

    if not path.exists():
        raise RuntimeError(
            f"Previously tracked file missing: {relative_path}"
        )

    state["critical_files"][
        relative_path
    ] = {
        "sha256": sha256_file(path)
    }


# ------------------------------------------------------------
# Add new UltraFeedback artifacts
# ------------------------------------------------------------

new_files = [
    (
        "data/ultrafeedback_fixed/"
        "ultrafeedback_train_15000_seed42.jsonl"
    ),
    (
        "data/ultrafeedback_fixed/"
        "ultrafeedback_val_1000_seed42.jsonl"
    ),
    (
        "data/ultrafeedback_fixed/"
        "selection_metadata.json"
    ),
    (
        "outputs/ultrafeedback_dpo_seed42/"
        "config.json"
    ),
]


for relative_path in new_files:

    path = ROOT / relative_path

    if not path.exists():
        raise RuntimeError(
            f"New artifact missing: {relative_path}"
        )

    state["critical_files"][
        relative_path
    ] = {
        "sha256": sha256_file(path)
    }


state["current_state"] = (
    "SFT and HH-RLHF DPO are complete. "
    "UltraFeedback fixed 15k/1k preference subsets are "
    "complete and verified. UltraFeedback DPO configuration "
    "is complete. A deterministic 32-pair AMP preflight using "
    "GradScaler init_scale=512 passed with all gradients finite. "
    "Full UltraFeedback DPO training has NOT yet started."
)

state["next_stage"] = (
    "Run full UltraFeedback from-scratch DPO training: "
    "policy and frozen reference both initialized independently "
    "from the original SFT best checkpoint; 15,000 train pairs, "
    "1,000 validation pairs, 469 optimizer updates."
)


MANIFEST.write_text(
    json.dumps(
        state,
        indent=2,
    ),
    encoding="utf-8",
)


print("===== UPDATED RESUME STATE =====")
print("Critical files:", len(state["critical_files"]))
print("Current state:")
print(state["current_state"])
print("\nNext stage:")
print(state["next_stage"])

print(
    "\nManifest SHA256:",
    sha256_file(MANIFEST),
)

print(
    "\n✓ Resume manifest updated for UltraFeedback"
)

===== UPDATED RESUME STATE =====
Critical files: 29
Current state:
SFT and HH-RLHF DPO are complete. UltraFeedback fixed 15k/1k preference subsets are complete and verified. UltraFeedback DPO configuration is complete. A deterministic 32-pair AMP preflight using GradScaler init_scale=512 passed with all gradients finite. Full UltraFeedback DPO training has NOT yet started.

Next stage:
Run full UltraFeedback from-scratch DPO training: policy and frozen reference both initialized independently from the original SFT best checkpoint; 15,000 train pairs, 1,000 validation pairs, 469 optimizer updates.

Manifest SHA256: cf8df34da87a0e6d65d42b28182bd6a3796b9b7d5ffbac4cdd8ce0d994d16837

✓ Resume manifest updated for UltraFeedback
